# Notebook 09 - Closed-Loop Control

## Purpose

Notebook 09 introduces the first true **closed-loop autonomous control layer** into HydroLab-3D.

The previous notebooks progressively established the physical and sensing foundations required for this step:

```text
00  Basic 3D Kinematics
        ↓
01  Orientation and Reference Frames
        ↓
02  Basic Dynamics
        ↓
03  Hydrodynamic Drag
        ↓
04  Gravity and Buoyancy
        ↓
05  Thruster Model
        ↓
06  Ocean Currents
        ↓
07  Complete 6-DOF UUV
        ↓
08  Sensor Models
        ↓
09  Closed-Loop Control
```

Notebook 07 established a complete lightweight six-degree-of-freedom underwater vehicle plant.

Notebook 08 then established the distinction between:

```text
physical ground truth
        ↓
sensor model
        ↓
measurement
```

Notebook 09 closes the loop.

The new architecture becomes:

```text
reference / goal
        ↓
controller
        ↓
actuator command
        ↓
thruster-generated wrench
        ↓
6-DOF UUV dynamics
        ↓
physical state
        ↓
sensor / feedback measurement
        ↓
controller
        ↺
```

The objective is not to create a large control framework.

The objective is to establish the **smallest trustworthy closed-loop control architecture** required to demonstrate that HydroLab-3D can regulate and autonomously guide its validated 6-DOF UUV using physically meaningful actuator commands and explicit feedback.

---

## Why This Matters

Until Notebook 09, HydroLab-3D could simulate how a vehicle moves under:

- inertia;
- added mass;
- Coriolis and centripetal effects;
- hydrodynamic damping;
- gravity;
- buoyancy;
- restoring moments;
- thruster forces and moments;
- ocean-current disturbances;
- imperfect sensor measurements.

However, none of those components decided what the vehicle **should do**.

The simulator could answer:

> What happens when this actuator command is applied?

Notebook 09 begins answering a different question:

> What actuator command should be applied so that the vehicle reaches or maintains a desired state?

That distinction introduces feedback.

An open-loop command follows:

```text
command
   ↓
vehicle
   ↓
motion
```

A closed-loop controller instead continuously compares desired behavior with measured behavior:

```text
desired state
     │
     ▼
   error
     │
     ▼
 controller
     │
     ▼
 actuator
     │
     ▼
   vehicle
     │
     ▼
 measurement
     │
     └─────────────── feedback ───────────────┐
                                              │
                                              ▼
                                            error
```

This allows the vehicle to react to:

- hydrodynamic drag;
- restoring forces;
- ocean currents;
- actuator saturation;
- dynamic transients;
- imperfect measurements;
- deviations from the commanded state.

Closed-loop control is therefore the bridge between the physical simulator built in Notebooks 00–08 and the autonomous behavior required by later HydroLab-3D development.

---

## Relationship to Notebook 07

Notebook 07 established the production six-degree-of-freedom UUV model with state:

$$
\eta
=
\begin{bmatrix}
x &
y &
z &
\phi &
\theta &
\psi
\end{bmatrix}^{T}
$$

and body-frame generalized velocity:

$$
\nu
=
\begin{bmatrix}
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T}.
$$

The pose contains:

```text
x, y, z       → world-frame position

phi           → roll
theta         → pitch
psi           → yaw / heading
```

while:

```text
u, v, w       → body-frame translational velocity

p, q, r       → body-frame angular velocity
```

The validated dynamics follow the marine-craft structure:

$$
M\dot{\nu}
+
C(\nu_r)\nu_r
+
D(\nu_r)\nu_r
+
g(\eta)
=
\tau
$$

with current-relative velocity represented through:

$$
\nu_r
=
\nu
-
\nu_c.
$$

Vehicle pose evolves through:

$$
\dot{\eta}
=
J(\eta)\nu.
$$

The production plant exposes the established interfaces:

```text
six_dof_state_derivative(...)
integrate_six_dof_euler(...)
simulate_six_dof(...)
```

with the derivative function returning:

```text
eta_dot
nu_dot
diagnostics
```

and the one-step integrator returning:

```text
next_pose
next_body_velocity
diagnostics
```

Notebook 09 will treat this production implementation as the physical plant.

The controller will **not** directly modify:

```text
eta
nu
position
orientation
velocity
```

to create artificial tracking behavior.

All controlled motion must arise through the existing physical actuation and dynamics chain.

---

## Relationship to Notebook 05 Actuation

Notebook 05 established the reduced actuator boundary:

```text
normalized command
        ↓
command saturation
        ↓
thruster force
```

with:

$$
u_{sat}
=
\operatorname{clip}(u,-1,1)
$$

and:

$$
F_T
=
u_{sat}F_{max}.
$$

For a body-fixed thruster direction:

$$
\hat{\mathbf d}_B,
$$

the corresponding body-frame force is:

$$
\mathbf F_{T,B}
=
F_T\hat{\mathbf d}_B.
$$

Notebook 07 generalized this into a six-component body-frame actuator wrench:

$$
\tau_T
=
\begin{bmatrix}
\mathbf F_{T,B}\\
\mathbf r_T\times\mathbf F_{T,B}
\end{bmatrix}.
$$

Therefore the Notebook 09 controller must respect the causal boundary:

```text
control error
      ↓
controller output
      ↓
bounded actuator command
      ↓
thruster force / moment
      ↓
generalized body wrench
      ↓
6-DOF dynamics
```

This is an important architectural rule.

The controller does not command acceleration directly.

It does not command velocity directly.

It does not teleport the vehicle toward the reference.

It commands the actuator layer already established by the previous notebooks.

---

## Relationship to Notebook 08

Notebook 08 established the first production sensor layer.

The available public sensing functions include models for:

```text
gyroscope
accelerometer
depth
DVL
heading
```

together with a common bias-and-noise measurement primitive.

The sensor models preserve the separation:

```text
ground truth
     ↓
sensor transformation
     ↓
measurement
```

and noisy observations never overwrite the underlying physical state.

Notebook 09 will use these sensor conventions wherever they are appropriate for the control problem being validated.

The initial feedback problem therefore becomes:

```text
physical UUV state
       ↓
sensor model
       ↓
measured feedback
       ↓
controller
       ↓
actuator command
```

This provides a more meaningful autonomy test than controlling directly from simulator truth alone.

However, HydroLab-3D does not yet contain a complete underwater localization system.

That limitation is especially important for waypoint navigation and will remain explicit throughout this notebook.

---

## Control Scope

The initial Notebook 09 control scope will remain deliberately compact.

The intended control problems are:

```text
1. depth regulation

2. heading regulation

3. simple waypoint navigation
```

The first two problems provide isolated closed-loop validation of important UUV degrees of freedom.

The waypoint experiment then combines these capabilities into a minimal autonomous navigation behavior.

This notebook will not become a general-purpose control library.

Instead, it will answer three concrete engineering questions:

```text
Can HydroLab-3D regulate depth?

Can HydroLab-3D regulate heading?

Can the validated plant and controller stack guide a UUV
toward a simple spatial goal?
```

The exact production abstraction will be chosen only after the inherited repository structure and controller requirements have been validated.

---

## PID Control Foundation

The initial classical control method will use proportional-integral-derivative control where appropriate.

For a scalar controlled variable $y$ and reference $y_{ref}$:

$$
e(t)
=
y_{ref}(t)
-
y(t).
$$

A conventional PID control law is:

$$
u(t)
=
K_P e(t)
+
K_I
\int_0^t
e(\tau)\,d\tau
+
K_D
\frac{de(t)}{dt}.
$$

where:

- $K_P$ provides proportional correction;
- $K_I$ removes persistent steady-state error;
- $K_D$ provides damping based on error evolution.

In discrete time with timestep $\Delta t$:

$$
I_k
=
I_{k-1}
+
e_k\Delta t
$$

and a simple derivative estimate is:

$$
\dot e_k
\approx
\frac{e_k-e_{k-1}}{\Delta t}.
$$

The corresponding controller output becomes:

$$
u_k
=
K_Pe_k
+
K_II_k
+
K_D\dot e_k.
$$

The actuator layer ultimately constrains the physically admissible command.

Because HydroLab-3D uses bounded normalized thruster commands, controller design must account for actuator saturation rather than assuming unlimited control authority.

If integral action is introduced, saturation behavior and integral windup must therefore be treated explicitly rather than ignored.

---

## Depth Convention

HydroLab-3D uses the world-frame convention:

```text
+x_W = horizontal x
+y_W = horizontal y
+z_W = upward
```

with the water surface at:

$$
z_W=0.
$$

Depth is therefore:

$$
\boxed{
d=-z_W
}
$$

for the default surface reference.

Thus:

```text
z = -5 m   → depth = 5 m

z = -20 m  → depth = 20 m
```

Increasing physical depth means moving toward **more negative world $z$**.

This creates an important sign distinction between:

```text
world vertical position z
```

and:

```text
positive depth d
```

The depth controller must respect this convention throughout:

```text
desired depth
      ↓
depth error
      ↓
controller
      ↓
physically correct vertical actuation
```

No control sign will be chosen merely from intuition.

The sign will be validated against the inherited actuator and six-degree-of-freedom dynamics conventions.

---

## Heading Convention

Heading is represented by vehicle yaw:

$$
\psi.
$$

Notebook 08 established the canonical heading interval:

$$
\boxed{
\psi
\in
(-\pi,\pi].
}
$$

Heading error is therefore a **circular quantity**.

A naïve subtraction:

$$
e_\psi
=
\psi_{ref}
-
\psi
$$

is incorrect near the angular boundary.

For example, headings immediately on opposite sides of $+\pi$ and $-\pi$ may represent almost identical orientations while naïve subtraction reports an error close to:

$$
2\pi.
$$

Notebook 09 will therefore reuse the already validated angle-wrapping convention so that heading error takes the shortest signed angular displacement:

$$
\boxed{
e_\psi
=
\operatorname{wrap}_{(-\pi,\pi]}
\left(
\psi_{ref}-\psi
\right).
}
$$

This circular-error handling is a required part of heading-control correctness.

---

## Planned Depth-Control Loop

Conceptually, depth regulation will follow:

```text
desired depth
      ↓
depth error
      ↓
depth controller
      ↓
vertical actuator command
      ↓
thruster / wrench model
      ↓
6-DOF dynamics
      ↓
actual depth
      ↓
depth sensor
      └──────────── feedback
```

The controller must demonstrate that the vehicle can converge toward and maintain a commanded depth without directly manipulating world $z$.

Validation will examine both transient behavior and steady regulation.

Relevant quantities may include:

```text
depth error
vertical velocity
controller output
actuator saturation
settling behavior
steady-state error
```

---

## Planned Heading-Control Loop

Heading regulation will follow the analogous structure:

```text
desired heading
       ↓
wrapped heading error
       ↓
heading controller
       ↓
yaw-producing actuator command
       ↓
generalized actuator wrench
       ↓
6-DOF rotational dynamics
       ↓
vehicle yaw
       ↓
heading sensor
       └──────────── feedback
```

The heading controller must act through a physically meaningful yaw moment.

Its validation must include at least one angular-boundary case to verify that the controller chooses the short rotation direction across the $-\pi/\pi$ discontinuity.

---

## Planned Waypoint Navigation

Once depth and heading control have been individually validated, they can support a minimal waypoint-navigation experiment.

For a vehicle position:

$$
\mathbf p
=
\begin{bmatrix}
x\\
y\\
z
\end{bmatrix}
$$

and waypoint:

$$
\mathbf p_g
=
\begin{bmatrix}
x_g\\
y_g\\
z_g
\end{bmatrix},
$$

the position error is:

$$
\mathbf e_p
=
\mathbf p_g-\mathbf p.
$$

The horizontal displacement is:

$$
\Delta x
=
x_g-x
$$

$$
\Delta y
=
y_g-y.
$$

A desired horizontal heading may be obtained from:

$$
\psi_{ref}
=
\operatorname{atan2}
\left(
\Delta y,
\Delta x
\right).
$$

The waypoint depth reference follows from the world/depth convention:

$$
d_{ref}
=
-z_g.
$$

A minimal navigation structure can therefore be expressed as:

```text
waypoint
   │
   ├── horizontal displacement
   │          ↓
   │    desired heading
   │          ↓
   │    heading controller
   │
   └── desired z
              ↓
         desired depth
              ↓
         depth controller

heading + depth regulation
              ↓
        forward propulsion
              ↓
          6-DOF UUV
              ↓
       waypoint approach
```

The exact controller decomposition and actuator geometry will be determined from the inherited production interfaces rather than assumed in advance.

---

## Feedback Assumptions

A distinction must be maintained between **measured quantities** and **perfect simulator knowledge**.

Notebook 08 currently provides direct sensor models for:

```text
depth
heading
angular velocity
specific force
water-relative body velocity
```

but it does not provide a complete globally referenced underwater position estimate.

In particular:

```text
DVL
+
depth
+
heading
```

does **not automatically imply perfect global XYZ localization**.

A real UUV would normally require some combination of:

- inertial navigation;
- DVL-aided dead reckoning;
- acoustic positioning;
- GPS fixes while surfaced;
- external localization;
- state estimation;
- map-based localization.

Building that complete estimation stack is outside the intended scope of Notebook 09.

Therefore, if the waypoint-navigation validation uses the simulator's world position as a temporary perfect-localization baseline, that assumption will be stated explicitly.

Conceptually:

```text
depth control
    → sensor feedback

heading control
    → sensor feedback

waypoint position
    → temporary perfect-localization baseline if required
```

This allows Notebook 09 to validate **control and navigation logic** without pretending that a localization system already exists.

The future estimator can later replace the perfect-position baseline without changing the fundamental control architecture.

---

## Controller / Plant Separation

Notebook 09 must preserve a strict architectural separation between:

```text
reference generation
controller
actuator
plant
sensor
```

The controller may compute a command.

The actuator converts that command into physically bounded force or moment.

The UUV model determines the resulting acceleration and motion.

The sensors produce observations of the resulting physical state.

The controller then reacts to those observations.

Therefore:

```text
controller output ≠ vehicle state
```

and:

```text
controller output ≠ arbitrary applied state correction
```

Instead:

```text
controller output
        ↓
physical actuator interface
        ↓
dynamics
        ↓
state evolution
```

This separation is fundamental to later:

- simulator-kernel integration;
- alternative controllers;
- learning-based control;
- disturbance testing;
- ROS 2 interfaces;
- hardware deployment.

---

## Validation Strategy

Notebook 09 will follow the established HydroLab-3D validation hierarchy.

Development will proceed through:

```text
inspect inherited implementation
        ↓
prototype minimum controller logic
        ↓
analytical / numerical validation
        ↓
isolated closed-loop test
        ↓
integration with production 6-DOF plant
        ↓
sensor-feedback validation
        ↓
disturbance / saturation checks
        ↓
waypoint-navigation validation
        ↓
production promotion
        ↓
production parity check
        ↓
fresh-process validation
        ↓
persistent artifacts
        ↓
dedicated automated tests
        ↓
full repository regression
```

The notebook will prefer high-information validation cases over large numbers of arbitrary simulations.

---

## Depth-Control Validation Criteria

A successful depth-control implementation should demonstrate that:

1. the controller responds with the physically correct sign;
2. the vehicle approaches a commanded depth through the actuator/dynamics chain;
3. the depth error decreases substantially from its initial value;
4. the response remains numerically finite and dynamically bounded;
5. actuator commands remain inside the physical command interface;
6. a steady reference can be maintained to a useful tolerance;
7. imperfect depth measurements can participate in the feedback loop without corrupting simulator truth.

The exact numerical tolerances will be selected after the canonical plant and actuator configuration has been inspected and exercised.

---

## Heading-Control Validation Criteria

A successful heading-control implementation should demonstrate that:

1. heading error uses the existing wrapped-angle convention;
2. commanded yaw actuation produces rotation through the 6-DOF dynamics;
3. the heading error converges toward zero;
4. the response remains dynamically bounded;
5. the controller behaves correctly near the $-\pi/\pi$ boundary;
6. actuator saturation is respected;
7. measured heading can be used without modifying ground truth.

---

## Waypoint-Navigation Validation Criteria

A successful minimal waypoint experiment should demonstrate that:

1. the vehicle begins away from the target;
2. the navigation layer generates physically meaningful heading and depth references;
3. the controller acts only through the validated actuator/dynamics chain;
4. distance to the waypoint decreases substantially;
5. the UUV enters a defined waypoint-success region;
6. the trajectory remains finite and dynamically plausible;
7. any use of perfect world-position knowledge is explicitly identified as a temporary localization assumption.

The waypoint experiment is intended as a control-and-autonomy integration test, not as a complete underwater navigation solution.

---

## Production-Promotion Strategy

Notebook 09 will continue the established HydroLab-3D development method:

```text
notebook prototype
        ↓
validation
        ↓
reusable production implementation
        ↓
automated regression tests
        ↓
notebook re-imports production code
```

No controller code will be promoted merely because it runs.

Promotion requires demonstrated:

```text
correct sign behavior
correct error convention
bounded actuation
numerical stability
plant compatibility
sensor compatibility
determinism where expected
```

The existing repository structure will determine the final production location.

A dedicated control package such as:

```text
src/hydrolab3d/control/
```

is architecturally plausible, but it will only be introduced if the actual implementation justifies that boundary.

HydroLab-3D will not create unused abstractions merely because they appear in the tentative repository plan.

---

## Artifact Strategy

Persistent Notebook 09 outputs will be stored under:

```text
results/notebooks/09_closed_loop_control/
```

Artifacts will be created only after the corresponding production behavior has been validated.

The final artifact set should remain compact and useful.

Likely validation outputs include representations of:

```text
depth-control response
heading-control response
waypoint trajectory
control effort / actuator command
summary metrics
```

The exact filenames and artifact count will be determined by the validated experiments rather than fixed prematurely.

No previous notebook outputs will be overwritten.

---

## Reproducibility

Notebook 09 will preserve the deterministic and reproducible development philosophy established by the previous notebooks.

Where noisy sensors are used:

```python
np.random.default_rng(seed)
```

will remain the explicit random-number mechanism.

Independent sensor streams will remain independently seeded where multiple stochastic sensors are involved.

Canonical controller tests should use fixed:

```text
initial states
vehicle parameters
controller gains
timesteps
references
currents
sensor seeds
```

so that regressions can be detected reliably.

---

## Expected Result

At the end of Notebook 09, HydroLab-3D should contain its first validated autonomous feedback behavior.

Conceptually:

```text
             REFERENCE
                 │
                 ▼
             CONTROLLER
                 │
                 ▼
              ACTUATOR
                 │
                 ▼
           SIX-DOF PLANT
                 │
                 ▼
              SENSORS
                 │
                 └───────────────┐
                                 │
                                 ▼
                              FEEDBACK
```

The simulator should be able to demonstrate that a vehicle can:

```text
hold / acquire depth
        +
hold / acquire heading
        +
approach a simple waypoint
```

while preserving the physical constraints introduced throughout Notebooks 00–08.

This will mark HydroLab-3D's transition from:

```text
a simulator that can reproduce vehicle motion
```

to:

```text
a simulator in which an autonomous controller can influence
vehicle motion through a closed physical feedback loop
```

---

## Current Limitations

Notebook 09 will deliberately **not** attempt to implement:

```text
full-state estimation
EKF / UKF localization
SLAM
acoustic positioning
GPS surfacing logic
trajectory optimization
LQR
MPC
sliding-mode control
adaptive control
reinforcement-learning control
multi-UUV control
collision avoidance
obstacle avoidance
mission planning
communication-aware autonomy
ROS 2 control interfaces
thruster allocation for a complex multi-thruster vehicle
hardware-specific motor dynamics
high-fidelity propulsion dynamics
```

These are future extensions of the control architecture established here.

The immediate goal is narrower:

$$
\boxed{
\text{prove a trustworthy closed-loop path from reference to actuator to plant to feedback}
}
$$

---

## Relationship to the HydroLab-3D Architecture

Notebook 09 occupies the first autonomy layer above the validated physical simulator:

```text
                    AUTONOMY
                       │
                       ▼
                Closed-Loop Control
                       │
                       ▼
                    Actuation
                       │
                       ▼
                  6-DOF Vehicle
                       │
               ┌───────┴────────┐
               ▼                ▼
          Environment         Sensors
               │                │
               └────────┬───────┘
                        ▼
                     Feedback
```

This controller/plant separation is intentionally compatible with later simulator development.

The same conceptual controller should eventually be able to receive observations and issue commands through a simulator interface rather than depending directly on notebook internals.

---

## Handoff Goal

Notebook 09 is the final major component-level notebook before the validated pieces are assembled into the simulator kernel.

Its handoff target is:

```text
Notebook 09
Closed-Loop Control
        ↓
validated reference-to-actuator feedback behavior
        ↓
Notebook 10
Simulator Kernel Integration
        ↓
Simulator.step(...)
```

Notebook 10 can then combine the already validated:

```text
vehicle state
dynamics
currents
thrusters
sensors
controllers
```

behind a coherent simulation stepping interface.

Therefore Notebook 09 should finish with a controller architecture that is:

```text
small
explicit
physically connected
sensor-compatible
numerically validated
modular
reproducible
ready for simulator-kernel integration
```

The guiding principle remains the same as the rest of HydroLab-3D:

$$
\boxed{
\text{maximum useful autonomy capability per unit of computational complexity}
}
$$

---

## Step 1 - Audit the Inherited Closed-Loop Interfaces

Before implementing any controller, inspect the existing HydroLab-3D production boundary that Notebook 09 must use.

This audit verifies:

```text
controller
    ↓
thruster / actuator interface
    ↓
6-DOF vehicle plant
    ↓
sensor feedback
```

The step is deliberately read-only.

We will confirm the exact production locations and callable signatures for:

- the validated thruster model;
- the 6-DOF state derivative and integrator;
- the public sensor API;
- the existing angle-wrapping convention;
- the current automated-test inventory;
- and whether a `control/` package already exists.

No controller architecture will be created until these inherited contracts have been inspected.

### Validation Criteria

This step passes if:

1. the repository root resolves correctly;
2. all required Notebook 09 dependencies exist;
3. inherited production functions import successfully;
4. their actual signatures and source locations can be inspected;
5. the sensor public API contains the validated Notebook 08 functions;
6. the current `control/` package state is reported without modification;
7. the test inventory is successfully discovered.

No production files are modified in this step.

In [1]:
from pathlib import Path
import importlib
import inspect
import sys


# ---------------------------------------------------------------------
# Resolve HydroLab-3D repository root
# ---------------------------------------------------------------------

start = Path.cwd().resolve()
repo_root = None

for candidate in [start, *start.parents]:
    if (
        (candidate / "README.md").exists()
        and (candidate / "src").exists()
        and (candidate / "notebooks").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the HydroLab-3D repository root from "
        f"{start}"
    )


src_path = repo_root / "src"

src_path_str = str(src_path)

if src_path_str not in sys.path:
    sys.path.insert(
        0,
        src_path_str,
    )

importlib.invalidate_caches()


# ---------------------------------------------------------------------
# Required inherited production files
# ---------------------------------------------------------------------

required_files = {
    "thrusters": (
        src_path
        / "hydrolab3d"
        / "dynamics"
        / "thrusters.py"
    ),
    "transforms": (
        src_path
        / "hydrolab3d"
        / "utils"
        / "transforms.py"
    ),
    "currents": (
        src_path
        / "hydrolab3d"
        / "world"
        / "currents.py"
    ),
    "uuv": (
        src_path
        / "hydrolab3d"
        / "vehicles"
        / "uuv.py"
    ),
    "sensor": (
        src_path
        / "hydrolab3d"
        / "sensors"
        / "sensor.py"
    ),
    "imu": (
        src_path
        / "hydrolab3d"
        / "sensors"
        / "imu.py"
    ),
    "depth": (
        src_path
        / "hydrolab3d"
        / "sensors"
        / "depth.py"
    ),
    "dvl": (
        src_path
        / "hydrolab3d"
        / "sensors"
        / "dvl.py"
    ),
    "heading": (
        src_path
        / "hydrolab3d"
        / "sensors"
        / "heading.py"
    ),
    "sensors_init": (
        src_path
        / "hydrolab3d"
        / "sensors"
        / "__init__.py"
    ),
}


for name, path in required_files.items():

    if not path.exists():
        raise FileNotFoundError(
            "Required Notebook 09 dependency is missing "
            f"({name}): {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected a file for {name}, "
            f"found another path type: {path}"
        )


# ---------------------------------------------------------------------
# Import inherited actuator functionality
# ---------------------------------------------------------------------

from hydrolab3d.dynamics.thrusters import (
    thruster_force_body,
    thruster_force_world,
)


# ---------------------------------------------------------------------
# Import inherited 6-DOF plant functionality
# ---------------------------------------------------------------------

from hydrolab3d.vehicles.uuv import (
    current_generalized_acceleration,
    six_dof_state_derivative,
    integrate_six_dof_euler,
    simulate_six_dof,
)


# ---------------------------------------------------------------------
# Import public sensor package
# ---------------------------------------------------------------------

import hydrolab3d.sensors as sensors


# ---------------------------------------------------------------------
# Functions most relevant to Notebook 09
# ---------------------------------------------------------------------

inherited_functions = {
    "thruster_force_body":
        thruster_force_body,

    "thruster_force_world":
        thruster_force_world,

    "current_generalized_acceleration":
        current_generalized_acceleration,

    "six_dof_state_derivative":
        six_dof_state_derivative,

    "integrate_six_dof_euler":
        integrate_six_dof_euler,

    "simulate_six_dof":
        simulate_six_dof,

    "depth_measurement":
        sensors.depth_measurement,

    "heading_measurement":
        sensors.heading_measurement,

    "dvl_measurement":
        sensors.dvl_measurement,

    "gyroscope_measurement":
        sensors.gyroscope_measurement,

    "wrap_angle_pi":
        sensors.wrap_angle_pi,
}


# ---------------------------------------------------------------------
# Verify inherited callables
# ---------------------------------------------------------------------

for name, function in inherited_functions.items():

    if not callable(function):
        raise TypeError(
            f"Inherited dependency is not callable: {name}"
        )


# ---------------------------------------------------------------------
# Inspect prospective control-package state
#
# Read only. Nothing is created.
# ---------------------------------------------------------------------

control_dir = (
    src_path
    / "hydrolab3d"
    / "control"
)

if control_dir.exists():

    if not control_dir.is_dir():
        raise RuntimeError(
            "Expected control path to be a directory: "
            f"{control_dir}"
        )

    control_files = sorted(
        path
        for path in control_dir.rglob("*")
        if path.is_file()
    )

else:
    control_files = []


# ---------------------------------------------------------------------
# Current automated-test inventory
# ---------------------------------------------------------------------

tests_dir = (
    repo_root
    / "tests"
)

if not tests_dir.exists():
    raise FileNotFoundError(
        f"Expected tests directory is missing: {tests_dir}"
    )

test_files = sorted(
    tests_dir.glob("test_*.py")
)

if not test_files:
    raise RuntimeError(
        "No test_*.py files were found."
    )


# ---------------------------------------------------------------------
# Report repository
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 09 - STEP 1 INTERFACE AUDIT")
print("=" * 80)

print("\nRepository root:")
print(f"  {repo_root}")


# ---------------------------------------------------------------------
# Required production files
# ---------------------------------------------------------------------

print("\nRequired inherited production files:")

for name, path in required_files.items():

    print(
        f"  {name:<18} -> "
        f"{path.relative_to(repo_root)}"
    )


# ---------------------------------------------------------------------
# Function signatures and source locations
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("INHERITED CALLABLE CONTRACTS")
print("=" * 80)

for name, function in inherited_functions.items():

    signature = inspect.signature(
        function
    )

    source_file = Path(
        inspect.getsourcefile(function)
        or inspect.getfile(function)
    ).resolve()

    try:
        display_source = (
            source_file.relative_to(
                repo_root
            )
        )
    except ValueError:
        display_source = source_file

    print()
    print(name)
    print("-" * len(name))

    print(
        f"signature: {name}{signature}"
    )

    print(
        f"source:    {display_source}"
    )


# ---------------------------------------------------------------------
# Public sensor API
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("PUBLIC SENSOR API")
print("=" * 80)

sensor_public_api = list(
    getattr(
        sensors,
        "__all__",
        [],
    )
)

if not sensor_public_api:
    raise AssertionError(
        "hydrolab3d.sensors.__all__ is missing or empty"
    )

for name in sensor_public_api:
    print(f"  {name}")


required_sensor_exports = {
    "depth_measurement",
    "heading_measurement",
    "dvl_measurement",
    "gyroscope_measurement",
    "wrap_angle_pi",
}

missing_sensor_exports = (
    required_sensor_exports
    - set(sensor_public_api)
)

if missing_sensor_exports:
    raise AssertionError(
        "Required Notebook 09 sensor exports are missing: "
        f"{sorted(missing_sensor_exports)}"
    )


# ---------------------------------------------------------------------
# Inspect current control-package state
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("CURRENT CONTROL PACKAGE STATE")
print("=" * 80)

if not control_dir.exists():

    print(
        "  src/hydrolab3d/control/ does not currently exist."
    )

else:

    print(
        f"  {control_dir.relative_to(repo_root)} exists."
    )

    if control_files:

        for path in control_files:
            print(
                "  "
                + str(
                    path.relative_to(
                        repo_root
                    )
                )
            )

    else:
        print(
            "  Directory exists but contains no files."
        )


# ---------------------------------------------------------------------
# Current automated tests
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("CURRENT TEST INVENTORY")
print("=" * 80)

for path in test_files:
    print(
        f"  {path.relative_to(repo_root)}"
    )


# ---------------------------------------------------------------------
# Read-only assertions
# ---------------------------------------------------------------------

assert callable(
    six_dof_state_derivative
)

assert callable(
    integrate_six_dof_euler
)

assert callable(
    thruster_force_body
)

assert callable(
    sensors.depth_measurement
)

assert callable(
    sensors.heading_measurement
)

assert callable(
    sensors.wrap_angle_pi
)


print("\n" + "=" * 80)
print("STEP 1 AUDIT PASSED")
print("=" * 80)

print(
    "Inherited actuator, plant, and sensor interfaces "
    "resolved successfully."
)

print(
    "Control-package state and test inventory inspected "
    "read-only."
)

print(
    "No production files were created or modified."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 1 INTERFACE AUDIT

Repository root:
  /home/agniacolyte/HydroLab-3D

Required inherited production files:
  thrusters          -> src/hydrolab3d/dynamics/thrusters.py
  transforms         -> src/hydrolab3d/utils/transforms.py
  currents           -> src/hydrolab3d/world/currents.py
  uuv                -> src/hydrolab3d/vehicles/uuv.py
  sensor             -> src/hydrolab3d/sensors/sensor.py
  imu                -> src/hydrolab3d/sensors/imu.py
  depth              -> src/hydrolab3d/sensors/depth.py
  dvl                -> src/hydrolab3d/sensors/dvl.py
  heading            -> src/hydrolab3d/sensors/heading.py
  sensors_init       -> src/hydrolab3d/sensors/__init__.py

INHERITED CALLABLE CONTRACTS

thruster_force_body
-------------------
signature: thruster_force_body(command, max_thrust, direction_body)
source:    src/hydrolab3d/dynamics/thrusters.py

thruster_force_world
--------------------
signature: thruster_force_world(command, max_thrust, direction_

## Step 2 - Inspect the Existing Control Package

The Step 1 audit revealed that HydroLab-3D already contains:

```text
src/hydrolab3d/control/
├── __init__.py
├── controller.py
├── pid.py
└── waypoint.py
```

Before implementing any Notebook 09 controller logic, we must determine what these files actually contain.

This step is read-only.

We will inspect:

- the control package public API;
- existing classes and functions;
- PID behavior already present;
- waypoint functionality already present;
- any abstract controller interface;
- imports and dependencies;
- and whether the existing implementation is complete, partial, or placeholder code.

### Validation Criteria

This step passes if the complete current contents of all four control-package files can be inspected without modifying them.

Only after this inspection will we decide the smallest useful Notebook 09 implementation path.

In [2]:
from pathlib import Path


# ---------------------------------------------------------------------
# Locate existing HydroLab-3D control package
# ---------------------------------------------------------------------

control_dir = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "control"
)

control_files = [
    control_dir / "__init__.py",
    control_dir / "controller.py",
    control_dir / "pid.py",
    control_dir / "waypoint.py",
]


# ---------------------------------------------------------------------
# Validate expected files
# ---------------------------------------------------------------------

for path in control_files:

    if not path.exists():
        raise FileNotFoundError(
            f"Expected control file is missing: {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected a regular file: {path}"
        )


# ---------------------------------------------------------------------
# Display complete existing source
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 09 - STEP 2 CONTROL PACKAGE INSPECTION")
print("=" * 80)

for path in control_files:

    relative_path = path.relative_to(
        repo_root
    )

    source = path.read_text(
        encoding="utf-8"
    )

    print("\n")
    print("=" * 80)
    print(relative_path)
    print("=" * 80)

    if source.strip():

        numbered_lines = source.splitlines()

        for line_number, line in enumerate(
            numbered_lines,
            start=1,
        ):
            print(
                f"{line_number:4d} | {line}"
            )

    else:

        print(
            "<EMPTY FILE>"
        )


# ---------------------------------------------------------------------
# Inspection summary
# ---------------------------------------------------------------------

print("\n")
print("=" * 80)
print("STEP 2 INSPECTION COMPLETE")
print("=" * 80)

print(
    "Existing control-package source displayed read-only."
)

print(
    "No controller code has been created or modified."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 2 CONTROL PACKAGE INSPECTION


src/hydrolab3d/control/__init__.py
<EMPTY FILE>


src/hydrolab3d/control/controller.py
<EMPTY FILE>


src/hydrolab3d/control/pid.py
<EMPTY FILE>


src/hydrolab3d/control/waypoint.py
<EMPTY FILE>


STEP 2 INSPECTION COMPLETE
Existing control-package source displayed read-only.
No controller code has been created or modified.


## Step 3 - Notebook-Local PID Primitive and Numerical Validation

The existing control package contains only empty placeholders, so Notebook 09 can define its controller contract from first principles.

We begin with the smallest reusable closed-loop primitive required by the planned depth and heading controllers: a scalar PID controller.

For error:

$$
e_k
=
r_k-y_k
$$

the discrete controller uses:

$$
I_k
=
I_{k-1}
+
e_k\Delta t
$$

and:

$$
\dot e_k
=
\frac{e_k-e_{k-1}}{\Delta t}.
$$

The unsaturated control output is:

$$
u_k^{*}
=
K_Pe_k
+
K_II_k
+
K_D\dot e_k.
$$

The physical actuator boundary requires bounded output:

$$
u_k
=
\operatorname{clip}
\left(
u_k^{*},
u_{\min},
u_{\max}
\right).
$$

The prototype will also support an explicit integral bound:

$$
I_k
\in
[I_{\min},I_{\max}]
$$

to prevent uncontrolled integral accumulation when the actuator is saturated.

This PID primitive is intentionally independent of:

```text
depth
heading
waypoints
thruster geometry
6-DOF dynamics
```

Those layers will provide the appropriate error signal later.

In particular, the PID controller itself will **not** contain angle wrapping. Heading-error wrapping remains a separate semantic operation using the already validated `wrap_angle_pi()` function.

### Validation Cases

This step validates the PID primitive independently of the UUV plant using deterministic cases:

1. proportional-only response;
2. integral accumulation;
3. derivative response;
4. actuator-output saturation;
5. integral-state clamping;
6. controller reset;
7. rejection of a non-positive timestep.

No production files are modified in this step.

In [3]:
from dataclasses import dataclass
import numpy as np


# ---------------------------------------------------------------------
# Notebook-local PID prototype
# ---------------------------------------------------------------------

@dataclass
class NotebookPID:
    """
    Minimal scalar PID controller for Notebook 09 validation.

    Parameters
    ----------
    kp, ki, kd : float
        Proportional, integral, and derivative gains.

    output_limits : tuple[float, float]
        Minimum and maximum controller output.

    integral_limits : tuple[float, float] or None
        Optional bounds on the accumulated integral state.
    """

    kp: float
    ki: float
    kd: float
    output_limits: tuple = (-1.0, 1.0)
    integral_limits: tuple | None = None

    def __post_init__(self):

        self.kp = float(self.kp)
        self.ki = float(self.ki)
        self.kd = float(self.kd)

        output_min, output_max = self.output_limits

        self.output_min = float(output_min)
        self.output_max = float(output_max)

        if self.output_min > self.output_max:
            raise ValueError(
                "output_limits must satisfy minimum <= maximum."
            )

        if self.integral_limits is None:

            self.integral_min = None
            self.integral_max = None

        else:

            integral_min, integral_max = self.integral_limits

            self.integral_min = float(integral_min)
            self.integral_max = float(integral_max)

            if self.integral_min > self.integral_max:
                raise ValueError(
                    "integral_limits must satisfy minimum <= maximum."
                )

        self.reset()


    def reset(self):
        """Reset accumulated PID state."""

        self.integral = 0.0
        self.previous_error = None


    def update(self, error, dt):
        """
        Advance the PID controller by one timestep.

        Parameters
        ----------
        error : float
            Current scalar control error.

        dt : float
            Positive controller timestep [s].

        Returns
        -------
        output : float
            Saturated controller output.

        diagnostics : dict
            Individual PID terms and internal state.
        """

        error = float(error)
        dt = float(dt)

        if not np.isfinite(error):
            raise ValueError(
                "error must be finite."
            )

        if not np.isfinite(dt) or dt <= 0.0:
            raise ValueError(
                "dt must be finite and strictly positive."
            )

        # -------------------------------------------------------------
        # Proportional term
        # -------------------------------------------------------------

        proportional = self.kp * error


        # -------------------------------------------------------------
        # Integral state
        # -------------------------------------------------------------

        self.integral += error * dt

        if self.integral_limits is not None:

            self.integral = float(
                np.clip(
                    self.integral,
                    self.integral_min,
                    self.integral_max,
                )
            )

        integral_term = self.ki * self.integral


        # -------------------------------------------------------------
        # Derivative term
        #
        # The first call has no previous sample, so its derivative
        # contribution is defined as zero.
        # -------------------------------------------------------------

        if self.previous_error is None:

            error_derivative = 0.0

        else:

            error_derivative = (
                error - self.previous_error
            ) / dt

        derivative = self.kd * error_derivative


        # -------------------------------------------------------------
        # Unsaturated and saturated outputs
        # -------------------------------------------------------------

        unsaturated_output = (
            proportional
            + integral_term
            + derivative
        )

        output = float(
            np.clip(
                unsaturated_output,
                self.output_min,
                self.output_max,
            )
        )


        # -------------------------------------------------------------
        # Store state for next update
        # -------------------------------------------------------------

        self.previous_error = error


        diagnostics = {
            "error": error,
            "proportional": proportional,
            "integral_state": self.integral,
            "integral": integral_term,
            "error_derivative": error_derivative,
            "derivative": derivative,
            "unsaturated_output": unsaturated_output,
            "output": output,
            "saturated": not np.isclose(
                output,
                unsaturated_output,
            ),
        }

        return output, diagnostics


# ---------------------------------------------------------------------
# Test 1 - Proportional-only response
# ---------------------------------------------------------------------

pid_p = NotebookPID(
    kp=2.0,
    ki=0.0,
    kd=0.0,
    output_limits=(-10.0, 10.0),
)

output_p, diag_p = pid_p.update(
    error=1.5,
    dt=0.1,
)

expected_p = 3.0

assert np.isclose(
    output_p,
    expected_p,
)

print("Test 1 - Proportional response")
print(f"  output:   {output_p:.6f}")
print(f"  expected: {expected_p:.6f}")
print("  PASS")


# ---------------------------------------------------------------------
# Test 2 - Integral accumulation
# ---------------------------------------------------------------------

pid_i = NotebookPID(
    kp=0.0,
    ki=2.0,
    kd=0.0,
    output_limits=(-100.0, 100.0),
)

integral_outputs = []

for _ in range(4):

    output_i, diag_i = pid_i.update(
        error=1.0,
        dt=0.5,
    )

    integral_outputs.append(
        output_i
    )

expected_integral_state = 2.0
expected_integral_output = 4.0

assert np.isclose(
    pid_i.integral,
    expected_integral_state,
)

assert np.isclose(
    integral_outputs[-1],
    expected_integral_output,
)

print("\nTest 2 - Integral accumulation")
print(
    f"  integral state: {pid_i.integral:.6f}"
)
print(
    f"  final output:   {integral_outputs[-1]:.6f}"
)
print(
    f"  expected:       {expected_integral_output:.6f}"
)
print("  PASS")


# ---------------------------------------------------------------------
# Test 3 - Derivative response
# ---------------------------------------------------------------------

pid_d = NotebookPID(
    kp=0.0,
    ki=0.0,
    kd=3.0,
    output_limits=(-100.0, 100.0),
)

first_output, first_diag = pid_d.update(
    error=1.0,
    dt=0.5,
)

second_output, second_diag = pid_d.update(
    error=2.0,
    dt=0.5,
)

expected_first_derivative = 0.0

expected_second_derivative = (
    3.0
    * ((2.0 - 1.0) / 0.5)
)

assert np.isclose(
    first_output,
    expected_first_derivative,
)

assert np.isclose(
    second_output,
    expected_second_derivative,
)

print("\nTest 3 - Derivative response")
print(
    f"  first output:  {first_output:.6f}"
)
print(
    f"  second output: {second_output:.6f}"
)
print(
    f"  expected:      {expected_second_derivative:.6f}"
)
print("  PASS")


# ---------------------------------------------------------------------
# Test 4 - Output saturation
# ---------------------------------------------------------------------

pid_sat = NotebookPID(
    kp=10.0,
    ki=0.0,
    kd=0.0,
    output_limits=(-1.0, 1.0),
)

output_sat, diag_sat = pid_sat.update(
    error=5.0,
    dt=0.1,
)

assert np.isclose(
    diag_sat["unsaturated_output"],
    50.0,
)

assert np.isclose(
    output_sat,
    1.0,
)

assert diag_sat["saturated"]

print("\nTest 4 - Output saturation")
print(
    f"  raw output:       "
    f"{diag_sat['unsaturated_output']:.6f}"
)
print(
    f"  saturated output: {output_sat:.6f}"
)
print("  PASS")


# ---------------------------------------------------------------------
# Test 5 - Integral-state clamping
# ---------------------------------------------------------------------

pid_clamp = NotebookPID(
    kp=0.0,
    ki=1.0,
    kd=0.0,
    output_limits=(-10.0, 10.0),
    integral_limits=(-0.5, 0.5),
)

for _ in range(20):

    output_clamp, diag_clamp = pid_clamp.update(
        error=1.0,
        dt=0.1,
    )

assert np.isclose(
    pid_clamp.integral,
    0.5,
)

assert np.isclose(
    output_clamp,
    0.5,
)

print("\nTest 5 - Integral-state clamping")
print(
    f"  integral state: {pid_clamp.integral:.6f}"
)
print(
    f"  output:         {output_clamp:.6f}"
)
print("  PASS")


# ---------------------------------------------------------------------
# Test 6 - Reset behavior
# ---------------------------------------------------------------------

pid_reset = NotebookPID(
    kp=1.0,
    ki=1.0,
    kd=1.0,
)

pid_reset.update(
    error=2.0,
    dt=0.1,
)

pid_reset.reset()

assert np.isclose(
    pid_reset.integral,
    0.0,
)

assert pid_reset.previous_error is None

print("\nTest 6 - Reset behavior")
print(
    f"  integral state: {pid_reset.integral:.6f}"
)
print(
    f"  previous error: {pid_reset.previous_error}"
)
print("  PASS")


# ---------------------------------------------------------------------
# Test 7 - Invalid timestep rejection
# ---------------------------------------------------------------------

pid_invalid = NotebookPID(
    kp=1.0,
    ki=0.0,
    kd=0.0,
)

invalid_dt_rejected = False

try:

    pid_invalid.update(
        error=1.0,
        dt=0.0,
    )

except ValueError:

    invalid_dt_rejected = True

assert invalid_dt_rejected

print("\nTest 7 - Invalid timestep rejection")
print("  dt = 0 rejected correctly")
print("  PASS")


# ---------------------------------------------------------------------
# Step summary
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 3 PID PROTOTYPE VALIDATION PASSED")
print("=" * 80)

print(
    "Notebook-local scalar PID behavior validated."
)

print(
    "Proportional, integral, derivative, saturation, "
    "integral clamping, reset, and timestep checks passed."
)

print(
    "No production files were created or modified."
)

Test 1 - Proportional response
  output:   3.000000
  expected: 3.000000
  PASS

Test 2 - Integral accumulation
  integral state: 2.000000
  final output:   4.000000
  expected:       4.000000
  PASS

Test 3 - Derivative response
  first output:  0.000000
  second output: 6.000000
  expected:      6.000000
  PASS

Test 4 - Output saturation
  raw output:       50.000000
  saturated output: 1.000000
  PASS

Test 5 - Integral-state clamping
  integral state: 0.500000
  output:         0.500000
  PASS

Test 6 - Reset behavior
  integral state: 0.000000
  previous error: None
  PASS

Test 7 - Invalid timestep rejection
  dt = 0 rejected correctly
  PASS

STEP 3 PID PROTOTYPE VALIDATION PASSED
Notebook-local scalar PID behavior validated.
Proportional, integral, derivative, saturation, integral clamping, reset, and timestep checks passed.
No production files were created or modified.


## Step 4 - Closed-Loop Depth Regulation Through the 6-DOF Plant

The PID primitive has now been validated independently.

We can therefore connect it to the production UUV dynamics.

The controlled variable is positive depth:

$$
d=-z_W
$$

with depth error:

$$
e_d
=
d_{ref}-d.
$$

The controller output is interpreted as a normalized vertical actuation command.

This command must not directly modify the vehicle position or velocity.

Instead, it is converted into a body-frame generalized actuator wrench:

```text
PID output
    ↓
bounded vertical force
    ↓
actuator wrench
    ↓
6-DOF plant
    ↓
new physical state
```

This experiment uses the canonical parameter representation established by Notebook 07:

```text
inertia
    → shape (3,)
    → [Ix, Iy, Iz]

added_inertia
    → shape (6,)
    → [m_Au, m_Av, m_Aw, m_Ap, m_Aq, m_Ar]
```

For this first integration experiment we use ideal depth feedback.

This isolates:

- depth-error sign;
- vertical actuator sign;
- PID/plant coupling;
- closed-loop stability;
- actuator saturation;
- convergence behavior.

Sensor noise will be introduced only after this deterministic loop is validated.

### Validation Criteria

The experiment passes if:

1. the initial vehicle depth differs substantially from the target;
2. the controller acts only through `actuator_wrench`;
3. the depth error decreases substantially;
4. the final depth is close to the commanded reference;
5. all simulated states remain finite;
6. controller commands remain within `[-1, 1]`;
7. the depth response does not diverge.

No production files are modified in this step.

In [5]:
import numpy as np

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)

from hydrolab3d.sensors import (
    ideal_depth_measurement,
)


# ---------------------------------------------------------------------
# Canonical Notebook 07 plant parameters
# ---------------------------------------------------------------------
#
# IMPORTANT:
#
# inertia must be shape (3,)
# added_inertia must be shape (6,)
#
# These follow the production representation validated in Notebook 07.
# ---------------------------------------------------------------------

mass = 20.0

inertia = np.array(
    [
        1.2,
        1.6,
        2.0,
    ],
    dtype=float,
)

added_inertia = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ],
    dtype=float,
)

linear_damping = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ],
    dtype=float,
)

quadratic_damping = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ],
    dtype=float,
)


# ---------------------------------------------------------------------
# Neutral-buoyancy configuration
# ---------------------------------------------------------------------

fluid_density = 1000.0

gravity = 9.81

displaced_volume = (
    mass
    / fluid_density
)

center_of_gravity = np.zeros(
    3,
    dtype=float,
)

center_of_buoyancy = np.zeros(
    3,
    dtype=float,
)

current_velocity_world = np.zeros(
    3,
    dtype=float,
)


# ---------------------------------------------------------------------
# Validate inherited parameter shapes before simulation
# ---------------------------------------------------------------------

assert inertia.shape == (3,)

assert added_inertia.shape == (6,)

assert linear_damping.shape == (6,)

assert quadratic_damping.shape == (6,)

assert center_of_gravity.shape == (3,)

assert center_of_buoyancy.shape == (3,)

assert current_velocity_world.shape == (3,)


# ---------------------------------------------------------------------
# Simulation setup
# ---------------------------------------------------------------------

dt = 0.02

duration = 20.0

steps = int(
    duration / dt
)

times = (
    np.arange(
        steps + 1,
        dtype=float,
    )
    * dt
)


# ---------------------------------------------------------------------
# Initial state
#
# HydroLab convention:
#
# +z_W = upward
#
# surface:
#
# z = 0
#
# depth:
#
# d = -z
#
# Therefore:
#
# z = -2 m
# corresponds to:
#
# depth = 2 m
# ---------------------------------------------------------------------

pose = np.array(
    [
        0.0,   # x
        0.0,   # y
        -2.0,  # z
        0.0,   # roll
        0.0,   # pitch
        0.0,   # yaw
    ],
    dtype=float,
)

body_velocity = np.zeros(
    6,
    dtype=float,
)

target_depth = 8.0


# ---------------------------------------------------------------------
# Depth PID
# ---------------------------------------------------------------------
#
# Positive depth error means:
#
# target_depth > measured_depth
#
# so the vehicle must descend.
#
# At zero attitude:
#
# +z_B aligns with +z_W
#
# and +z points upward.
#
# Therefore downward force requires NEGATIVE body-z force.
#
# The generic PID remains sign-agnostic.
# The physical sign is introduced at the actuator boundary.
# ---------------------------------------------------------------------

depth_pid = NotebookPID(
    kp=0.55,
    ki=0.08,
    kd=0.65,
    output_limits=(-1.0, 1.0),
    integral_limits=(-4.0, 4.0),
)

max_vertical_force = 60.0


# ---------------------------------------------------------------------
# History
# ---------------------------------------------------------------------

pose_history = np.zeros(
    (steps + 1, 6),
    dtype=float,
)

velocity_history = np.zeros(
    (steps + 1, 6),
    dtype=float,
)

depth_history = np.zeros(
    steps + 1,
    dtype=float,
)

error_history = np.zeros(
    steps + 1,
    dtype=float,
)

command_history = np.zeros(
    steps + 1,
    dtype=float,
)

force_history = np.zeros(
    steps + 1,
    dtype=float,
)


# ---------------------------------------------------------------------
# Initial history
# ---------------------------------------------------------------------

pose_history[0] = pose

velocity_history[0] = (
    body_velocity
)

depth_history[0] = (
    ideal_depth_measurement(
        pose
    )
)

error_history[0] = (
    target_depth
    - depth_history[0]
)


# ---------------------------------------------------------------------
# Closed-loop simulation
# ---------------------------------------------------------------------

for k in range(steps):

    # -------------------------------------------------------------
    # Ideal depth feedback
    # -------------------------------------------------------------

    measured_depth = (
        ideal_depth_measurement(
            pose
        )
    )


    # -------------------------------------------------------------
    # Depth error
    # -------------------------------------------------------------

    depth_error = (
        target_depth
        - measured_depth
    )


    # -------------------------------------------------------------
    # PID update
    # -------------------------------------------------------------

    pid_output, pid_diagnostics = (
        depth_pid.update(
            error=depth_error,
            dt=dt,
        )
    )


    # -------------------------------------------------------------
    # Convert normalized control command into physical body-z force
    #
    # Positive depth error:
    #
    #     PID > 0
    #
    # Required physical motion:
    #
    #     downward
    #
    # Since +z_B points upward at zero attitude:
    #
    #     F_z < 0
    # -------------------------------------------------------------

    vertical_force = (
        -pid_output
        * max_vertical_force
    )


    # -------------------------------------------------------------
    # Generalized actuator wrench
    #
    # tau =
    #
    # [Fx, Fy, Fz, Mx, My, Mz]
    #
    # Only body-z force is commanded here.
    # -------------------------------------------------------------

    actuator_wrench = np.array(
        [
            0.0,
            0.0,
            vertical_force,
            0.0,
            0.0,
            0.0,
        ],
        dtype=float,
    )


    # -------------------------------------------------------------
    # Production 6-DOF integration
    # -------------------------------------------------------------

    (
        pose,
        body_velocity,
        plant_diagnostics,
    ) = integrate_six_dof_euler(
        pose,
        body_velocity,
        dt,

        mass=mass,

        inertia=inertia,

        added_inertia=(
            added_inertia
        ),

        linear_damping=(
            linear_damping
        ),

        quadratic_damping=(
            quadratic_damping
        ),

        fluid_density=(
            fluid_density
        ),

        displaced_volume=(
            displaced_volume
        ),

        center_of_gravity=(
            center_of_gravity
        ),

        center_of_buoyancy=(
            center_of_buoyancy
        ),

        current_velocity_world=(
            current_velocity_world
        ),

        actuator_wrench=(
            actuator_wrench
        ),

        gravity=gravity,
    )


    # -------------------------------------------------------------
    # Record physical response
    # -------------------------------------------------------------

    index = k + 1

    pose_history[index] = (
        pose
    )

    velocity_history[index] = (
        body_velocity
    )

    depth_history[index] = (
        ideal_depth_measurement(
            pose
        )
    )

    error_history[index] = (
        target_depth
        - depth_history[index]
    )

    command_history[index] = (
        pid_output
    )

    force_history[index] = (
        vertical_force
    )


# ---------------------------------------------------------------------
# Numerical metrics
# ---------------------------------------------------------------------

initial_depth = float(
    depth_history[0]
)

final_depth = float(
    depth_history[-1]
)

initial_error = abs(
    target_depth
    - initial_depth
)

final_error = abs(
    target_depth
    - final_depth
)

maximum_command = float(
    np.max(
        np.abs(
            command_history
        )
    )
)

minimum_depth = float(
    np.min(
        depth_history
    )
)

maximum_depth = float(
    np.max(
        depth_history
    )
)

maximum_vertical_speed = float(
    np.max(
        np.abs(
            velocity_history[:, 2]
        )
    )
)

final_vertical_speed = float(
    velocity_history[-1, 2]
)


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

assert np.all(
    np.isfinite(
        pose_history
    )
)

assert np.all(
    np.isfinite(
        velocity_history
    )
)

assert np.all(
    np.isfinite(
        depth_history
    )
)

assert np.all(
    np.isfinite(
        command_history
    )
)

assert np.all(
    np.isfinite(
        force_history
    )
)


# Controller output must respect normalized limits.
assert maximum_command <= (
    1.0 + 1e-12
)


# Error must decrease dramatically.
assert final_error < (
    0.10
    * initial_error
)


# Final depth regulation must be useful.
assert final_error < 0.25


# Vehicle must move substantially toward the deeper command.
assert final_depth > (
    initial_depth + 4.0
)


# Final motion should be close to settled.
assert abs(
    final_vertical_speed
) < 0.10


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)
print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 4 DEPTH CONTROL"
)
print("=" * 80)


print("\nPlant representation:")

print(
    f"  inertia shape:        "
    f"{inertia.shape}"
)

print(
    f"  added inertia shape:  "
    f"{added_inertia.shape}"
)


print("\nDepth response:")

print(
    f"  initial depth:        "
    f"{initial_depth:.6f} m"
)

print(
    f"  target depth:         "
    f"{target_depth:.6f} m"
)

print(
    f"  final depth:          "
    f"{final_depth:.6f} m"
)


print("\nTracking error:")

print(
    f"  initial abs. error:   "
    f"{initial_error:.6f} m"
)

print(
    f"  final abs. error:     "
    f"{final_error:.6f} m"
)

print(
    f"  error reduction:      "
    f"{100.0 * (1.0 - final_error / initial_error):.3f} %"
)


print("\nClosed-loop bounds:")

print(
    f"  minimum depth:        "
    f"{minimum_depth:.6f} m"
)

print(
    f"  maximum depth:        "
    f"{maximum_depth:.6f} m"
)

print(
    f"  maximum |command|:    "
    f"{maximum_command:.6f}"
)

print(
    f"  maximum |w|:          "
    f"{maximum_vertical_speed:.6f} m/s"
)

print(
    f"  final w:              "
    f"{final_vertical_speed:.6f} m/s"
)


print("\n" + "=" * 80)
print(
    "STEP 4 CLOSED-LOOP DEPTH CONTROL PASSED"
)
print("=" * 80)

print(
    "Depth PID acted through the production "
    "6-DOF actuator-wrench interface."
)

print(
    "Notebook 07 inertia conventions were "
    "respected exactly."
)

print(
    "The vehicle converged toward the commanded "
    "depth without direct state manipulation."
)

print(
    "All simulated states remained finite and "
    "controller output respected saturation."
)

print(
    "No production files were created or modified."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 4 DEPTH CONTROL

Plant representation:
  inertia shape:        (3,)
  added inertia shape:  (6,)

Depth response:
  initial depth:        2.000000 m
  target depth:         8.000000 m
  final depth:          8.040337 m

Tracking error:
  initial abs. error:   6.000000 m
  final abs. error:     0.040337 m
  error reduction:      99.328 %

Closed-loop bounds:
  minimum depth:        2.000000 m
  maximum depth:        8.623408 m
  maximum |command|:    1.000000
  maximum |w|:          2.258301 m/s
  final w:              0.007406 m/s

STEP 4 CLOSED-LOOP DEPTH CONTROL PASSED
Depth PID acted through the production 6-DOF actuator-wrench interface.
Notebook 07 inertia conventions were respected exactly.
The vehicle converged toward the commanded depth without direct state manipulation.
All simulated states remained finite and controller output respected saturation.
No production files were created or modified.


## Step 5 - Closed-Loop Heading Regulation with Circular Error

Depth regulation has demonstrated that the notebook-local PID controller can act successfully through the production six-degree-of-freedom plant.

The next isolated control problem is heading.

HydroLab-3D represents heading using yaw:

$$
\psi
$$

with the canonical interval:

$$
\psi
\in
(-\pi,\pi].
$$

Heading error must therefore be circular.

Instead of using naïve subtraction:

$$
\psi_{ref}-\psi
$$

the controller uses:

$$
e_{\psi}
=
\operatorname{wrap}_{(-\pi,\pi]}
\left(
\psi_{ref}-\psi
\right).
$$

The feedback loop is:

```text
desired heading
      ↓
ideal heading measurement
      ↓
wrapped heading error
      ↓
PID
      ↓
body-z yaw moment
      ↓
6-DOF rotational dynamics
      ↓
new heading
      ↺
```

As with depth regulation, the controller does not directly modify yaw or angular velocity.

Two deterministic cases are validated:

```text
Case A
0 deg → +90 deg

Case B
+170 deg → -170 deg
```

Case B is especially important.

The physical shortest rotation between these headings is approximately:

```text
+20 deg
```

not:

```text
-340 deg
```

so successful convergence verifies that the validated angle-wrapping convention is correctly integrated into the feedback loop.

### Validation Criteria

This step passes if:

1. both heading cases converge toward their references;
2. all heading errors use `wrap_angle_pi()`;
3. actuator commands remain inside `[-1, 1]`;
4. motion occurs only through the yaw component of `actuator_wrench`;
5. all states remain finite;
6. the boundary case begins with an approximately 20-degree wrapped error;
7. both final absolute heading errors are small;
8. final yaw rate is close to zero.

No production files are modified in this step.

In [6]:
import numpy as np

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)

from hydrolab3d.sensors import (
    ideal_heading_measurement,
    wrap_angle_pi,
)


# ---------------------------------------------------------------------
# Reuse canonical Step 4 plant parameters
# ---------------------------------------------------------------------

mass = 20.0

inertia = np.array(
    [
        1.2,
        1.6,
        2.0,
    ],
    dtype=float,
)

added_inertia = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ],
    dtype=float,
)

linear_damping = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ],
    dtype=float,
)

quadratic_damping = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ],
    dtype=float,
)

fluid_density = 1000.0
gravity = 9.81

displaced_volume = (
    mass / fluid_density
)

center_of_gravity = np.zeros(
    3,
    dtype=float,
)

center_of_buoyancy = np.zeros(
    3,
    dtype=float,
)

current_velocity_world = np.zeros(
    3,
    dtype=float,
)


# ---------------------------------------------------------------------
# Heading-control experiment helper
# ---------------------------------------------------------------------

def run_heading_control(
    initial_heading,
    target_heading,
    *,
    duration=12.0,
    dt=0.02,
):
    """
    Run one deterministic closed-loop heading-control experiment.

    Heading feedback uses the production ideal heading measurement and
    wrap_angle_pi() convention.
    """

    initial_heading = float(
        wrap_angle_pi(
            initial_heading
        )
    )

    target_heading = float(
        wrap_angle_pi(
            target_heading
        )
    )

    steps = int(
        duration / dt
    )

    times = (
        np.arange(
            steps + 1,
            dtype=float,
        )
        * dt
    )


    # -----------------------------------------------------------------
    # Initial physical state
    # -----------------------------------------------------------------

    pose = np.array(
        [
            0.0,
            0.0,
            -5.0,
            0.0,
            0.0,
            initial_heading,
        ],
        dtype=float,
    )

    body_velocity = np.zeros(
        6,
        dtype=float,
    )


    # -----------------------------------------------------------------
    # Heading PID
    #
    # Circular error is computed OUTSIDE the generic PID primitive.
    # -----------------------------------------------------------------

    heading_pid = NotebookPID(
        kp=0.80,
        ki=0.02,
        kd=0.80,
        output_limits=(-1.0, 1.0),
        integral_limits=(-2.0, 2.0),
    )

    max_yaw_moment = 3.0


    # -----------------------------------------------------------------
    # History
    # -----------------------------------------------------------------

    pose_history = np.zeros(
        (steps + 1, 6),
        dtype=float,
    )

    velocity_history = np.zeros(
        (steps + 1, 6),
        dtype=float,
    )

    heading_history = np.zeros(
        steps + 1,
        dtype=float,
    )

    error_history = np.zeros(
        steps + 1,
        dtype=float,
    )

    command_history = np.zeros(
        steps + 1,
        dtype=float,
    )

    moment_history = np.zeros(
        steps + 1,
        dtype=float,
    )


    # -----------------------------------------------------------------
    # Initial history
    # -----------------------------------------------------------------

    pose_history[0] = pose

    velocity_history[0] = (
        body_velocity
    )

    heading_history[0] = (
        ideal_heading_measurement(
            pose
        )
    )

    error_history[0] = float(
        wrap_angle_pi(
            target_heading
            - heading_history[0]
        )
    )


    # -----------------------------------------------------------------
    # Closed-loop simulation
    # -----------------------------------------------------------------

    for k in range(steps):

        measured_heading = (
            ideal_heading_measurement(
                pose
            )
        )

        heading_error = float(
            wrap_angle_pi(
                target_heading
                - measured_heading
            )
        )

        pid_output, pid_diagnostics = (
            heading_pid.update(
                error=heading_error,
                dt=dt,
            )
        )


        # -------------------------------------------------------------
        # Positive yaw error -> positive body-z moment
        # -------------------------------------------------------------

        yaw_moment = (
            pid_output
            * max_yaw_moment
        )


        # -------------------------------------------------------------
        # Generalized actuator wrench:
        #
        # [Fx, Fy, Fz, Mx, My, Mz]
        #
        # Only yaw moment is applied.
        # -------------------------------------------------------------

        actuator_wrench = np.array(
            [
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
                yaw_moment,
            ],
            dtype=float,
        )


        # -------------------------------------------------------------
        # Production 6-DOF integration
        # -------------------------------------------------------------

        (
            pose,
            body_velocity,
            plant_diagnostics,
        ) = integrate_six_dof_euler(
            pose,
            body_velocity,
            dt,

            mass=mass,

            inertia=inertia,

            added_inertia=(
                added_inertia
            ),

            linear_damping=(
                linear_damping
            ),

            quadratic_damping=(
                quadratic_damping
            ),

            fluid_density=(
                fluid_density
            ),

            displaced_volume=(
                displaced_volume
            ),

            center_of_gravity=(
                center_of_gravity
            ),

            center_of_buoyancy=(
                center_of_buoyancy
            ),

            current_velocity_world=(
                current_velocity_world
            ),

            actuator_wrench=(
                actuator_wrench
            ),

            gravity=gravity,
        )


        # -------------------------------------------------------------
        # Record response
        # -------------------------------------------------------------

        index = k + 1

        pose_history[index] = (
            pose
        )

        velocity_history[index] = (
            body_velocity
        )

        heading_history[index] = (
            ideal_heading_measurement(
                pose
            )
        )

        error_history[index] = float(
            wrap_angle_pi(
                target_heading
                - heading_history[index]
            )
        )

        command_history[index] = (
            pid_output
        )

        moment_history[index] = (
            yaw_moment
        )


    return {
        "times":
            times,

        "pose":
            pose_history,

        "velocity":
            velocity_history,

        "heading":
            heading_history,

        "error":
            error_history,

        "command":
            command_history,

        "moment":
            moment_history,

        "initial_heading":
            initial_heading,

        "target_heading":
            target_heading,
    }


# ---------------------------------------------------------------------
# Case A - Ordinary heading change
#
# 0 deg -> +90 deg
# ---------------------------------------------------------------------

case_a = run_heading_control(
    initial_heading=np.deg2rad(
        0.0
    ),
    target_heading=np.deg2rad(
        90.0
    ),
)


# ---------------------------------------------------------------------
# Case B - Boundary-crossing heading change
#
# +170 deg -> -170 deg
#
# Wrapped shortest-path error should be approximately +20 deg.
# ---------------------------------------------------------------------

case_b = run_heading_control(
    initial_heading=np.deg2rad(
        170.0
    ),
    target_heading=np.deg2rad(
        -170.0
    ),
)


# ---------------------------------------------------------------------
# Validation helper
# ---------------------------------------------------------------------

def heading_metrics(case):

    initial_error = abs(
        float(
            case["error"][0]
        )
    )

    final_error = abs(
        float(
            case["error"][-1]
        )
    )

    maximum_command = float(
        np.max(
            np.abs(
                case["command"]
            )
        )
    )

    maximum_yaw_rate = float(
        np.max(
            np.abs(
                case["velocity"][:, 5]
            )
        )
    )

    final_yaw_rate = float(
        case["velocity"][-1, 5]
    )

    return {
        "initial_error":
            initial_error,

        "final_error":
            final_error,

        "maximum_command":
            maximum_command,

        "maximum_yaw_rate":
            maximum_yaw_rate,

        "final_yaw_rate":
            final_yaw_rate,
    }


metrics_a = heading_metrics(
    case_a
)

metrics_b = heading_metrics(
    case_b
)


# ---------------------------------------------------------------------
# Numerical validation
# ---------------------------------------------------------------------

for case in (
    case_a,
    case_b,
):

    assert np.all(
        np.isfinite(
            case["pose"]
        )
    )

    assert np.all(
        np.isfinite(
            case["velocity"]
        )
    )

    assert np.all(
        np.isfinite(
            case["heading"]
        )
    )

    assert np.all(
        np.isfinite(
            case["error"]
        )
    )

    assert np.max(
        np.abs(
            case["command"]
        )
    ) <= (
        1.0 + 1e-12
    )


# ---------------------------------------------------------------------
# Case A convergence
# ---------------------------------------------------------------------

assert metrics_a["final_error"] < (
    np.deg2rad(
        3.0
    )
)

assert metrics_a["final_error"] < (
    0.10
    * metrics_a["initial_error"]
)

assert abs(
    metrics_a["final_yaw_rate"]
) < 0.05


# ---------------------------------------------------------------------
# Case B wrapped-error correctness
# ---------------------------------------------------------------------

expected_boundary_error = (
    np.deg2rad(
        20.0
    )
)

assert np.isclose(
    case_b["error"][0],
    expected_boundary_error,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Case B convergence
# ---------------------------------------------------------------------

assert metrics_b["final_error"] < (
    np.deg2rad(
        3.0
    )
)

assert metrics_b["final_error"] < (
    0.20
    * metrics_b["initial_error"]
)

assert abs(
    metrics_b["final_yaw_rate"]
) < 0.05


# ---------------------------------------------------------------------
# Report helper
# ---------------------------------------------------------------------

def print_heading_report(
    name,
    case,
    metrics,
):

    print(name)

    print(
        f"  initial heading:      "
        f"{np.rad2deg(case['initial_heading']):.6f} deg"
    )

    print(
        f"  target heading:       "
        f"{np.rad2deg(case['target_heading']):.6f} deg"
    )

    print(
        f"  initial wrapped error:"
        f" {np.rad2deg(case['error'][0]):.6f} deg"
    )

    print(
        f"  final heading:        "
        f"{np.rad2deg(case['heading'][-1]):.6f} deg"
    )

    print(
        f"  final abs. error:     "
        f"{np.rad2deg(metrics['final_error']):.6f} deg"
    )

    print(
        f"  maximum |command|:    "
        f"{metrics['maximum_command']:.6f}"
    )

    print(
        f"  maximum |r|:          "
        f"{metrics['maximum_yaw_rate']:.6f} rad/s"
    )

    print(
        f"  final r:              "
        f"{metrics['final_yaw_rate']:.6f} rad/s"
    )


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)
print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 5 HEADING CONTROL"
)
print("=" * 80)

print()

print_heading_report(
    "Case A - Ordinary +90 deg command",
    case_a,
    metrics_a,
)

print()

print_heading_report(
    "Case B - +170 deg to -170 deg boundary crossing",
    case_b,
    metrics_b,
)


print("\n" + "=" * 80)
print(
    "STEP 5 CLOSED-LOOP HEADING CONTROL PASSED"
)
print("=" * 80)

print(
    "Heading PID acted through the production "
    "6-DOF yaw-moment interface."
)

print(
    "Heading error used the validated "
    "wrap_angle_pi() convention."
)

print(
    "The +170 deg to -170 deg case correctly began "
    "with a +20 deg shortest-path error."
)

print(
    "Both heading cases converged without direct "
    "orientation-state manipulation."
)

print(
    "No production files were created or modified."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 5 HEADING CONTROL

Case A - Ordinary +90 deg command
  initial heading:      0.000000 deg
  target heading:       90.000000 deg
  initial wrapped error: 90.000000 deg
  final heading:        92.116097 deg
  final abs. error:     2.116097 deg
  maximum |command|:    1.000000
  maximum |r|:          0.797766 rad/s
  final r:              -0.002025 rad/s

Case B - +170 deg to -170 deg boundary crossing
  initial heading:      170.000000 deg
  target heading:       -170.000000 deg
  initial wrapped error: 20.000000 deg
  final heading:        -169.527045 deg
  final abs. error:     0.472955 deg
  maximum |command|:    0.279532
  maximum |r|:          0.187053 rad/s
  final r:              -0.000606 rad/s

STEP 5 CLOSED-LOOP HEADING CONTROL PASSED
Heading PID acted through the production 6-DOF yaw-moment interface.
Heading error used the validated wrap_angle_pi() convention.
The +170 deg to -170 deg case correctly began with a +20 deg shortest-path error.
Both

## Step 6 - Notebook-Local Waypoint Guidance Geometry

Depth and heading regulation have now been validated independently.

The next requirement is a minimal guidance layer that converts a spatial waypoint into references that those controllers can track.

For vehicle position:

$$
\mathbf p
=
\begin{bmatrix}
x\\
y\\
z
\end{bmatrix}
$$

and waypoint:

$$
\mathbf p_g
=
\begin{bmatrix}
x_g\\
y_g\\
z_g
\end{bmatrix},
$$

the displacement is:

$$
\Delta \mathbf p
=
\mathbf p_g-\mathbf p.
$$

The horizontal displacement is:

$$
\Delta x=x_g-x
$$

$$
\Delta y=y_g-y.
$$

The desired heading is:

$$
\psi_{ref}
=
\operatorname{atan2}
\left(
\Delta y,\Delta x
\right).
$$

Because HydroLab-3D uses:

$$
d=-z,
$$

the waypoint depth reference is:

$$
d_{ref}=-z_g.
$$

The Euclidean waypoint distance is:

$$
r
=
\left\|
\mathbf p_g-\mathbf p
\right\|.
$$

The horizontal distance is:

$$
r_{xy}
=
\sqrt{
(\Delta x)^2
+
(\Delta y)^2
}.
$$

The guidance layer therefore performs:

```text
current position + waypoint
          ↓
   geometric displacement
          ↓
 ┌────────┴────────┐
 ↓                 ↓
desired heading   desired depth
          ↓
     control layer
```

This step does not command actuators and does not move the vehicle.

Its purpose is to validate waypoint-reference generation independently before it is connected to the closed-loop plant.

### Important Localization Assumption

The function accepts world-frame vehicle position directly.

For Notebook 09 this represents a temporary perfect-localization baseline.

It does **not** imply that DVL, depth, and heading measurements provide perfect global XYZ localization.

A future localization or state-estimation layer may replace this position source without changing the waypoint-guidance mathematics.

### Validation Cases

We validate:

1. waypoint directly east;
2. waypoint directly north;
3. waypoint in the southwest quadrant;
4. waypoint at greater depth;
5. waypoint at shallower depth;
6. zero horizontal displacement;
7. Euclidean and horizontal distance calculations;
8. invalid input rejection.

No production files are modified in this step.

In [7]:
import numpy as np


# ---------------------------------------------------------------------
# Notebook-local waypoint guidance prototype
# ---------------------------------------------------------------------

def waypoint_guidance(
    position_world,
    waypoint_world,
):
    """
    Convert a world-frame waypoint into simple navigation references.

    Parameters
    ----------
    position_world : array-like, shape (3,)
        Current world-frame position [x, y, z].

    waypoint_world : array-like, shape (3,)
        Desired world-frame waypoint [x_g, y_g, z_g].

    Returns
    -------
    guidance : dict

        displacement_world
            [dx, dy, dz]

        desired_heading
            atan2(dy, dx), wrapped naturally to [-pi, pi]

        desired_depth
            -z_goal

        horizontal_distance
            sqrt(dx^2 + dy^2)

        distance
            Euclidean 3D waypoint distance
    """

    position_world = np.asarray(
        position_world,
        dtype=float,
    )

    waypoint_world = np.asarray(
        waypoint_world,
        dtype=float,
    )


    # -----------------------------------------------------------------
    # Input validation
    # -----------------------------------------------------------------

    if position_world.shape != (3,):

        raise ValueError(
            "position_world must have shape (3,)"
        )

    if waypoint_world.shape != (3,):

        raise ValueError(
            "waypoint_world must have shape (3,)"
        )

    if not np.all(
        np.isfinite(
            position_world
        )
    ):

        raise ValueError(
            "position_world must contain only finite values"
        )

    if not np.all(
        np.isfinite(
            waypoint_world
        )
    ):

        raise ValueError(
            "waypoint_world must contain only finite values"
        )


    # -----------------------------------------------------------------
    # Relative geometry
    # -----------------------------------------------------------------

    displacement_world = (
        waypoint_world
        - position_world
    )

    dx = float(
        displacement_world[0]
    )

    dy = float(
        displacement_world[1]
    )


    # -----------------------------------------------------------------
    # Heading reference
    # -----------------------------------------------------------------

    desired_heading = float(
        np.arctan2(
            dy,
            dx,
        )
    )


    # -----------------------------------------------------------------
    # Depth reference
    #
    # HydroLab convention:
    #
    # depth = -z
    # -----------------------------------------------------------------

    desired_depth = float(
        -waypoint_world[2]
    )


    # -----------------------------------------------------------------
    # Distances
    # -----------------------------------------------------------------

    horizontal_distance = float(
        np.hypot(
            dx,
            dy,
        )
    )

    distance = float(
        np.linalg.norm(
            displacement_world
        )
    )


    return {
        "displacement_world":
            displacement_world,

        "desired_heading":
            desired_heading,

        "desired_depth":
            desired_depth,

        "horizontal_distance":
            horizontal_distance,

        "distance":
            distance,
    }


# ---------------------------------------------------------------------
# Test 1 - East
# ---------------------------------------------------------------------

east = waypoint_guidance(
    position_world=[
        0.0,
        0.0,
        -5.0,
    ],
    waypoint_world=[
        10.0,
        0.0,
        -5.0,
    ],
)

assert np.isclose(
    east["desired_heading"],
    0.0,
)

assert np.isclose(
    east["desired_depth"],
    5.0,
)

assert np.isclose(
    east["horizontal_distance"],
    10.0,
)

assert np.isclose(
    east["distance"],
    10.0,
)

print(
    "Test 1 - East waypoint: PASS"
)


# ---------------------------------------------------------------------
# Test 2 - North
# ---------------------------------------------------------------------

north = waypoint_guidance(
    position_world=[
        0.0,
        0.0,
        -5.0,
    ],
    waypoint_world=[
        0.0,
        10.0,
        -5.0,
    ],
)

assert np.isclose(
    north["desired_heading"],
    np.pi / 2.0,
)

print(
    "Test 2 - North waypoint: PASS"
)


# ---------------------------------------------------------------------
# Test 3 - Southwest
# ---------------------------------------------------------------------

southwest = waypoint_guidance(
    position_world=[
        0.0,
        0.0,
        -5.0,
    ],
    waypoint_world=[
        -10.0,
        -10.0,
        -5.0,
    ],
)

expected_sw_heading = (
    -3.0 * np.pi / 4.0
)

assert np.isclose(
    southwest["desired_heading"],
    expected_sw_heading,
)

assert np.isclose(
    southwest["horizontal_distance"],
    np.sqrt(
        200.0
    ),
)

print(
    "Test 3 - Southwest waypoint: PASS"
)


# ---------------------------------------------------------------------
# Test 4 - Deeper waypoint
# ---------------------------------------------------------------------

deeper = waypoint_guidance(
    position_world=[
        2.0,
        3.0,
        -4.0,
    ],
    waypoint_world=[
        2.0,
        3.0,
        -9.0,
    ],
)

assert np.isclose(
    deeper["desired_depth"],
    9.0,
)

assert np.isclose(
    deeper["horizontal_distance"],
    0.0,
)

assert np.isclose(
    deeper["distance"],
    5.0,
)

print(
    "Test 4 - Deeper waypoint: PASS"
)


# ---------------------------------------------------------------------
# Test 5 - Shallower waypoint
# ---------------------------------------------------------------------

shallower = waypoint_guidance(
    position_world=[
        2.0,
        3.0,
        -9.0,
    ],
    waypoint_world=[
        2.0,
        3.0,
        -3.0,
    ],
)

assert np.isclose(
    shallower["desired_depth"],
    3.0,
)

assert np.isclose(
    shallower["distance"],
    6.0,
)

print(
    "Test 5 - Shallower waypoint: PASS"
)


# ---------------------------------------------------------------------
# Test 6 - Zero horizontal displacement
#
# atan2(0, 0) evaluates to 0.0 in NumPy.
#
# This value is acceptable here because heading is irrelevant when the
# waypoint has no horizontal displacement.
# ---------------------------------------------------------------------

vertical_only = waypoint_guidance(
    position_world=[
        4.0,
        -2.0,
        -5.0,
    ],
    waypoint_world=[
        4.0,
        -2.0,
        -8.0,
    ],
)

assert np.isclose(
    vertical_only["horizontal_distance"],
    0.0,
)

assert np.isclose(
    vertical_only["desired_heading"],
    0.0,
)

print(
    "Test 6 - Zero horizontal displacement: PASS"
)


# ---------------------------------------------------------------------
# Test 7 - General 3D geometry
# ---------------------------------------------------------------------

general = waypoint_guidance(
    position_world=[
        1.0,
        2.0,
        -3.0,
    ],
    waypoint_world=[
        4.0,
        6.0,
        -15.0,
    ],
)

expected_displacement = np.array(
    [
        3.0,
        4.0,
        -12.0,
    ]
)

assert np.allclose(
    general["displacement_world"],
    expected_displacement,
)

assert np.isclose(
    general["horizontal_distance"],
    5.0,
)

assert np.isclose(
    general["distance"],
    13.0,
)

assert np.isclose(
    general["desired_depth"],
    15.0,
)

assert np.isclose(
    general["desired_heading"],
    np.arctan2(
        4.0,
        3.0,
    ),
)

print(
    "Test 7 - General 3D geometry: PASS"
)


# ---------------------------------------------------------------------
# Test 8 - Invalid input rejection
# ---------------------------------------------------------------------

bad_shape_rejected = False

try:

    waypoint_guidance(
        position_world=[
            0.0,
            0.0,
        ],
        waypoint_world=[
            1.0,
            2.0,
            3.0,
        ],
    )

except ValueError:

    bad_shape_rejected = True


assert bad_shape_rejected


nonfinite_rejected = False

try:

    waypoint_guidance(
        position_world=[
            0.0,
            0.0,
            -5.0,
        ],
        waypoint_world=[
            np.nan,
            2.0,
            -5.0,
        ],
    )

except ValueError:

    nonfinite_rejected = True


assert nonfinite_rejected

print(
    "Test 8 - Invalid input rejection: PASS"
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("\n" + "=" * 80)

print(
    "STEP 6 WAYPOINT GUIDANCE "
    "GEOMETRY PASSED"
)

print("=" * 80)

print(
    "World-frame waypoint geometry was converted "
    "into desired heading, desired depth, and "
    "distance references correctly."
)

print(
    "No actuator commands or physical state "
    "updates were performed."
)

print(
    "Global position is explicitly treated as a "
    "temporary perfect-localization baseline."
)

print(
    "No production files were created or modified."
)

Test 1 - East waypoint: PASS
Test 2 - North waypoint: PASS
Test 3 - Southwest waypoint: PASS
Test 4 - Deeper waypoint: PASS
Test 5 - Shallower waypoint: PASS
Test 6 - Zero horizontal displacement: PASS
Test 7 - General 3D geometry: PASS
Test 8 - Invalid input rejection: PASS

STEP 6 WAYPOINT GUIDANCE GEOMETRY PASSED
World-frame waypoint geometry was converted into desired heading, desired depth, and distance references correctly.
No actuator commands or physical state updates were performed.
Global position is explicitly treated as a temporary perfect-localization baseline.
No production files were created or modified.


## Step 7 - Closed-Loop 3D Waypoint Navigation Through the 6-DOF Plant

The isolated waypoint-guidance, depth-control, and heading-control components have each been validated independently.

We can now combine them into the first autonomous 3D waypoint-navigation loop.

The initial combined experiment also exposed an important six-degree-of-freedom requirement.

With:

```text
center of gravity = center of buoyancy
```

the vehicle possessed no hydrostatic roll/pitch restoring lever.

Simultaneous surge, heave, and yaw motion can excite coupled rotational dynamics. The navigation vehicle therefore uses a small physically meaningful vertical separation between its center of gravity and center of buoyancy:

```text
CG below CB
```

while retaining neutral net buoyancy.

The resulting architecture is:

```text
world-frame waypoint
        ↓
waypoint guidance
        ↓
desired heading + desired depth
        ↓
heading PID + depth PID
        ↓
forward force + vertical force + yaw moment
        ↓
production 6-DOF plant
        ↓
updated physical state
        ↺
```

At every timestep:

1. world position is used as the temporary perfect-localization baseline;
2. waypoint guidance generates heading and depth references;
3. ideal heading and depth sensors provide feedback;
4. wrapped heading error drives the yaw controller;
5. depth error drives the vertical controller;
6. forward propulsion is reduced when heading alignment is poor;
7. the complete generalized actuator wrench is passed to the production plant.

No state variable is directly modified to force waypoint convergence.

### Vehicle Stability Assumption

The vehicle remains neutrally buoyant:

$$
F_B=F_G
$$

but uses:

$$
\mathbf r_G
\neq
\mathbf r_B.
$$

The center of buoyancy is slightly above the center of gravity, producing passive restoring roll and pitch moments.

### Attitude Validation

The earlier trial used an arbitrary requirement:

```text
|pitch| < 20 deg
```

That threshold is not a fundamental validity condition of the HydroLab-3D plant.

The production kinematic model instead has a genuine mathematical limitation at the Z-Y-X Euler singularity:

$$
\theta=\pm\frac{\pi}{2}.
$$

The corrected validation therefore checks that the trajectory:

- remains finite;
- remains underwater;
- retains a clear numerical margin from the Euler pitch singularity.

The singularity margin is evaluated using:

$$
|\cos\theta|.
$$

At the singularity:

$$
|\cos\theta|=0.
$$

The navigation trajectory must maintain:

$$
|\cos\theta|>0.1
$$

throughout the experiment.

Roll and pitch excursions are still measured and reported, but an arbitrary 20-degree ceiling is no longer used as a pass/fail criterion.

### Localization Assumption

World-frame position remains a temporary perfect-localization baseline.

This does not represent an implemented underwater localization system.

Depth and heading feedback continue to pass through the validated sensor interfaces.

### Validation Criteria

The experiment passes if:

1. the vehicle starts substantially away from the waypoint;
2. waypoint distance decreases strongly;
3. the vehicle enters the defined success radius;
4. all actuator commands remain bounded;
5. all physical states remain finite;
6. the vehicle remains below the modeled water surface;
7. the trajectory remains safely separated from the Euler pitch singularity;
8. heading and depth references are continuously generated from waypoint geometry;
9. all motion occurs through the generalized actuator wrench;
10. no state variables are directly overwritten to manufacture convergence.

No production files are modified in this step.

In [11]:
import numpy as np

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)

from hydrolab3d.sensors import (
    ideal_depth_measurement,
    ideal_heading_measurement,
    wrap_angle_pi,
)


# ---------------------------------------------------------------------
# Canonical 6-DOF plant parameters
# ---------------------------------------------------------------------

mass = 20.0

inertia = np.array(
    [
        1.2,
        1.6,
        2.0,
    ],
    dtype=float,
)

added_inertia = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ],
    dtype=float,
)

linear_damping = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ],
    dtype=float,
)

quadratic_damping = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ],
    dtype=float,
)

fluid_density = 1000.0
gravity = 9.81

displaced_volume = (
    mass / fluid_density
)


# ---------------------------------------------------------------------
# Passive attitude stability
#
# Neutral buoyancy is preserved:
#
# rho * V = mass
#
# but CB lies slightly above CG.
# ---------------------------------------------------------------------

center_of_gravity = np.array(
    [
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

center_of_buoyancy = np.array(
    [
        0.0,
        0.0,
        0.08,
    ],
    dtype=float,
)

current_velocity_world = np.zeros(
    3,
    dtype=float,
)


# ---------------------------------------------------------------------
# Simulation setup
# ---------------------------------------------------------------------

dt = 0.02
duration = 40.0

steps = int(
    duration / dt
)

times = (
    np.arange(
        steps + 1,
        dtype=float,
    )
    * dt
)


# ---------------------------------------------------------------------
# Initial physical state
# ---------------------------------------------------------------------

pose = np.array(
    [
        0.0,
        0.0,
        -2.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

body_velocity = np.zeros(
    6,
    dtype=float,
)


# ---------------------------------------------------------------------
# Target waypoint
# ---------------------------------------------------------------------

waypoint_world = np.array(
    [
        20.0,
        12.0,
        -8.0,
    ],
    dtype=float,
)

success_radius = 1.0


# ---------------------------------------------------------------------
# Controllers
# ---------------------------------------------------------------------

depth_pid = NotebookPID(
    kp=0.55,
    ki=0.08,
    kd=0.65,
    output_limits=(-1.0, 1.0),
    integral_limits=(-4.0, 4.0),
)

heading_pid = NotebookPID(
    kp=0.80,
    ki=0.02,
    kd=0.80,
    output_limits=(-1.0, 1.0),
    integral_limits=(-2.0, 2.0),
)


# ---------------------------------------------------------------------
# Actuator authority
# ---------------------------------------------------------------------

max_forward_force = 25.0
max_vertical_force = 60.0
max_yaw_moment = 3.0


# ---------------------------------------------------------------------
# Forward-guidance gain
# ---------------------------------------------------------------------

forward_distance_gain = 0.10


# ---------------------------------------------------------------------
# Histories
# ---------------------------------------------------------------------

pose_history = np.zeros(
    (steps + 1, 6),
    dtype=float,
)

velocity_history = np.zeros(
    (steps + 1, 6),
    dtype=float,
)

distance_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

horizontal_distance_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

desired_heading_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

desired_depth_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

heading_error_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

depth_error_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

forward_command_history = np.zeros(
    steps + 1,
    dtype=float,
)

heading_command_history = np.zeros(
    steps + 1,
    dtype=float,
)

depth_command_history = np.zeros(
    steps + 1,
    dtype=float,
)


# ---------------------------------------------------------------------
# Initial guidance and feedback
# ---------------------------------------------------------------------

initial_guidance = waypoint_guidance(
    position_world=pose[:3],
    waypoint_world=waypoint_world,
)

initial_heading = float(
    ideal_heading_measurement(
        pose
    )
)

initial_depth = float(
    ideal_depth_measurement(
        pose
    )
)

pose_history[0] = pose

velocity_history[0] = (
    body_velocity
)

distance_history[0] = (
    initial_guidance["distance"]
)

horizontal_distance_history[0] = (
    initial_guidance[
        "horizontal_distance"
    ]
)

desired_heading_history[0] = (
    initial_guidance[
        "desired_heading"
    ]
)

desired_depth_history[0] = (
    initial_guidance[
        "desired_depth"
    ]
)

heading_error_history[0] = float(
    wrap_angle_pi(
        initial_guidance[
            "desired_heading"
        ]
        - initial_heading
    )
)

depth_error_history[0] = (
    initial_guidance[
        "desired_depth"
    ]
    - initial_depth
)


# ---------------------------------------------------------------------
# Navigation state
# ---------------------------------------------------------------------

reached_waypoint = False
final_index = 0


# ---------------------------------------------------------------------
# Closed-loop navigation
# ---------------------------------------------------------------------

for k in range(steps):

    # -------------------------------------------------------------
    # Temporary perfect-localization baseline
    # -------------------------------------------------------------

    position_world = (
        pose[:3].copy()
    )


    # -------------------------------------------------------------
    # Guidance
    # -------------------------------------------------------------

    guidance = waypoint_guidance(
        position_world=position_world,
        waypoint_world=waypoint_world,
    )

    desired_heading = float(
        guidance[
            "desired_heading"
        ]
    )

    desired_depth = float(
        guidance[
            "desired_depth"
        ]
    )

    waypoint_distance = float(
        guidance[
            "distance"
        ]
    )

    horizontal_distance = float(
        guidance[
            "horizontal_distance"
        ]
    )


    # -------------------------------------------------------------
    # Success condition
    # -------------------------------------------------------------

    if (
        waypoint_distance
        <= success_radius
    ):

        reached_waypoint = True
        final_index = k

        break


    # -------------------------------------------------------------
    # Sensor feedback
    # -------------------------------------------------------------

    measured_heading = float(
        ideal_heading_measurement(
            pose
        )
    )

    measured_depth = float(
        ideal_depth_measurement(
            pose
        )
    )


    # -------------------------------------------------------------
    # Wrapped heading error
    # -------------------------------------------------------------

    heading_error = float(
        wrap_angle_pi(
            desired_heading
            - measured_heading
        )
    )


    # -------------------------------------------------------------
    # Depth error
    # -------------------------------------------------------------

    depth_error = (
        desired_depth
        - measured_depth
    )


    # -------------------------------------------------------------
    # Heading PID
    # -------------------------------------------------------------

    (
        heading_command,
        heading_diagnostics,
    ) = heading_pid.update(
        error=heading_error,
        dt=dt,
    )


    # -------------------------------------------------------------
    # Depth PID
    # -------------------------------------------------------------

    (
        depth_command,
        depth_diagnostics,
    ) = depth_pid.update(
        error=depth_error,
        dt=dt,
    )


    # -------------------------------------------------------------
    # Forward propulsion
    # -------------------------------------------------------------

    distance_command = float(
        np.clip(
            forward_distance_gain
            * horizontal_distance,
            0.0,
            1.0,
        )
    )

    heading_alignment = float(
        np.clip(
            np.cos(
                heading_error
            ),
            0.0,
            1.0,
        )
    )

    forward_command = (
        distance_command
        * heading_alignment
    )


    # -------------------------------------------------------------
    # Physical actuator wrench
    # -------------------------------------------------------------

    forward_force = (
        forward_command
        * max_forward_force
    )

    vertical_force = (
        -depth_command
        * max_vertical_force
    )

    yaw_moment = (
        heading_command
        * max_yaw_moment
    )

    actuator_wrench = np.array(
        [
            forward_force,
            0.0,
            vertical_force,
            0.0,
            0.0,
            yaw_moment,
        ],
        dtype=float,
    )


    # -------------------------------------------------------------
    # Production 6-DOF integration
    # -------------------------------------------------------------

    (
        pose,
        body_velocity,
        plant_diagnostics,
    ) = integrate_six_dof_euler(
        pose,
        body_velocity,
        dt,

        mass=mass,

        inertia=inertia,

        added_inertia=(
            added_inertia
        ),

        linear_damping=(
            linear_damping
        ),

        quadratic_damping=(
            quadratic_damping
        ),

        fluid_density=(
            fluid_density
        ),

        displaced_volume=(
            displaced_volume
        ),

        center_of_gravity=(
            center_of_gravity
        ),

        center_of_buoyancy=(
            center_of_buoyancy
        ),

        current_velocity_world=(
            current_velocity_world
        ),

        actuator_wrench=(
            actuator_wrench
        ),

        gravity=gravity,
    )


    # -------------------------------------------------------------
    # Numerical sanity
    # -------------------------------------------------------------

    if not np.all(
        np.isfinite(
            pose
        )
    ):
        raise RuntimeError(
            "Waypoint navigation produced "
            "a non-finite pose."
        )

    if not np.all(
        np.isfinite(
            body_velocity
        )
    ):
        raise RuntimeError(
            "Waypoint navigation produced "
            "a non-finite velocity."
        )


    # -------------------------------------------------------------
    # Physical surface check
    #
    # No clipping is performed.
    # -------------------------------------------------------------

    if pose[2] > 1e-9:

        raise RuntimeError(
            "Waypoint navigation crossed above "
            "the modeled water surface at "
            f"t={(k + 1) * dt:.3f} s: "
            f"z={pose[2]:.6f} m"
        )


    # -------------------------------------------------------------
    # Store physical state
    # -------------------------------------------------------------

    index = k + 1
    final_index = index

    pose_history[index] = (
        pose
    )

    velocity_history[index] = (
        body_velocity
    )


    # -------------------------------------------------------------
    # Updated guidance and measurements
    # -------------------------------------------------------------

    updated_guidance = (
        waypoint_guidance(
            position_world=pose[:3],
            waypoint_world=waypoint_world,
        )
    )

    updated_heading = float(
        ideal_heading_measurement(
            pose
        )
    )

    updated_depth = float(
        ideal_depth_measurement(
            pose
        )
    )


    # -------------------------------------------------------------
    # Store feedback histories
    # -------------------------------------------------------------

    distance_history[index] = (
        updated_guidance[
            "distance"
        ]
    )

    horizontal_distance_history[index] = (
        updated_guidance[
            "horizontal_distance"
        ]
    )

    desired_heading_history[index] = (
        updated_guidance[
            "desired_heading"
        ]
    )

    desired_depth_history[index] = (
        updated_guidance[
            "desired_depth"
        ]
    )

    heading_error_history[index] = float(
        wrap_angle_pi(
            updated_guidance[
                "desired_heading"
            ]
            - updated_heading
        )
    )

    depth_error_history[index] = (
        updated_guidance[
            "desired_depth"
        ]
        - updated_depth
    )

    forward_command_history[index] = (
        forward_command
    )

    heading_command_history[index] = (
        heading_command
    )

    depth_command_history[index] = (
        depth_command
    )


# ---------------------------------------------------------------------
# Final success check
# ---------------------------------------------------------------------

if not reached_waypoint:

    final_guidance = (
        waypoint_guidance(
            position_world=pose[:3],
            waypoint_world=waypoint_world,
        )
    )

    if (
        final_guidance[
            "distance"
        ]
        <= success_radius
    ):

        reached_waypoint = True


# ---------------------------------------------------------------------
# Trim histories
# ---------------------------------------------------------------------

used = slice(
    0,
    final_index + 1,
)

times_nav = (
    times[used]
)

pose_nav = (
    pose_history[used]
)

velocity_nav = (
    velocity_history[used]
)

distance_nav = (
    distance_history[used]
)

horizontal_distance_nav = (
    horizontal_distance_history[
        used
    ]
)

desired_heading_nav = (
    desired_heading_history[
        used
    ]
)

desired_depth_nav = (
    desired_depth_history[
        used
    ]
)

heading_error_nav = (
    heading_error_history[
        used
    ]
)

depth_error_nav = (
    depth_error_history[
        used
    ]
)

forward_command_nav = (
    forward_command_history[
        used
    ]
)

heading_command_nav = (
    heading_command_history[
        used
    ]
)

depth_command_nav = (
    depth_command_history[
        used
    ]
)


# ---------------------------------------------------------------------
# Navigation metrics
# ---------------------------------------------------------------------

initial_distance = float(
    distance_nav[0]
)

final_distance = float(
    distance_nav[-1]
)

final_position = (
    pose_nav[-1, :3].copy()
)

minimum_depth = float(
    np.min(
        -pose_nav[:, 2]
    )
)

maximum_depth = float(
    np.max(
        -pose_nav[:, 2]
    )
)

maximum_abs_roll = float(
    np.max(
        np.abs(
            pose_nav[:, 3]
        )
    )
)

maximum_abs_pitch = float(
    np.max(
        np.abs(
            pose_nav[:, 4]
        )
    )
)

final_roll = float(
    pose_nav[-1, 3]
)

final_pitch = float(
    pose_nav[-1, 4]
)


# ---------------------------------------------------------------------
# Euler singularity margin
#
# Z-Y-X Euler-rate transformation becomes singular when:
#
# cos(pitch) = 0
#
# A minimum absolute cosine above 0.1 provides a clear margin.
# ---------------------------------------------------------------------

abs_cos_pitch = np.abs(
    np.cos(
        pose_nav[:, 4]
    )
)

minimum_abs_cos_pitch = float(
    np.min(
        abs_cos_pitch
    )
)

minimum_singularity_margin_deg = float(
    np.rad2deg(
        np.arcsin(
            np.clip(
                minimum_abs_cos_pitch,
                0.0,
                1.0,
            )
        )
    )
)


# ---------------------------------------------------------------------
# Command metrics
# ---------------------------------------------------------------------

maximum_forward_command = float(
    np.max(
        np.abs(
            forward_command_nav
        )
    )
)

maximum_heading_command = float(
    np.max(
        np.abs(
            heading_command_nav
        )
    )
)

maximum_depth_command = float(
    np.max(
        np.abs(
            depth_command_nav
        )
    )
)


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

assert np.all(
    np.isfinite(
        pose_nav
    )
)

assert np.all(
    np.isfinite(
        velocity_nav
    )
)

assert np.all(
    np.isfinite(
        distance_nav
    )
)


# Vehicle must remain underwater.
assert np.all(
    pose_nav[:, 2]
    <= 1e-9
)


# Normalized actuator commands must remain bounded.
assert maximum_forward_command <= (
    1.0 + 1e-12
)

assert maximum_heading_command <= (
    1.0 + 1e-12
)

assert maximum_depth_command <= (
    1.0 + 1e-12
)


# Waypoint distance must fall dramatically.
assert final_distance < (
    0.10
    * initial_distance
)


# Success region must be reached.
assert final_distance <= (
    success_radius
)

assert reached_waypoint


# ---------------------------------------------------------------------
# Corrected attitude-validity criterion
#
# Do NOT impose an arbitrary 20-degree pitch ceiling.
#
# Instead verify that the entire trajectory stays comfortably away
# from the genuine Z-Y-X Euler singularity.
# ---------------------------------------------------------------------

assert minimum_abs_cos_pitch > 0.10


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)

print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 7 WAYPOINT NAVIGATION"
)

print("=" * 80)


print("\nVehicle stability configuration:")

print(
    f"  center of gravity:     "
    f"{center_of_gravity}"
)

print(
    f"  center of buoyancy:    "
    f"{center_of_buoyancy}"
)

print(
    f"  CB-CG vertical offset: "
    f"{center_of_buoyancy[2] - center_of_gravity[2]:.6f} m"
)


print("\nWaypoint geometry:")

print(
    f"  start position:        "
    f"{pose_nav[0, :3]}"
)

print(
    f"  target waypoint:       "
    f"{waypoint_world}"
)

print(
    f"  final position:        "
    f"{final_position}"
)


print("\nNavigation performance:")

print(
    f"  initial distance:      "
    f"{initial_distance:.6f} m"
)

print(
    f"  final distance:        "
    f"{final_distance:.6f} m"
)

print(
    f"  distance reduction:    "
    f"{100.0 * (1.0 - final_distance / initial_distance):.3f} %"
)

print(
    f"  success radius:        "
    f"{success_radius:.6f} m"
)

print(
    f"  reached waypoint:      "
    f"{reached_waypoint}"
)

print(
    f"  navigation time:       "
    f"{times_nav[-1]:.6f} s"
)


print("\nDepth bounds:")

print(
    f"  minimum depth:         "
    f"{minimum_depth:.6f} m"
)

print(
    f"  maximum depth:         "
    f"{maximum_depth:.6f} m"
)


print("\nAttitude diagnostics:")

print(
    f"  maximum |roll|:        "
    f"{np.rad2deg(maximum_abs_roll):.6f} deg"
)

print(
    f"  maximum |pitch|:       "
    f"{np.rad2deg(maximum_abs_pitch):.6f} deg"
)

print(
    f"  final roll:            "
    f"{np.rad2deg(final_roll):.6f} deg"
)

print(
    f"  final pitch:           "
    f"{np.rad2deg(final_pitch):.6f} deg"
)

print(
    f"  minimum |cos(pitch)|:  "
    f"{minimum_abs_cos_pitch:.6f}"
)

print(
    f"  nearest singularity "
    f"margin: "
    f"{minimum_singularity_margin_deg:.6f} deg"
)


print("\nCommand bounds:")

print(
    f"  max |forward cmd|:     "
    f"{maximum_forward_command:.6f}"
)

print(
    f"  max |heading cmd|:     "
    f"{maximum_heading_command:.6f}"
)

print(
    f"  max |depth cmd|:       "
    f"{maximum_depth_command:.6f}"
)


print("\n" + "=" * 80)

print(
    "STEP 7 CLOSED-LOOP WAYPOINT "
    "NAVIGATION PASSED"
)

print("=" * 80)

print(
    "Waypoint guidance, heading PID, depth PID, "
    "and forward propulsion were integrated through "
    "the production 6-DOF actuator-wrench interface."
)

print(
    "The vehicle entered the waypoint success region "
    "without direct state manipulation."
)

print(
    "The complete trajectory remained underwater and "
    "safely separated from the Euler pitch singularity."
)

print(
    "Roll and pitch excursions were measured rather "
    "than judged against an arbitrary 20-degree limit."
)

print(
    "World position remained only the explicitly declared "
    "perfect-localization baseline."
)

print(
    "No production files were created or modified."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 7 WAYPOINT NAVIGATION

Vehicle stability configuration:
  center of gravity:     [0. 0. 0.]
  center of buoyancy:    [0.   0.   0.08]
  CB-CG vertical offset: 0.080000 m

Waypoint geometry:
  start position:        [ 0.  0. -2.]
  target waypoint:       [20. 12. -8.]
  final position:        [19.72046935 11.04664308 -8.07753747]

Navigation performance:
  initial distance:      24.083189 m
  final distance:        0.996513 m
  distance reduction:    95.862 %
  success radius:        1.000000 m
  reached waypoint:      True
  navigation time:       18.620000 s

Depth bounds:
  minimum depth:         2.000000 m
  maximum depth:         8.549818 m

Attitude diagnostics:
  maximum |roll|:        19.617814 deg
  maximum |pitch|:       42.678614 deg
  final roll:            5.939317 deg
  final pitch:           3.126653 deg
  minimum |cos(pitch)|:  0.735168
  nearest singularity margin: 47.321386 deg

Command bounds:
  max |forward cmd|:     1.000000
  max |hea

## Step 8 - Waypoint Navigation with Noisy Sensor Feedback

The deterministic waypoint experiment has established that the complete guidance and control architecture works when depth and heading measurements are ideal.

Notebook 08, however, introduced imperfect sensor observations specifically so that autonomy algorithms do not need to operate directly on perfect physical state.

The next validation therefore replaces:

```text
ideal depth feedback
ideal heading feedback
```

with:

```text
noisy depth measurement
noisy heading measurement
```

while preserving:

```text
physical ground truth
        ≠
sensor measurement
```

The architecture becomes:

```text
                  perfect-position baseline
                           │
                           ▼
                       guidance
                           │
                 ┌─────────┴─────────┐
                 ▼                   ▼
          desired heading       desired depth
                 │                   │
                 ▼                   ▼
        heading measurement   depth measurement
                 │                   │
                 ▼                   ▼
             heading PID          depth PID
                 │                   │
                 └─────────┬─────────┘
                           ▼
                    actuator wrench
                           ↓
                     6-DOF plant
                           ↓
                    physical state
                           │
                  ┌────────┴────────┐
                  ▼                 ▼
            heading sensor     depth sensor
```

Sensor randomness uses explicit independent random-number streams:

```python
np.random.default_rng(seed)
```

The experiment deliberately uses modest bias and noise rather than extreme corruption.

The purpose is not to optimize controller robustness.

The purpose is to verify that the validated closed-loop architecture can operate from imperfect Notebook 08 measurements without:

- corrupting simulator truth;
- becoming numerically unstable;
- violating actuator bounds;
- or losing the waypoint entirely.

### Sensor Configuration

The validation uses fixed deterministic sensor imperfections:

```text
depth bias
depth Gaussian noise

heading bias
heading Gaussian noise
```

with independent RNG streams for the two sensors.

The random seeds are fixed so the experiment is exactly reproducible.

### Localization Assumption

Only depth and heading feedback are made imperfect here.

World-frame XYZ position remains the explicitly declared perfect-localization baseline used by the waypoint-guidance layer.

A localization estimator is still outside Notebook 09.

### Validation Criteria

The experiment passes if:

1. noisy measurements are generated through the production sensor models;
2. independent seeded RNG streams are used;
3. physical ground truth is never overwritten by sensor measurements;
4. all simulated states remain finite;
5. normalized actuator commands remain bounded;
6. the vehicle remains below the modeled water surface;
7. the trajectory remains safely away from the Euler pitch singularity;
8. waypoint distance decreases substantially;
9. the vehicle enters the defined waypoint-success radius;
10. measured depth and heading demonstrably differ from ground truth during the run.

No production files are modified in this step.

In [12]:
import numpy as np

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)

from hydrolab3d.sensors import (
    depth_measurement,
    heading_measurement,
    ideal_depth_measurement,
    ideal_heading_measurement,
    wrap_angle_pi,
)


# ---------------------------------------------------------------------
# Canonical plant parameters
# ---------------------------------------------------------------------

mass = 20.0

inertia = np.array(
    [
        1.2,
        1.6,
        2.0,
    ],
    dtype=float,
)

added_inertia = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.20,
        0.30,
        0.40,
    ],
    dtype=float,
)

linear_damping = np.array(
    [
        2.0,
        3.0,
        4.0,
        0.10,
        0.15,
        0.20,
    ],
    dtype=float,
)

quadratic_damping = np.array(
    [
        5.0,
        6.0,
        7.0,
        0.30,
        0.40,
        0.50,
    ],
    dtype=float,
)

fluid_density = 1000.0
gravity = 9.81

displaced_volume = (
    mass / fluid_density
)

center_of_gravity = np.array(
    [
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

center_of_buoyancy = np.array(
    [
        0.0,
        0.0,
        0.08,
    ],
    dtype=float,
)

current_velocity_world = np.zeros(
    3,
    dtype=float,
)


# ---------------------------------------------------------------------
# Sensor configuration
#
# These values are deliberately modest.
#
# Units:
#
# depth:
#     meters
#
# heading:
#     radians
# ---------------------------------------------------------------------

depth_bias = 0.03
depth_noise_std = 0.04

heading_bias = np.deg2rad(
    0.75
)

heading_noise_std = np.deg2rad(
    1.0
)


# ---------------------------------------------------------------------
# Independent deterministic sensor RNG streams
# ---------------------------------------------------------------------

depth_rng = np.random.default_rng(
    9041
)

heading_rng = np.random.default_rng(
    9042
)


# ---------------------------------------------------------------------
# Simulation setup
# ---------------------------------------------------------------------

dt = 0.02
duration = 45.0

steps = int(
    duration / dt
)

times = (
    np.arange(
        steps + 1,
        dtype=float,
    )
    * dt
)


# ---------------------------------------------------------------------
# Initial physical state
# ---------------------------------------------------------------------

pose = np.array(
    [
        0.0,
        0.0,
        -2.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

body_velocity = np.zeros(
    6,
    dtype=float,
)


# ---------------------------------------------------------------------
# Target waypoint
# ---------------------------------------------------------------------

waypoint_world = np.array(
    [
        20.0,
        12.0,
        -8.0,
    ],
    dtype=float,
)

success_radius = 1.0


# ---------------------------------------------------------------------
# Controllers
# ---------------------------------------------------------------------

depth_pid = NotebookPID(
    kp=0.55,
    ki=0.08,
    kd=0.65,
    output_limits=(-1.0, 1.0),
    integral_limits=(-4.0, 4.0),
)

heading_pid = NotebookPID(
    kp=0.80,
    ki=0.02,
    kd=0.80,
    output_limits=(-1.0, 1.0),
    integral_limits=(-2.0, 2.0),
)


# ---------------------------------------------------------------------
# Actuator authority
# ---------------------------------------------------------------------

max_forward_force = 25.0
max_vertical_force = 60.0
max_yaw_moment = 3.0

forward_distance_gain = 0.10


# ---------------------------------------------------------------------
# Histories
# ---------------------------------------------------------------------

pose_history_noisy = np.zeros(
    (steps + 1, 6),
    dtype=float,
)

velocity_history_noisy = np.zeros(
    (steps + 1, 6),
    dtype=float,
)

distance_history_noisy = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

true_depth_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

measured_depth_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

true_heading_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

measured_heading_history = np.full(
    steps + 1,
    np.nan,
    dtype=float,
)

forward_command_history_noisy = np.zeros(
    steps + 1,
    dtype=float,
)

heading_command_history_noisy = np.zeros(
    steps + 1,
    dtype=float,
)

depth_command_history_noisy = np.zeros(
    steps + 1,
    dtype=float,
)


# ---------------------------------------------------------------------
# Initial truth and sensor measurements
# ---------------------------------------------------------------------

initial_guidance = waypoint_guidance(
    position_world=pose[:3],
    waypoint_world=waypoint_world,
)

true_depth = float(
    ideal_depth_measurement(
        pose
    )
)

measured_depth = float(
    depth_measurement(
        pose,
        bias=depth_bias,
        noise_std=depth_noise_std,
        rng=depth_rng,
    )
)

true_heading = float(
    ideal_heading_measurement(
        pose
    )
)

measured_heading = float(
    heading_measurement(
        pose,
        bias=heading_bias,
        noise_std=heading_noise_std,
        rng=heading_rng,
    )
)


pose_history_noisy[0] = (
    pose
)

velocity_history_noisy[0] = (
    body_velocity
)

distance_history_noisy[0] = (
    initial_guidance[
        "distance"
    ]
)

true_depth_history[0] = (
    true_depth
)

measured_depth_history[0] = (
    measured_depth
)

true_heading_history[0] = (
    true_heading
)

measured_heading_history[0] = (
    measured_heading
)


# ---------------------------------------------------------------------
# Navigation state
# ---------------------------------------------------------------------

reached_waypoint_noisy = False
final_index = 0


# ---------------------------------------------------------------------
# Closed-loop noisy-feedback navigation
# ---------------------------------------------------------------------

for k in range(steps):

    # -------------------------------------------------------------
    # Temporary perfect-position localization baseline
    # -------------------------------------------------------------

    position_world = (
        pose[:3].copy()
    )


    # -------------------------------------------------------------
    # Waypoint guidance
    # -------------------------------------------------------------

    guidance = waypoint_guidance(
        position_world=position_world,
        waypoint_world=waypoint_world,
    )

    waypoint_distance = float(
        guidance[
            "distance"
        ]
    )

    horizontal_distance = float(
        guidance[
            "horizontal_distance"
        ]
    )

    desired_heading = float(
        guidance[
            "desired_heading"
        ]
    )

    desired_depth = float(
        guidance[
            "desired_depth"
        ]
    )


    # -------------------------------------------------------------
    # Success condition
    #
    # Uses physical position because this is the simulation's
    # objective validation condition, not controller feedback.
    # -------------------------------------------------------------

    if (
        waypoint_distance
        <= success_radius
    ):

        reached_waypoint_noisy = True
        final_index = k

        break


    # -------------------------------------------------------------
    # Production noisy sensor measurements
    # -------------------------------------------------------------

    measured_depth = float(
        depth_measurement(
            pose,
            bias=depth_bias,
            noise_std=depth_noise_std,
            rng=depth_rng,
        )
    )

    measured_heading = float(
        heading_measurement(
            pose,
            bias=heading_bias,
            noise_std=heading_noise_std,
            rng=heading_rng,
        )
    )


    # -------------------------------------------------------------
    # Errors from MEASURED feedback
    # -------------------------------------------------------------

    depth_error = (
        desired_depth
        - measured_depth
    )

    heading_error = float(
        wrap_angle_pi(
            desired_heading
            - measured_heading
        )
    )


    # -------------------------------------------------------------
    # PID updates
    # -------------------------------------------------------------

    (
        depth_command,
        depth_diagnostics,
    ) = depth_pid.update(
        error=depth_error,
        dt=dt,
    )

    (
        heading_command,
        heading_diagnostics,
    ) = heading_pid.update(
        error=heading_error,
        dt=dt,
    )


    # -------------------------------------------------------------
    # Simple surge guidance
    # -------------------------------------------------------------

    distance_command = float(
        np.clip(
            forward_distance_gain
            * horizontal_distance,
            0.0,
            1.0,
        )
    )

    heading_alignment = float(
        np.clip(
            np.cos(
                heading_error
            ),
            0.0,
            1.0,
        )
    )

    forward_command = (
        distance_command
        * heading_alignment
    )


    # -------------------------------------------------------------
    # Physical actuator wrench
    # -------------------------------------------------------------

    forward_force = (
        forward_command
        * max_forward_force
    )

    vertical_force = (
        -depth_command
        * max_vertical_force
    )

    yaw_moment = (
        heading_command
        * max_yaw_moment
    )

    actuator_wrench = np.array(
        [
            forward_force,
            0.0,
            vertical_force,
            0.0,
            0.0,
            yaw_moment,
        ],
        dtype=float,
    )


    # -------------------------------------------------------------
    # Production 6-DOF integration
    # -------------------------------------------------------------

    (
        pose,
        body_velocity,
        plant_diagnostics,
    ) = integrate_six_dof_euler(
        pose,
        body_velocity,
        dt,

        mass=mass,
        inertia=inertia,
        added_inertia=added_inertia,

        linear_damping=linear_damping,
        quadratic_damping=quadratic_damping,

        fluid_density=fluid_density,
        displaced_volume=displaced_volume,

        center_of_gravity=center_of_gravity,
        center_of_buoyancy=center_of_buoyancy,

        current_velocity_world=current_velocity_world,

        actuator_wrench=actuator_wrench,

        gravity=gravity,
    )


    # -------------------------------------------------------------
    # Numerical sanity
    # -------------------------------------------------------------

    if not np.all(
        np.isfinite(
            pose
        )
    ):

        raise RuntimeError(
            "Noisy-feedback navigation produced "
            "a non-finite pose."
        )

    if not np.all(
        np.isfinite(
            body_velocity
        )
    ):

        raise RuntimeError(
            "Noisy-feedback navigation produced "
            "a non-finite velocity."
        )


    # -------------------------------------------------------------
    # Physical water-surface constraint
    #
    # Diagnostic only. No clipping.
    # -------------------------------------------------------------

    if pose[2] > 1e-9:

        raise RuntimeError(
            "Noisy-feedback navigation crossed "
            "above the modeled water surface at "
            f"t={(k + 1) * dt:.3f} s: "
            f"z={pose[2]:.6f} m"
        )


    # -------------------------------------------------------------
    # Record new physical state
    # -------------------------------------------------------------

    index = k + 1
    final_index = index

    pose_history_noisy[index] = (
        pose
    )

    velocity_history_noisy[index] = (
        body_velocity
    )


    # -------------------------------------------------------------
    # Updated ground-truth navigation distance
    # -------------------------------------------------------------

    updated_guidance = waypoint_guidance(
        position_world=pose[:3],
        waypoint_world=waypoint_world,
    )

    distance_history_noisy[index] = (
        updated_guidance[
            "distance"
        ]
    )


    # -------------------------------------------------------------
    # Ground-truth sensor quantities
    # -------------------------------------------------------------

    true_depth_history[index] = float(
        ideal_depth_measurement(
            pose
        )
    )

    true_heading_history[index] = float(
        ideal_heading_measurement(
            pose
        )
    )


    # -------------------------------------------------------------
    # Store the measurements that actually drove this step
    #
    # Measurement at index corresponds to the feedback used to
    # generate the transition ending at this physical state.
    # -------------------------------------------------------------

    measured_depth_history[index] = (
        measured_depth
    )

    measured_heading_history[index] = (
        measured_heading
    )


    # -------------------------------------------------------------
    # Command histories
    # -------------------------------------------------------------

    forward_command_history_noisy[index] = (
        forward_command
    )

    heading_command_history_noisy[index] = (
        heading_command
    )

    depth_command_history_noisy[index] = (
        depth_command
    )


# ---------------------------------------------------------------------
# Final waypoint check
# ---------------------------------------------------------------------

if not reached_waypoint_noisy:

    final_guidance = waypoint_guidance(
        position_world=pose[:3],
        waypoint_world=waypoint_world,
    )

    if (
        final_guidance[
            "distance"
        ]
        <= success_radius
    ):

        reached_waypoint_noisy = True


# ---------------------------------------------------------------------
# Trim histories
# ---------------------------------------------------------------------

used = slice(
    0,
    final_index + 1,
)

times_noisy = (
    times[used]
)

pose_noisy = (
    pose_history_noisy[used]
)

velocity_noisy = (
    velocity_history_noisy[used]
)

distance_noisy = (
    distance_history_noisy[used]
)

true_depth_noisy = (
    true_depth_history[used]
)

measured_depth_noisy = (
    measured_depth_history[used]
)

true_heading_noisy = (
    true_heading_history[used]
)

measured_heading_noisy = (
    measured_heading_history[used]
)

forward_command_noisy = (
    forward_command_history_noisy[used]
)

heading_command_noisy = (
    heading_command_history_noisy[used]
)

depth_command_noisy = (
    depth_command_history_noisy[used]
)


# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------

initial_distance_noisy = float(
    distance_noisy[0]
)

final_distance_noisy = float(
    distance_noisy[-1]
)

final_position_noisy = (
    pose_noisy[-1, :3].copy()
)

maximum_forward_command_noisy = float(
    np.max(
        np.abs(
            forward_command_noisy
        )
    )
)

maximum_heading_command_noisy = float(
    np.max(
        np.abs(
            heading_command_noisy
        )
    )
)

maximum_depth_command_noisy = float(
    np.max(
        np.abs(
            depth_command_noisy
        )
    )
)


# ---------------------------------------------------------------------
# Sensor discrepancy metrics
#
# Because measurements drive the transition from sample k-1 to k,
# compare stored measurement sample k against truth at sample k-1.
# Exclude sample zero from this alignment metric.
# ---------------------------------------------------------------------

if len(
    measured_depth_noisy
) < 2:

    raise RuntimeError(
        "Noisy sensor experiment produced insufficient samples."
    )


depth_measurement_error = (
    measured_depth_noisy[1:]
    - true_depth_noisy[:-1]
)

heading_measurement_error = np.array(
    [
        float(
            wrap_angle_pi(
                measured_heading_noisy[i]
                - true_heading_noisy[i - 1]
            )
        )
        for i in range(
            1,
            len(
                measured_heading_noisy
            ),
        )
    ],
    dtype=float,
)


depth_error_mean = float(
    np.mean(
        depth_measurement_error
    )
)

depth_error_std = float(
    np.std(
        depth_measurement_error
    )
)

heading_error_mean = float(
    np.mean(
        heading_measurement_error
    )
)

heading_error_std = float(
    np.std(
        heading_measurement_error
    )
)


# ---------------------------------------------------------------------
# Euler-singularity margin
# ---------------------------------------------------------------------

minimum_abs_cos_pitch_noisy = float(
    np.min(
        np.abs(
            np.cos(
                pose_noisy[:, 4]
            )
        )
    )
)


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

assert np.all(
    np.isfinite(
        pose_noisy
    )
)

assert np.all(
    np.isfinite(
        velocity_noisy
    )
)

assert np.all(
    np.isfinite(
        distance_noisy
    )
)


# Vehicle must remain underwater.
assert np.all(
    pose_noisy[:, 2]
    <= 1e-9
)


# Commands remain normalized.
assert maximum_forward_command_noisy <= (
    1.0 + 1e-12
)

assert maximum_heading_command_noisy <= (
    1.0 + 1e-12
)

assert maximum_depth_command_noisy <= (
    1.0 + 1e-12
)


# Euler representation remains valid.
assert (
    minimum_abs_cos_pitch_noisy
    > 0.10
)


# Strong waypoint-distance reduction.
assert final_distance_noisy < (
    0.10
    * initial_distance_noisy
)


# Success region reached.
assert (
    final_distance_noisy
    <= success_radius
)

assert reached_waypoint_noisy


# Measurements must actually differ from truth.
assert not np.allclose(
    depth_measurement_error,
    0.0,
)

assert not np.allclose(
    heading_measurement_error,
    0.0,
)


# Sensor statistics should show the intended imperfections
# without enforcing exact finite-sample equality.
assert abs(
    depth_error_mean
    - depth_bias
) < 0.03

assert abs(
    heading_error_mean
    - heading_bias
) < np.deg2rad(
    0.75
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)

print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 8 NOISY SENSOR FEEDBACK"
)

print("=" * 80)


print("\nSensor configuration:")

print(
    f"  depth bias:           "
    f"{depth_bias:.6f} m"
)

print(
    f"  depth noise std:      "
    f"{depth_noise_std:.6f} m"
)

print(
    f"  heading bias:         "
    f"{np.rad2deg(heading_bias):.6f} deg"
)

print(
    f"  heading noise std:    "
    f"{np.rad2deg(heading_noise_std):.6f} deg"
)


print("\nObserved sensor discrepancy:")

print(
    f"  depth error mean:     "
    f"{depth_error_mean:.6f} m"
)

print(
    f"  depth error std:      "
    f"{depth_error_std:.6f} m"
)

print(
    f"  heading error mean:   "
    f"{np.rad2deg(heading_error_mean):.6f} deg"
)

print(
    f"  heading error std:    "
    f"{np.rad2deg(heading_error_std):.6f} deg"
)


print("\nNavigation performance:")

print(
    f"  initial distance:     "
    f"{initial_distance_noisy:.6f} m"
)

print(
    f"  final distance:       "
    f"{final_distance_noisy:.6f} m"
)

print(
    f"  distance reduction:   "
    f"{100.0 * (1.0 - final_distance_noisy / initial_distance_noisy):.3f} %"
)

print(
    f"  reached waypoint:     "
    f"{reached_waypoint_noisy}"
)

print(
    f"  navigation time:      "
    f"{times_noisy[-1]:.6f} s"
)

print(
    f"  final position:       "
    f"{final_position_noisy}"
)


print("\nCommand bounds:")

print(
    f"  max |forward cmd|:    "
    f"{maximum_forward_command_noisy:.6f}"
)

print(
    f"  max |heading cmd|:    "
    f"{maximum_heading_command_noisy:.6f}"
)

print(
    f"  max |depth cmd|:      "
    f"{maximum_depth_command_noisy:.6f}"
)

print(
    f"  minimum |cos(pitch)|: "
    f"{minimum_abs_cos_pitch_noisy:.6f}"
)


print("\n" + "=" * 80)

print(
    "STEP 8 NOISY-FEEDBACK "
    "NAVIGATION PASSED"
)

print("=" * 80)

print(
    "Production depth and heading sensor models "
    "successfully closed the control loop."
)

print(
    "Independent seeded sensor RNG streams preserved "
    "deterministic reproducibility."
)

print(
    "Sensor measurements differed from physical truth "
    "without overwriting the physical state."
)

print(
    "The vehicle still entered the waypoint success "
    "region using imperfect feedback."
)

print(
    "World XYZ position remained the explicitly declared "
    "perfect-localization baseline."
)

print(
    "No production files were created or modified."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 8 NOISY SENSOR FEEDBACK

Sensor configuration:
  depth bias:           0.030000 m
  depth noise std:      0.040000 m
  heading bias:         0.750000 deg
  heading noise std:    1.000000 deg

Observed sensor discrepancy:
  depth error mean:     0.028149 m
  depth error std:      0.039301 m
  heading error mean:   0.777158 deg
  heading error std:    1.025826 deg

Navigation performance:
  initial distance:     24.083189 m
  final distance:       0.995951 m
  distance reduction:   95.865 %
  reached waypoint:     True
  navigation time:      19.340000 s
  final position:       [20.14623696 11.02227965 -7.87918502]

Command bounds:
  max |forward cmd|:    1.000000
  max |heading cmd|:    1.000000
  max |depth cmd|:      1.000000
  minimum |cos(pitch)|: 0.759936

STEP 8 NOISY-FEEDBACK NAVIGATION PASSED
Production depth and heading sensor models successfully closed the control loop.
Independent seeded sensor RNG streams preserved deterministic reproducibility

## Step 9 - Promote Validated PID and Waypoint Guidance into Production

The notebook-local control primitives have now been validated through:

- isolated PID tests;
- closed-loop depth regulation;
- wrapped heading regulation;
- deterministic waypoint navigation;
- noisy-sensor waypoint navigation.

They are therefore ready to move into the HydroLab-3D source package.

Only the reusable controller primitives will be promoted:

```text
src/hydrolab3d/control/pid.py
    → PIDController

src/hydrolab3d/control/waypoint.py
    → waypoint_guidance

src/hydrolab3d/control/__init__.py
    → public control API
```

The existing:

```text
src/hydrolab3d/control/controller.py
```

will remain untouched because Notebook 09 has not demonstrated a need for an abstract controller hierarchy.

The notebook-specific:

- vehicle parameters;
- PID gains;
- waypoint target;
- actuator authority;
- simulation loops;
- perfect-localization assumption;
- validation assertions;

will remain inside the notebook.

This keeps the production control layer minimal and reusable.

### Validation Criteria

This step passes if:

1. `pid.py` contains the validated scalar PID implementation;
2. `waypoint.py` contains the validated waypoint-reference geometry;
3. `__init__.py` exposes the new public API;
4. `controller.py` remains untouched;
5. no notebook-specific experiment code is promoted;
6. the promoted files can be displayed exactly after writing.

Functional production parity will be tested in the next step.

In [13]:
from pathlib import Path


# ---------------------------------------------------------------------
# Production control-package paths
# ---------------------------------------------------------------------

control_dir = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "control"
)

pid_path = (
    control_dir
    / "pid.py"
)

waypoint_path = (
    control_dir
    / "waypoint.py"
)

init_path = (
    control_dir
    / "__init__.py"
)

controller_path = (
    control_dir
    / "controller.py"
)


# ---------------------------------------------------------------------
# Validate expected pre-existing package state
# ---------------------------------------------------------------------

if not control_dir.exists():
    raise FileNotFoundError(
        f"Expected control package is missing: {control_dir}"
    )

if not controller_path.exists():
    raise FileNotFoundError(
        f"Expected controller placeholder is missing: {controller_path}"
    )


# Preserve controller.py exactly as inherited.
controller_before = controller_path.read_text(
    encoding="utf-8"
)


# ---------------------------------------------------------------------
# Production PID implementation
# ---------------------------------------------------------------------

pid_source = '''"""Minimal classical feedback controllers for HydroLab-3D."""

from dataclasses import dataclass

import numpy as np


@dataclass
class PIDController:
    """Minimal scalar PID controller.

    Parameters
    ----------
    kp, ki, kd : float
        Proportional, integral, and derivative gains.

    output_limits : tuple[float, float]
        Minimum and maximum controller output.

    integral_limits : tuple[float, float] or None
        Optional bounds on accumulated integral state.
    """

    kp: float
    ki: float
    kd: float
    output_limits: tuple = (-1.0, 1.0)
    integral_limits: tuple | None = None

    def __post_init__(self):
        self.kp = float(self.kp)
        self.ki = float(self.ki)
        self.kd = float(self.kd)

        output_min, output_max = self.output_limits

        self.output_min = float(output_min)
        self.output_max = float(output_max)

        if self.output_min > self.output_max:
            raise ValueError(
                "output_limits must satisfy minimum <= maximum"
            )

        if self.integral_limits is None:
            self.integral_min = None
            self.integral_max = None
        else:
            integral_min, integral_max = self.integral_limits

            self.integral_min = float(integral_min)
            self.integral_max = float(integral_max)

            if self.integral_min > self.integral_max:
                raise ValueError(
                    "integral_limits must satisfy minimum <= maximum"
                )

        self.reset()

    def reset(self):
        """Reset accumulated PID state."""

        self.integral = 0.0
        self.previous_error = None

    def update(self, error, dt):
        """Advance the PID controller by one timestep.

        Parameters
        ----------
        error : float
            Current scalar control error.

        dt : float
            Positive controller timestep [s].

        Returns
        -------
        output : float
            Saturated controller output.

        diagnostics : dict
            PID terms and controller internal state.
        """

        error = float(error)
        dt = float(dt)

        if not np.isfinite(error):
            raise ValueError(
                "error must be finite"
            )

        if not np.isfinite(dt) or dt <= 0.0:
            raise ValueError(
                "dt must be finite and strictly positive"
            )

        proportional = (
            self.kp
            * error
        )

        self.integral += (
            error
            * dt
        )

        if self.integral_limits is not None:
            self.integral = float(
                np.clip(
                    self.integral,
                    self.integral_min,
                    self.integral_max,
                )
            )

        integral_term = (
            self.ki
            * self.integral
        )

        if self.previous_error is None:
            error_derivative = 0.0
        else:
            error_derivative = (
                error
                - self.previous_error
            ) / dt

        derivative = (
            self.kd
            * error_derivative
        )

        unsaturated_output = (
            proportional
            + integral_term
            + derivative
        )

        output = float(
            np.clip(
                unsaturated_output,
                self.output_min,
                self.output_max,
            )
        )

        self.previous_error = (
            error
        )

        diagnostics = {
            "error":
                error,

            "proportional":
                proportional,

            "integral_state":
                self.integral,

            "integral":
                integral_term,

            "error_derivative":
                error_derivative,

            "derivative":
                derivative,

            "unsaturated_output":
                unsaturated_output,

            "output":
                output,

            "saturated":
                not np.isclose(
                    output,
                    unsaturated_output,
                ),
        }

        return (
            output,
            diagnostics,
        )
'''


# ---------------------------------------------------------------------
# Production waypoint-guidance implementation
# ---------------------------------------------------------------------

waypoint_source = '''"""Minimal waypoint-reference generation for HydroLab-3D."""

import numpy as np


def waypoint_guidance(
    position_world,
    waypoint_world,
):
    """Convert a world-frame waypoint into navigation references.

    Parameters
    ----------
    position_world : array-like, shape (3,)
        Current world-frame position [x, y, z].

    waypoint_world : array-like, shape (3,)
        Desired world-frame waypoint [x_g, y_g, z_g].

    Returns
    -------
    guidance : dict
        Contains displacement, desired heading, desired depth,
        horizontal distance, and full 3D distance.

    Notes
    -----
    HydroLab-3D uses +z world upward and depth = -z.

    This function performs geometry only. It does not provide
    localization and does not modify vehicle state.
    """

    position_world = np.asarray(
        position_world,
        dtype=float,
    )

    waypoint_world = np.asarray(
        waypoint_world,
        dtype=float,
    )

    if position_world.shape != (3,):
        raise ValueError(
            "position_world must have shape (3,)"
        )

    if waypoint_world.shape != (3,):
        raise ValueError(
            "waypoint_world must have shape (3,)"
        )

    if not np.all(
        np.isfinite(
            position_world
        )
    ):
        raise ValueError(
            "position_world must contain only finite values"
        )

    if not np.all(
        np.isfinite(
            waypoint_world
        )
    ):
        raise ValueError(
            "waypoint_world must contain only finite values"
        )

    displacement_world = (
        waypoint_world
        - position_world
    )

    dx = float(
        displacement_world[0]
    )

    dy = float(
        displacement_world[1]
    )

    desired_heading = float(
        np.arctan2(
            dy,
            dx,
        )
    )

    desired_depth = float(
        -waypoint_world[2]
    )

    horizontal_distance = float(
        np.hypot(
            dx,
            dy,
        )
    )

    distance = float(
        np.linalg.norm(
            displacement_world
        )
    )

    return {
        "displacement_world":
            displacement_world,

        "desired_heading":
            desired_heading,

        "desired_depth":
            desired_depth,

        "horizontal_distance":
            horizontal_distance,

        "distance":
            distance,
    }
'''


# ---------------------------------------------------------------------
# Public control API
# ---------------------------------------------------------------------

init_source = '''"""Control utilities for HydroLab-3D."""

from .pid import PIDController
from .waypoint import waypoint_guidance


__all__ = [
    "PIDController",
    "waypoint_guidance",
]
'''


# ---------------------------------------------------------------------
# Write validated production implementation
# ---------------------------------------------------------------------

pid_path.write_text(
    pid_source,
    encoding="utf-8",
)

waypoint_path.write_text(
    waypoint_source,
    encoding="utf-8",
)

init_path.write_text(
    init_source,
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Confirm controller.py was not modified
# ---------------------------------------------------------------------

controller_after = controller_path.read_text(
    encoding="utf-8"
)

assert (
    controller_after
    == controller_before
)


# ---------------------------------------------------------------------
# Display exact promoted source
# ---------------------------------------------------------------------

promoted_files = [
    init_path,
    pid_path,
    waypoint_path,
    controller_path,
]


print("=" * 80)

print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 9 PRODUCTION PROMOTION"
)

print("=" * 80)


for path in promoted_files:

    source = path.read_text(
        encoding="utf-8"
    )

    print("\n")
    print("=" * 80)

    print(
        path.relative_to(
            repo_root
        )
    )

    print("=" * 80)

    if source.strip():

        for line_number, line in enumerate(
            source.splitlines(),
            start=1,
        ):

            print(
                f"{line_number:4d} | {line}"
            )

    else:

        print(
            "<EMPTY FILE>"
        )


# ---------------------------------------------------------------------
# Promotion assertions
# ---------------------------------------------------------------------

assert "class PIDController" in (
    pid_path.read_text(
        encoding="utf-8"
    )
)

assert "def waypoint_guidance" in (
    waypoint_path.read_text(
        encoding="utf-8"
    )
)

assert "PIDController" in (
    init_path.read_text(
        encoding="utf-8"
    )
)

assert "waypoint_guidance" in (
    init_path.read_text(
        encoding="utf-8"
    )
)


print("\n" + "=" * 80)

print(
    "STEP 9 PRODUCTION PROMOTION PASSED"
)

print("=" * 80)

print(
    "Validated PID and waypoint-guidance primitives "
    "were promoted into src/hydrolab3d/control/."
)

print(
    "controller.py remained untouched."
)

print(
    "Notebook-specific plant, tuning, and simulation "
    "logic were not promoted."
)

print(
    "Production behavior will be re-imported and "
    "validated in the next step."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 9 PRODUCTION PROMOTION


src/hydrolab3d/control/__init__.py
   1 | """Control utilities for HydroLab-3D."""
   2 | 
   3 | from .pid import PIDController
   4 | from .waypoint import waypoint_guidance
   5 | 
   6 | 
   7 | __all__ = [
   8 |     "PIDController",
   9 |     "waypoint_guidance",
  10 | ]


src/hydrolab3d/control/pid.py
   1 | """Minimal classical feedback controllers for HydroLab-3D."""
   2 | 
   3 | from dataclasses import dataclass
   4 | 
   5 | import numpy as np
   6 | 
   7 | 
   8 | @dataclass
   9 | class PIDController:
  10 |     """Minimal scalar PID controller.
  11 | 
  12 |     Parameters
  13 |     ----------
  14 |     kp, ki, kd : float
  15 |         Proportional, integral, and derivative gains.
  16 | 
  17 |     output_limits : tuple[float, float]
  18 |         Minimum and maximum controller output.
  19 | 
  20 |     integral_limits : tuple[float, float] or None
  21 |         Optional bounds on accumulated integral s

## Step 10 - Re-Import Production Control Code and Validate Parity

The validated PID and waypoint-guidance primitives have now been promoted into:

```text
src/hydrolab3d/control/
```

Promotion alone is not sufficient.

The notebook must now stop relying on its local prototype implementations and verify that the actual production package behaves identically.

This step therefore:

1. invalidates Python import caches;
2. removes previously loaded HydroLab-3D control modules from `sys.modules`;
3. imports `hydrolab3d.control` fresh from the source package;
4. verifies the public API;
5. re-runs deterministic PID behavior checks using `PIDController`;
6. compares production `waypoint_guidance()` against the previously validated notebook-local guidance function;
7. confirms that production source locations resolve inside `src/hydrolab3d/control/`.

The purpose is to establish:

```text
validated notebook prototype
        ↓
production promotion
        ↓
fresh production import
        ↓
behavioral parity
```

### Validation Criteria

This step passes if:

1. `PIDController` imports from the production package;
2. `waypoint_guidance` imports from the production package;
3. both objects resolve to production source files;
4. proportional, integral, derivative, saturation, clamping, and reset behavior remain correct;
5. production waypoint guidance matches the validated notebook-local implementation;
6. the public `hydrolab3d.control` API exposes exactly the intended reusable functionality.

No production files are modified in this step.

In [14]:
import importlib
import inspect
import sys
from pathlib import Path

import numpy as np


# ---------------------------------------------------------------------
# Remove previously imported control modules
#
# This forces Python to load the promoted implementation from disk.
# ---------------------------------------------------------------------

control_module_names = [
    name
    for name in list(
        sys.modules
    )
    if (
        name == "hydrolab3d.control"
        or name.startswith(
            "hydrolab3d.control."
        )
    )
]

for name in control_module_names:
    del sys.modules[name]


importlib.invalidate_caches()


# ---------------------------------------------------------------------
# Fresh production import
# ---------------------------------------------------------------------

import hydrolab3d.control as control

from hydrolab3d.control import (
    PIDController,
    waypoint_guidance as production_waypoint_guidance,
)


# ---------------------------------------------------------------------
# Validate public API
# ---------------------------------------------------------------------

expected_public_api = {
    "PIDController",
    "waypoint_guidance",
}

actual_public_api = set(
    getattr(
        control,
        "__all__",
        [],
    )
)

assert actual_public_api == (
    expected_public_api
)


# ---------------------------------------------------------------------
# Verify production source locations
# ---------------------------------------------------------------------

pid_source_path = Path(
    inspect.getsourcefile(
        PIDController
    )
).resolve()

waypoint_source_path = Path(
    inspect.getsourcefile(
        production_waypoint_guidance
    )
).resolve()

expected_control_dir = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "control"
).resolve()

assert (
    pid_source_path.parent
    == expected_control_dir
)

assert (
    waypoint_source_path.parent
    == expected_control_dir
)


# ---------------------------------------------------------------------
# Test 1 - Production proportional response
# ---------------------------------------------------------------------

pid_p_prod = PIDController(
    kp=2.0,
    ki=0.0,
    kd=0.0,
    output_limits=(-10.0, 10.0),
)

output_p_prod, diag_p_prod = (
    pid_p_prod.update(
        error=1.5,
        dt=0.1,
    )
)

assert np.isclose(
    output_p_prod,
    3.0,
)

print(
    "Test 1 - Production proportional response: PASS"
)


# ---------------------------------------------------------------------
# Test 2 - Production integral accumulation
# ---------------------------------------------------------------------

pid_i_prod = PIDController(
    kp=0.0,
    ki=2.0,
    kd=0.0,
    output_limits=(-100.0, 100.0),
)

for _ in range(4):

    output_i_prod, diag_i_prod = (
        pid_i_prod.update(
            error=1.0,
            dt=0.5,
        )
    )

assert np.isclose(
    pid_i_prod.integral,
    2.0,
)

assert np.isclose(
    output_i_prod,
    4.0,
)

print(
    "Test 2 - Production integral accumulation: PASS"
)


# ---------------------------------------------------------------------
# Test 3 - Production derivative response
# ---------------------------------------------------------------------

pid_d_prod = PIDController(
    kp=0.0,
    ki=0.0,
    kd=3.0,
    output_limits=(-100.0, 100.0),
)

first_output_prod, _ = (
    pid_d_prod.update(
        error=1.0,
        dt=0.5,
    )
)

second_output_prod, _ = (
    pid_d_prod.update(
        error=2.0,
        dt=0.5,
    )
)

assert np.isclose(
    first_output_prod,
    0.0,
)

assert np.isclose(
    second_output_prod,
    6.0,
)

print(
    "Test 3 - Production derivative response: PASS"
)


# ---------------------------------------------------------------------
# Test 4 - Production saturation
# ---------------------------------------------------------------------

pid_sat_prod = PIDController(
    kp=10.0,
    ki=0.0,
    kd=0.0,
    output_limits=(-1.0, 1.0),
)

output_sat_prod, diag_sat_prod = (
    pid_sat_prod.update(
        error=5.0,
        dt=0.1,
    )
)

assert np.isclose(
    diag_sat_prod[
        "unsaturated_output"
    ],
    50.0,
)

assert np.isclose(
    output_sat_prod,
    1.0,
)

assert (
    diag_sat_prod[
        "saturated"
    ]
)

print(
    "Test 4 - Production output saturation: PASS"
)


# ---------------------------------------------------------------------
# Test 5 - Production integral clamping
# ---------------------------------------------------------------------

pid_clamp_prod = PIDController(
    kp=0.0,
    ki=1.0,
    kd=0.0,
    output_limits=(-10.0, 10.0),
    integral_limits=(-0.5, 0.5),
)

for _ in range(20):

    output_clamp_prod, _ = (
        pid_clamp_prod.update(
            error=1.0,
            dt=0.1,
        )
    )

assert np.isclose(
    pid_clamp_prod.integral,
    0.5,
)

assert np.isclose(
    output_clamp_prod,
    0.5,
)

print(
    "Test 5 - Production integral clamping: PASS"
)


# ---------------------------------------------------------------------
# Test 6 - Production reset
# ---------------------------------------------------------------------

pid_reset_prod = PIDController(
    kp=1.0,
    ki=1.0,
    kd=1.0,
)

pid_reset_prod.update(
    error=2.0,
    dt=0.1,
)

pid_reset_prod.reset()

assert np.isclose(
    pid_reset_prod.integral,
    0.0,
)

assert (
    pid_reset_prod.previous_error
    is None
)

print(
    "Test 6 - Production reset behavior: PASS"
)


# ---------------------------------------------------------------------
# Test 7 - Production invalid timestep rejection
# ---------------------------------------------------------------------

invalid_dt_rejected_prod = False

try:

    PIDController(
        kp=1.0,
        ki=0.0,
        kd=0.0,
    ).update(
        error=1.0,
        dt=0.0,
    )

except ValueError:

    invalid_dt_rejected_prod = True


assert invalid_dt_rejected_prod

print(
    "Test 7 - Production invalid timestep rejection: PASS"
)


# ---------------------------------------------------------------------
# Test 8 - Production waypoint parity
#
# Compare the promoted implementation against the notebook-local
# waypoint_guidance() function already validated in Step 6.
# ---------------------------------------------------------------------

parity_cases = [
    (
        np.array(
            [0.0, 0.0, -5.0]
        ),
        np.array(
            [10.0, 0.0, -5.0]
        ),
    ),
    (
        np.array(
            [1.0, 2.0, -3.0]
        ),
        np.array(
            [4.0, 6.0, -15.0]
        ),
    ),
    (
        np.array(
            [-4.0, 3.0, -8.0]
        ),
        np.array(
            [-12.0, -7.0, -2.0]
        ),
    ),
]


for (
    position_world,
    waypoint_world,
) in parity_cases:

    local_result = waypoint_guidance(
        position_world=position_world,
        waypoint_world=waypoint_world,
    )

    production_result = (
        production_waypoint_guidance(
            position_world=position_world,
            waypoint_world=waypoint_world,
        )
    )

    assert np.allclose(
        local_result[
            "displacement_world"
        ],
        production_result[
            "displacement_world"
        ],
    )

    assert np.isclose(
        local_result[
            "desired_heading"
        ],
        production_result[
            "desired_heading"
        ],
    )

    assert np.isclose(
        local_result[
            "desired_depth"
        ],
        production_result[
            "desired_depth"
        ],
    )

    assert np.isclose(
        local_result[
            "horizontal_distance"
        ],
        production_result[
            "horizontal_distance"
        ],
    )

    assert np.isclose(
        local_result[
            "distance"
        ],
        production_result[
            "distance"
        ],
    )


print(
    "Test 8 - Production waypoint parity: PASS"
)


# ---------------------------------------------------------------------
# Test 9 - Production waypoint input validation
# ---------------------------------------------------------------------

bad_waypoint_rejected = False

try:

    production_waypoint_guidance(
        position_world=[
            0.0,
            0.0,
        ],
        waypoint_world=[
            1.0,
            2.0,
            3.0,
        ],
    )

except ValueError:

    bad_waypoint_rejected = True


assert bad_waypoint_rejected

print(
    "Test 9 - Production waypoint input validation: PASS"
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("\n" + "=" * 80)

print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 10 PRODUCTION PARITY"
)

print("=" * 80)


print("\nProduction sources:")

print(
    "  PIDController:      "
    + str(
        pid_source_path.relative_to(
            repo_root
        )
    )
)

print(
    "  waypoint_guidance:  "
    + str(
        waypoint_source_path.relative_to(
            repo_root
        )
    )
)


print("\nPublic API:")

for name in sorted(
    actual_public_api
):

    print(
        f"  {name}"
    )


print("\n" + "=" * 80)

print(
    "STEP 10 PRODUCTION RE-IMPORT "
    "AND PARITY PASSED"
)

print("=" * 80)

print(
    "Fresh production imports reproduced the "
    "validated notebook PID behavior."
)

print(
    "Production waypoint guidance matched the "
    "validated notebook-local implementation."
)

print(
    "The public hydrolab3d.control API exposes "
    "only the intended reusable primitives."
)

print(
    "No production files were modified in this step."
)

Test 1 - Production proportional response: PASS
Test 2 - Production integral accumulation: PASS
Test 3 - Production derivative response: PASS
Test 4 - Production output saturation: PASS
Test 5 - Production integral clamping: PASS
Test 6 - Production reset behavior: PASS
Test 7 - Production invalid timestep rejection: PASS
Test 8 - Production waypoint parity: PASS
Test 9 - Production waypoint input validation: PASS

HYDROLAB-3D NOTEBOOK 09 - STEP 10 PRODUCTION PARITY

Production sources:
  PIDController:      src/hydrolab3d/control/pid.py
  waypoint_guidance:  src/hydrolab3d/control/waypoint.py

Public API:
  PIDController
  waypoint_guidance

STEP 10 PRODUCTION RE-IMPORT AND PARITY PASSED
Fresh production imports reproduced the validated notebook PID behavior.
Production waypoint guidance matched the validated notebook-local implementation.
The public hydrolab3d.control API exposes only the intended reusable primitives.
No production files were modified in this step.


## Step 11 - Fresh-Process Production Validation

Production parity has been verified inside the current notebook kernel.

A stronger validation is now required.

Jupyter notebooks can retain:

```text
old class definitions
old function definitions
cached modules
modified sys.path state
previous imports
```

which may accidentally allow promoted code to appear valid.

This step therefore launches a completely separate Python interpreter.

The fresh process must independently import:

```python
from hydrolab3d.control import (
    PIDController,
    waypoint_guidance,
)
```

from the production source tree.

It then performs a compact deterministic smoke test of:

- the public control API;
- PID proportional behavior;
- PID integral clamping;
- PID output saturation;
- waypoint heading generation;
- waypoint depth generation;
- waypoint-distance geometry;
- input validation.

The subprocess receives only the repository source path required to import HydroLab-3D.

It does not receive notebook-local controller implementations.

### Validation Criteria

This step passes if:

1. the subprocess exits with status code `0`;
2. production control imports succeed in a new interpreter;
3. PID behavior is correct;
4. waypoint guidance is correct;
5. invalid inputs are rejected;
6. the subprocess confirms that production source files are being used.

No production files are modified in this step.

In [15]:
import subprocess
import sys
import textwrap


# ---------------------------------------------------------------------
# Fresh-process validation program
# ---------------------------------------------------------------------

fresh_process_code = textwrap.dedent(
    """
    import inspect
    import sys
    from pathlib import Path

    import numpy as np


    # -------------------------------------------------------------
    # Import production HydroLab-3D control package
    # -------------------------------------------------------------

    import hydrolab3d.control as control

    from hydrolab3d.control import (
        PIDController,
        waypoint_guidance,
    )


    # -------------------------------------------------------------
    # Public API
    # -------------------------------------------------------------

    expected_api = {
        "PIDController",
        "waypoint_guidance",
    }

    actual_api = set(
        control.__all__
    )

    assert actual_api == expected_api


    # -------------------------------------------------------------
    # Verify source files are production modules
    # -------------------------------------------------------------

    pid_source = Path(
        inspect.getsourcefile(
            PIDController
        )
    ).resolve()

    waypoint_source = Path(
        inspect.getsourcefile(
            waypoint_guidance
        )
    ).resolve()

    assert pid_source.name == "pid.py"

    assert waypoint_source.name == "waypoint.py"

    assert pid_source.parent.name == "control"

    assert waypoint_source.parent.name == "control"


    # -------------------------------------------------------------
    # PID proportional behavior
    # -------------------------------------------------------------

    pid = PIDController(
        kp=2.0,
        ki=0.0,
        kd=0.0,
        output_limits=(-10.0, 10.0),
    )

    output, diagnostics = pid.update(
        error=1.5,
        dt=0.1,
    )

    assert np.isclose(
        output,
        3.0,
    )

    assert np.isclose(
        diagnostics[
            "proportional"
        ],
        3.0,
    )


    # -------------------------------------------------------------
    # PID saturation
    # -------------------------------------------------------------

    pid_sat = PIDController(
        kp=10.0,
        ki=0.0,
        kd=0.0,
        output_limits=(-1.0, 1.0),
    )

    output_sat, diagnostics_sat = (
        pid_sat.update(
            error=5.0,
            dt=0.1,
        )
    )

    assert np.isclose(
        output_sat,
        1.0,
    )

    assert diagnostics_sat[
        "saturated"
    ]


    # -------------------------------------------------------------
    # Integral clamping
    # -------------------------------------------------------------

    pid_integral = PIDController(
        kp=0.0,
        ki=1.0,
        kd=0.0,
        output_limits=(-10.0, 10.0),
        integral_limits=(-0.5, 0.5),
    )

    for _ in range(20):

        output_integral, _ = (
            pid_integral.update(
                error=1.0,
                dt=0.1,
            )
        )

    assert np.isclose(
        pid_integral.integral,
        0.5,
    )

    assert np.isclose(
        output_integral,
        0.5,
    )


    # -------------------------------------------------------------
    # Waypoint geometry
    # -------------------------------------------------------------

    guidance = waypoint_guidance(
        position_world=[
            1.0,
            2.0,
            -3.0,
        ],
        waypoint_world=[
            4.0,
            6.0,
            -15.0,
        ],
    )

    assert np.allclose(
        guidance[
            "displacement_world"
        ],
        [
            3.0,
            4.0,
            -12.0,
        ],
    )

    assert np.isclose(
        guidance[
            "horizontal_distance"
        ],
        5.0,
    )

    assert np.isclose(
        guidance[
            "distance"
        ],
        13.0,
    )

    assert np.isclose(
        guidance[
            "desired_depth"
        ],
        15.0,
    )

    assert np.isclose(
        guidance[
            "desired_heading"
        ],
        np.arctan2(
            4.0,
            3.0,
        ),
    )


    # -------------------------------------------------------------
    # Invalid waypoint input rejection
    # -------------------------------------------------------------

    invalid_rejected = False

    try:

        waypoint_guidance(
            position_world=[
                0.0,
                0.0,
            ],
            waypoint_world=[
                1.0,
                2.0,
                3.0,
            ],
        )

    except ValueError:

        invalid_rejected = True


    assert invalid_rejected


    # -------------------------------------------------------------
    # Fresh-process report
    # -------------------------------------------------------------

    print(
        "Fresh production import: PASS"
    )

    print(
        "Public API: "
        + ", ".join(
            sorted(
                actual_api
            )
        )
    )

    print(
        "PID source: "
        + str(
            pid_source
        )
    )

    print(
        "Waypoint source: "
        + str(
            waypoint_source
        )
    )

    print(
        "PID smoke tests: PASS"
    )

    print(
        "Waypoint smoke tests: PASS"
    )

    print(
        "Fresh-process control validation: PASS"
    )
    """
)


# ---------------------------------------------------------------------
# Execute completely separate Python interpreter
#
# Use cwd=repo_root so the validation reflects a normal repository
# invocation rather than inheriting notebook-relative assumptions.
#
# PYTHONPATH explicitly exposes only the production src/ tree.
# ---------------------------------------------------------------------

environment = dict(
    __import__(
        "os"
    ).environ
)

existing_pythonpath = environment.get(
    "PYTHONPATH",
    "",
)

production_src = str(
    repo_root
    / "src"
)

if existing_pythonpath:

    environment[
        "PYTHONPATH"
    ] = (
        production_src
        + __import__(
            "os"
        ).pathsep
        + existing_pythonpath
    )

else:

    environment[
        "PYTHONPATH"
    ] = production_src


result = subprocess.run(
    [
        sys.executable,
        "-c",
        fresh_process_code,
    ],
    cwd=repo_root,
    env=environment,
    capture_output=True,
    text=True,
)


# ---------------------------------------------------------------------
# Always display subprocess output
# ---------------------------------------------------------------------

print("=" * 80)

print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 11 FRESH-PROCESS VALIDATION"
)

print("=" * 80)


print("\nSubprocess stdout:")

if result.stdout.strip():

    print(
        result.stdout.rstrip()
    )

else:

    print(
        "<EMPTY>"
    )


print("\nSubprocess stderr:")

if result.stderr.strip():

    print(
        result.stderr.rstrip()
    )

else:

    print(
        "<EMPTY>"
    )


print(
    f"\nReturn code: {result.returncode}"
)


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

assert result.returncode == 0, (
    "Fresh-process production validation failed."
)

assert (
    "Fresh-process control validation: PASS"
    in result.stdout
)


print("\n" + "=" * 80)

print(
    "STEP 11 FRESH-PROCESS "
    "PRODUCTION VALIDATION PASSED"
)

print("=" * 80)

print(
    "The production control package imported and "
    "validated successfully in a clean Python process."
)

print(
    "No notebook-local controller definitions were "
    "available to the subprocess."
)

print(
    "No production files were modified in this step."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 11 FRESH-PROCESS VALIDATION

Subprocess stdout:
Fresh production import: PASS
Public API: PIDController, waypoint_guidance
PID source: /home/agniacolyte/HydroLab-3D/src/hydrolab3d/control/pid.py
Waypoint source: /home/agniacolyte/HydroLab-3D/src/hydrolab3d/control/waypoint.py
PID smoke tests: PASS
Waypoint smoke tests: PASS
Fresh-process control validation: PASS

Subprocess stderr:
<EMPTY>

Return code: 0

STEP 11 FRESH-PROCESS PRODUCTION VALIDATION PASSED
The production control package imported and validated successfully in a clean Python process.
No notebook-local controller definitions were available to the subprocess.
No production files were modified in this step.


## Step 12 - Generate Compact Persistent Validation Artifacts

The production control primitives have passed notebook-local validation, production promotion, parity checks, and fresh-process validation.

Notebook 09 can therefore persist the most useful results from its validated experiments.

Because the notebook contains several simulations with different durations, artifact generation must not depend on generic variables such as:

```text
times
final_distance
maximum_depth
```

that may have been reassigned by later cells.

Instead, each artifact reconstructs or uses the timebase and metrics belonging specifically to its own validated experiment.

The compact artifact set is:

```text
depth_control_response.png
heading_control_response.png
waypoint_navigation.png
noisy_feedback_navigation.png
summary_metrics.csv
```

All outputs are stored under:

```text
results/notebooks/09_closed_loop_control/
```

No previous notebook output directory is modified.

### Validation Criteria

This step passes if:

1. all five intended artifacts are produced;
2. each experiment uses its own correctly sized timebase;
3. summary values are recomputed directly from the validated histories;
4. all artifacts exist and have non-zero size;
5. every output is confined to the Notebook 09 results directory.

No production source files are modified in this step.

In [17]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Notebook 09 output directory
# ---------------------------------------------------------------------

output_dir = (
    repo_root
    / "results"
    / "notebooks"
    / "09_closed_loop_control"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# Close any figure left open by the failed previous artifact attempt
# ---------------------------------------------------------------------

plt.close(
    "all"
)


# ---------------------------------------------------------------------
# Reconstruct experiment-specific timebases
#
# Do not use the generic notebook variable `times`, because later
# experiments intentionally reused that name with different durations.
# ---------------------------------------------------------------------

depth_dt = 0.02

depth_times_artifact = (
    np.arange(
        len(
            depth_history
        ),
        dtype=float,
    )
    * depth_dt
)


heading_times_a_artifact = np.asarray(
    case_a["times"],
    dtype=float,
)

heading_times_b_artifact = np.asarray(
    case_b["times"],
    dtype=float,
)


waypoint_times_artifact = np.asarray(
    times_nav,
    dtype=float,
)

noisy_times_artifact = np.asarray(
    times_noisy,
    dtype=float,
)


# ---------------------------------------------------------------------
# Validate time/history consistency before plotting
# ---------------------------------------------------------------------

assert (
    len(
        depth_times_artifact
    )
    == len(
        depth_history
    )
)

assert (
    len(
        heading_times_a_artifact
    )
    == len(
        case_a["heading"]
    )
)

assert (
    len(
        heading_times_b_artifact
    )
    == len(
        case_b["heading"]
    )
)

assert (
    len(
        waypoint_times_artifact
    )
    == len(
        pose_nav
    )
)

assert (
    len(
        noisy_times_artifact
    )
    == len(
        pose_noisy
    )
)


# ---------------------------------------------------------------------
# Recompute validated metrics directly from experiment histories
#
# This avoids accidentally using generic variables that may have been
# reassigned by later notebook steps.
# ---------------------------------------------------------------------

depth_target_artifact = 8.0

depth_initial_error_artifact = abs(
    depth_target_artifact
    - float(
        depth_history[0]
    )
)

depth_final_error_artifact = abs(
    depth_target_artifact
    - float(
        depth_history[-1]
    )
)

depth_maximum_artifact = float(
    np.max(
        depth_history
    )
)


heading_a_final_error_deg_artifact = float(
    np.rad2deg(
        abs(
            float(
                case_a["error"][-1]
            )
        )
    )
)

heading_b_initial_error_deg_artifact = float(
    np.rad2deg(
        float(
            case_b["error"][0]
        )
    )
)

heading_b_final_error_deg_artifact = float(
    np.rad2deg(
        abs(
            float(
                case_b["error"][-1]
            )
        )
    )
)


waypoint_initial_distance_artifact = float(
    distance_nav[0]
)

waypoint_final_distance_artifact = float(
    distance_nav[-1]
)

waypoint_navigation_time_artifact = float(
    waypoint_times_artifact[-1]
)

waypoint_maximum_abs_pitch_deg_artifact = float(
    np.rad2deg(
        np.max(
            np.abs(
                pose_nav[:, 4]
            )
        )
    )
)


noisy_initial_distance_artifact = float(
    distance_noisy[0]
)

noisy_final_distance_artifact = float(
    distance_noisy[-1]
)

noisy_navigation_time_artifact = float(
    noisy_times_artifact[-1]
)


# ---------------------------------------------------------------------
# Recompute noisy-sensor discrepancy metrics directly
# ---------------------------------------------------------------------

depth_measurement_error_artifact = (
    measured_depth_noisy[1:]
    - true_depth_noisy[:-1]
)

heading_measurement_error_artifact = np.array(
    [
        float(
            wrap_angle_pi(
                measured_heading_noisy[i]
                - true_heading_noisy[i - 1]
            )
        )
        for i in range(
            1,
            len(
                measured_heading_noisy
            ),
        )
    ],
    dtype=float,
)

depth_error_mean_artifact = float(
    np.mean(
        depth_measurement_error_artifact
    )
)

heading_error_mean_deg_artifact = float(
    np.rad2deg(
        np.mean(
            heading_measurement_error_artifact
        )
    )
)


# ---------------------------------------------------------------------
# Artifact 1 - Depth-control response
# ---------------------------------------------------------------------

depth_plot_path = (
    output_dir
    / "depth_control_response.png"
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    depth_times_artifact,
    depth_history,
    label="Depth",
)

plt.axhline(
    depth_target_artifact,
    linestyle="--",
    label="Target depth",
)

plt.xlabel(
    "Time [s]"
)

plt.ylabel(
    "Depth [m]"
)

plt.title(
    "Closed-Loop Depth Regulation"
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()

plt.tight_layout()

plt.savefig(
    depth_plot_path,
    dpi=160,
)

plt.close()


# ---------------------------------------------------------------------
# Artifact 2 - Heading-control response
# ---------------------------------------------------------------------

heading_plot_path = (
    output_dir
    / "heading_control_response.png"
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    heading_times_a_artifact,
    np.rad2deg(
        case_a["heading"]
    ),
    label="0° → 90°",
)

plt.plot(
    heading_times_b_artifact,
    np.rad2deg(
        case_b["heading"]
    ),
    label="170° → -170°",
)

plt.axhline(
    90.0,
    linestyle="--",
    label="90° target",
)

plt.axhline(
    -170.0,
    linestyle=":",
    label="-170° target",
)

plt.xlabel(
    "Time [s]"
)

plt.ylabel(
    "Heading [deg]"
)

plt.title(
    "Closed-Loop Heading Regulation"
)

plt.grid(
    True,
    alpha=0.3,
)

plt.legend()

plt.tight_layout()

plt.savefig(
    heading_plot_path,
    dpi=160,
)

plt.close()


# ---------------------------------------------------------------------
# Artifact 3 - Ideal waypoint navigation
# ---------------------------------------------------------------------

waypoint_plot_path = (
    output_dir
    / "waypoint_navigation.png"
)

fig = plt.figure(
    figsize=(8, 6)
)

ax = fig.add_subplot(
    111,
    projection="3d",
)

ax.plot(
    pose_nav[:, 0],
    pose_nav[:, 1],
    pose_nav[:, 2],
    label="UUV trajectory",
)

ax.scatter(
    pose_nav[0, 0],
    pose_nav[0, 1],
    pose_nav[0, 2],
    s=50,
    label="Start",
)

ax.scatter(
    waypoint_world[0],
    waypoint_world[1],
    waypoint_world[2],
    s=70,
    marker="x",
    label="Waypoint",
)

ax.set_xlabel(
    "x [m]"
)

ax.set_ylabel(
    "y [m]"
)

ax.set_zlabel(
    "z [m]"
)

ax.set_title(
    "Closed-Loop 3D Waypoint Navigation"
)

ax.legend()

plt.tight_layout()

plt.savefig(
    waypoint_plot_path,
    dpi=160,
)

plt.close()


# ---------------------------------------------------------------------
# Artifact 4 - Noisy-feedback waypoint navigation
# ---------------------------------------------------------------------

noisy_plot_path = (
    output_dir
    / "noisy_feedback_navigation.png"
)

fig = plt.figure(
    figsize=(8, 6)
)

ax = fig.add_subplot(
    111,
    projection="3d",
)

ax.plot(
    pose_noisy[:, 0],
    pose_noisy[:, 1],
    pose_noisy[:, 2],
    label="Noisy-feedback trajectory",
)

ax.scatter(
    pose_noisy[0, 0],
    pose_noisy[0, 1],
    pose_noisy[0, 2],
    s=50,
    label="Start",
)

ax.scatter(
    waypoint_world[0],
    waypoint_world[1],
    waypoint_world[2],
    s=70,
    marker="x",
    label="Waypoint",
)

ax.set_xlabel(
    "x [m]"
)

ax.set_ylabel(
    "y [m]"
)

ax.set_zlabel(
    "z [m]"
)

ax.set_title(
    "Waypoint Navigation with Noisy Sensor Feedback"
)

ax.legend()

plt.tight_layout()

plt.savefig(
    noisy_plot_path,
    dpi=160,
)

plt.close()


# ---------------------------------------------------------------------
# Artifact 5 - Summary metrics
# ---------------------------------------------------------------------

summary_metrics_path = (
    output_dir
    / "summary_metrics.csv"
)

summary_rows = [
    {
        "experiment":
            "depth_control",

        "metric":
            "initial_depth_error_m",

        "value":
            depth_initial_error_artifact,
    },
    {
        "experiment":
            "depth_control",

        "metric":
            "final_depth_error_m",

        "value":
            depth_final_error_artifact,
    },
    {
        "experiment":
            "depth_control",

        "metric":
            "maximum_depth_m",

        "value":
            depth_maximum_artifact,
    },
    {
        "experiment":
            "heading_control_90deg",

        "metric":
            "final_heading_error_deg",

        "value":
            heading_a_final_error_deg_artifact,
    },
    {
        "experiment":
            "heading_control_wrap",

        "metric":
            "initial_wrapped_error_deg",

        "value":
            heading_b_initial_error_deg_artifact,
    },
    {
        "experiment":
            "heading_control_wrap",

        "metric":
            "final_heading_error_deg",

        "value":
            heading_b_final_error_deg_artifact,
    },
    {
        "experiment":
            "waypoint_navigation",

        "metric":
            "initial_distance_m",

        "value":
            waypoint_initial_distance_artifact,
    },
    {
        "experiment":
            "waypoint_navigation",

        "metric":
            "final_distance_m",

        "value":
            waypoint_final_distance_artifact,
    },
    {
        "experiment":
            "waypoint_navigation",

        "metric":
            "navigation_time_s",

        "value":
            waypoint_navigation_time_artifact,
    },
    {
        "experiment":
            "waypoint_navigation",

        "metric":
            "maximum_abs_pitch_deg",

        "value":
            waypoint_maximum_abs_pitch_deg_artifact,
    },
    {
        "experiment":
            "noisy_feedback_navigation",

        "metric":
            "initial_distance_m",

        "value":
            noisy_initial_distance_artifact,
    },
    {
        "experiment":
            "noisy_feedback_navigation",

        "metric":
            "final_distance_m",

        "value":
            noisy_final_distance_artifact,
    },
    {
        "experiment":
            "noisy_feedback_navigation",

        "metric":
            "navigation_time_s",

        "value":
            noisy_navigation_time_artifact,
    },
    {
        "experiment":
            "noisy_feedback_navigation",

        "metric":
            "depth_measurement_error_mean_m",

        "value":
            depth_error_mean_artifact,
    },
    {
        "experiment":
            "noisy_feedback_navigation",

        "metric":
            "heading_measurement_error_mean_deg",

        "value":
            heading_error_mean_deg_artifact,
    },
]


summary_df = pd.DataFrame(
    summary_rows
)

summary_df.to_csv(
    summary_metrics_path,
    index=False,
)


# ---------------------------------------------------------------------
# Artifact validation
# ---------------------------------------------------------------------

expected_artifacts = [
    depth_plot_path,
    heading_plot_path,
    waypoint_plot_path,
    noisy_plot_path,
    summary_metrics_path,
]


for path in expected_artifacts:

    if not path.exists():

        raise FileNotFoundError(
            "Expected artifact was not created: "
            f"{path}"
        )

    if path.stat().st_size <= 0:

        raise RuntimeError(
            f"Artifact is empty: {path}"
        )


# ---------------------------------------------------------------------
# Confirm output isolation
# ---------------------------------------------------------------------

resolved_output_dir = (
    output_dir.resolve()
)

for path in expected_artifacts:

    assert (
        path.parent.resolve()
        == resolved_output_dir
    )


# ---------------------------------------------------------------------
# Validate CSV contents after writing
# ---------------------------------------------------------------------

summary_check = pd.read_csv(
    summary_metrics_path
)

assert (
    len(
        summary_check
    )
    == len(
        summary_rows
    )
)

assert set(
    summary_check.columns
) == {
    "experiment",
    "metric",
    "value",
}

assert np.all(
    np.isfinite(
        summary_check[
            "value"
        ].to_numpy(
            dtype=float
        )
    )
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)

print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 12 ARTIFACT GENERATION"
)

print("=" * 80)


print("\nExperiment timebases:")

print(
    f"  depth samples:        "
    f"{len(depth_times_artifact)}"
)

print(
    f"  heading A samples:    "
    f"{len(heading_times_a_artifact)}"
)

print(
    f"  heading B samples:    "
    f"{len(heading_times_b_artifact)}"
)

print(
    f"  waypoint samples:     "
    f"{len(waypoint_times_artifact)}"
)

print(
    f"  noisy samples:        "
    f"{len(noisy_times_artifact)}"
)


print("\nGenerated artifacts:")

for path in expected_artifacts:

    print(
        f"  {path.relative_to(repo_root)}"
        f"  ({path.stat().st_size} bytes)"
    )


print("\nSummary metrics preview:")

print(
    summary_df.to_string(
        index=False
    )
)


print("\n" + "=" * 80)

print(
    "STEP 12 PERSISTENT ARTIFACT "
    "GENERATION PASSED"
)

print("=" * 80)

print(
    "Each validation artifact used its own "
    "experiment-specific history and timebase."
)

print(
    "Summary metrics were recomputed directly from "
    "validated histories rather than mutable notebook aliases."
)

print(
    "All outputs were written only to the Notebook 09 "
    "results directory."
)

print(
    "No production source files were modified."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 12 ARTIFACT GENERATION

Experiment timebases:
  depth samples:        1001
  heading A samples:    601
  heading B samples:    601
  waypoint samples:     932
  noisy samples:        968

Generated artifacts:
  results/notebooks/09_closed_loop_control/depth_control_response.png  (50942 bytes)
  results/notebooks/09_closed_loop_control/heading_control_response.png  (58659 bytes)
  results/notebooks/09_closed_loop_control/waypoint_navigation.png  (203745 bytes)
  results/notebooks/09_closed_loop_control/noisy_feedback_navigation.png  (208044 bytes)
  results/notebooks/09_closed_loop_control/summary_metrics.csv  (910 bytes)

Summary metrics preview:
               experiment                             metric     value
            depth_control              initial_depth_error_m  6.000000
            depth_control                final_depth_error_m  0.040337
            depth_control                    maximum_depth_m  8.623408
    heading_control_90deg     

## Step 13 - Add Dedicated Automated Tests for the Production Control API

The first dedicated-test attempt exposed a test-runner environment issue.

HydroLab-3D uses a `src/` package layout:

```text
repository/
├── src/
│   └── hydrolab3d/
└── tests/
```

A separate Python subprocess launched from the repository root does not necessarily place:

```text
repository/src/
```

on `sys.path`.

Step 11 handled this explicitly through `PYTHONPATH`, but the initial Step 13 pytest invocation did not.

The corrected test step therefore:

1. writes the dedicated `tests/test_control.py` regression module;
2. preserves byte-for-byte snapshots of the production control files;
3. launches pytest in a separate process;
4. explicitly prepends `src/` to `PYTHONPATH`;
5. displays stdout and stderr before evaluating the return code;
6. confirms production source files were not changed by testing.

The regression suite covers the stable production contract only:

```text
PIDController
waypoint_guidance
```

The larger 6-DOF closed-loop experiments remain notebook-level integration validation rather than permanent unit-test contracts.

### Validation Criteria

This step passes if:

1. `tests/test_control.py` is written successfully;
2. pytest can import HydroLab-3D through the repository `src/` tree;
3. all dedicated control tests pass;
4. the subprocess exits with code `0`;
5. production control files remain byte-for-byte unchanged.

The complete repository test suite will be run separately afterward.

In [19]:
from pathlib import Path
import os
import subprocess
import sys
import textwrap


# ---------------------------------------------------------------------
# Dedicated control-test path
#
# The previous failed Step 13 attempt already created this file, so
# this corrected replacement intentionally rewrites that same
# Notebook 09-owned test module.
# ---------------------------------------------------------------------

test_control_path = (
    repo_root
    / "tests"
    / "test_control.py"
)


# ---------------------------------------------------------------------
# Production files that MUST NOT change during this test step
# ---------------------------------------------------------------------

production_control_files = [
    repo_root
    / "src"
    / "hydrolab3d"
    / "control"
    / "__init__.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "control"
    / "pid.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "control"
    / "waypoint.py",

    repo_root
    / "src"
    / "hydrolab3d"
    / "control"
    / "controller.py",
]


# ---------------------------------------------------------------------
# Snapshot production source BEFORE writing/running tests
# ---------------------------------------------------------------------

production_snapshots = {
    path:
        path.read_bytes()
    for path in production_control_files
}


# ---------------------------------------------------------------------
# Dedicated production-control regression tests
# ---------------------------------------------------------------------

test_control_source = textwrap.dedent(
    '''
    import numpy as np
    import pytest

    import hydrolab3d.control as control

    from hydrolab3d.control import (
        PIDController,
        waypoint_guidance,
    )


    # -----------------------------------------------------------------
    # Public API
    # -----------------------------------------------------------------

    def test_control_public_api():

        assert set(
            control.__all__
        ) == {
            "PIDController",
            "waypoint_guidance",
        }


    # -----------------------------------------------------------------
    # PID - proportional response
    # -----------------------------------------------------------------

    def test_pid_proportional_response():

        pid = PIDController(
            kp=2.0,
            ki=0.0,
            kd=0.0,
            output_limits=(-10.0, 10.0),
        )

        output, diagnostics = pid.update(
            error=1.5,
            dt=0.1,
        )

        assert np.isclose(
            output,
            3.0,
        )

        assert np.isclose(
            diagnostics["proportional"],
            3.0,
        )


    # -----------------------------------------------------------------
    # PID - integral accumulation
    # -----------------------------------------------------------------

    def test_pid_integral_accumulation():

        pid = PIDController(
            kp=0.0,
            ki=2.0,
            kd=0.0,
            output_limits=(-100.0, 100.0),
        )

        for _ in range(4):

            output, _ = pid.update(
                error=1.0,
                dt=0.5,
            )

        assert np.isclose(
            pid.integral,
            2.0,
        )

        assert np.isclose(
            output,
            4.0,
        )


    # -----------------------------------------------------------------
    # PID - derivative response
    # -----------------------------------------------------------------

    def test_pid_derivative_response():

        pid = PIDController(
            kp=0.0,
            ki=0.0,
            kd=3.0,
            output_limits=(-100.0, 100.0),
        )

        first_output, first_diagnostics = pid.update(
            error=1.0,
            dt=0.5,
        )

        second_output, second_diagnostics = pid.update(
            error=2.0,
            dt=0.5,
        )

        assert np.isclose(
            first_output,
            0.0,
        )

        assert np.isclose(
            first_diagnostics["error_derivative"],
            0.0,
        )

        assert np.isclose(
            second_output,
            6.0,
        )

        assert np.isclose(
            second_diagnostics["error_derivative"],
            2.0,
        )


    # -----------------------------------------------------------------
    # PID - saturation
    # -----------------------------------------------------------------

    def test_pid_output_saturation():

        pid = PIDController(
            kp=10.0,
            ki=0.0,
            kd=0.0,
            output_limits=(-1.0, 1.0),
        )

        output, diagnostics = pid.update(
            error=5.0,
            dt=0.1,
        )

        assert np.isclose(
            diagnostics["unsaturated_output"],
            50.0,
        )

        assert np.isclose(
            output,
            1.0,
        )

        assert diagnostics["saturated"]


    # -----------------------------------------------------------------
    # PID - integral clamping
    # -----------------------------------------------------------------

    def test_pid_integral_clamping():

        pid = PIDController(
            kp=0.0,
            ki=1.0,
            kd=0.0,
            output_limits=(-10.0, 10.0),
            integral_limits=(-0.5, 0.5),
        )

        for _ in range(20):

            output, _ = pid.update(
                error=1.0,
                dt=0.1,
            )

        assert np.isclose(
            pid.integral,
            0.5,
        )

        assert np.isclose(
            output,
            0.5,
        )


    # -----------------------------------------------------------------
    # PID - reset
    # -----------------------------------------------------------------

    def test_pid_reset():

        pid = PIDController(
            kp=1.0,
            ki=1.0,
            kd=1.0,
        )

        pid.update(
            error=2.0,
            dt=0.1,
        )

        pid.reset()

        assert np.isclose(
            pid.integral,
            0.0,
        )

        assert (
            pid.previous_error
            is None
        )


    # -----------------------------------------------------------------
    # PID - invalid timestep
    # -----------------------------------------------------------------

    @pytest.mark.parametrize(
        "dt",
        [
            0.0,
            -0.1,
            np.nan,
            np.inf,
        ],
    )
    def test_pid_rejects_invalid_timestep(dt):

        pid = PIDController(
            kp=1.0,
            ki=0.0,
            kd=0.0,
        )

        with pytest.raises(
            ValueError
        ):

            pid.update(
                error=1.0,
                dt=dt,
            )


    # -----------------------------------------------------------------
    # PID - invalid error
    # -----------------------------------------------------------------

    @pytest.mark.parametrize(
        "error",
        [
            np.nan,
            np.inf,
            -np.inf,
        ],
    )
    def test_pid_rejects_nonfinite_error(error):

        pid = PIDController(
            kp=1.0,
            ki=0.0,
            kd=0.0,
        )

        with pytest.raises(
            ValueError
        ):

            pid.update(
                error=error,
                dt=0.1,
            )


    # -----------------------------------------------------------------
    # PID - invalid limits
    # -----------------------------------------------------------------

    def test_pid_rejects_reversed_output_limits():

        with pytest.raises(
            ValueError
        ):

            PIDController(
                kp=1.0,
                ki=0.0,
                kd=0.0,
                output_limits=(1.0, -1.0),
            )


    def test_pid_rejects_reversed_integral_limits():

        with pytest.raises(
            ValueError
        ):

            PIDController(
                kp=1.0,
                ki=0.0,
                kd=0.0,
                integral_limits=(1.0, -1.0),
            )


    # -----------------------------------------------------------------
    # Waypoint guidance - east
    # -----------------------------------------------------------------

    def test_waypoint_guidance_east():

        guidance = waypoint_guidance(
            position_world=[
                0.0,
                0.0,
                -5.0,
            ],
            waypoint_world=[
                10.0,
                0.0,
                -5.0,
            ],
        )

        assert np.isclose(
            guidance["desired_heading"],
            0.0,
        )

        assert np.isclose(
            guidance["desired_depth"],
            5.0,
        )

        assert np.isclose(
            guidance["horizontal_distance"],
            10.0,
        )

        assert np.isclose(
            guidance["distance"],
            10.0,
        )


    # -----------------------------------------------------------------
    # Waypoint guidance - north
    # -----------------------------------------------------------------

    def test_waypoint_guidance_north():

        guidance = waypoint_guidance(
            position_world=[
                0.0,
                0.0,
                -5.0,
            ],
            waypoint_world=[
                0.0,
                10.0,
                -5.0,
            ],
        )

        assert np.isclose(
            guidance["desired_heading"],
            np.pi / 2.0,
        )


    # -----------------------------------------------------------------
    # Waypoint guidance - southwest
    # -----------------------------------------------------------------

    def test_waypoint_guidance_southwest():

        guidance = waypoint_guidance(
            position_world=[
                0.0,
                0.0,
                -5.0,
            ],
            waypoint_world=[
                -10.0,
                -10.0,
                -5.0,
            ],
        )

        assert np.isclose(
            guidance["desired_heading"],
            -3.0 * np.pi / 4.0,
        )


    # -----------------------------------------------------------------
    # Waypoint guidance - general 3D geometry
    # -----------------------------------------------------------------

    def test_waypoint_guidance_general_geometry():

        guidance = waypoint_guidance(
            position_world=[
                1.0,
                2.0,
                -3.0,
            ],
            waypoint_world=[
                4.0,
                6.0,
                -15.0,
            ],
        )

        assert np.allclose(
            guidance["displacement_world"],
            [
                3.0,
                4.0,
                -12.0,
            ],
        )

        assert np.isclose(
            guidance["horizontal_distance"],
            5.0,
        )

        assert np.isclose(
            guidance["distance"],
            13.0,
        )

        assert np.isclose(
            guidance["desired_depth"],
            15.0,
        )

        assert np.isclose(
            guidance["desired_heading"],
            np.arctan2(
                4.0,
                3.0,
            ),
        )


    # -----------------------------------------------------------------
    # Waypoint guidance - vertical-only target
    # -----------------------------------------------------------------

    def test_waypoint_guidance_vertical_only():

        guidance = waypoint_guidance(
            position_world=[
                4.0,
                -2.0,
                -5.0,
            ],
            waypoint_world=[
                4.0,
                -2.0,
                -8.0,
            ],
        )

        assert np.isclose(
            guidance["horizontal_distance"],
            0.0,
        )

        assert np.isclose(
            guidance["desired_heading"],
            0.0,
        )

        assert np.isclose(
            guidance["desired_depth"],
            8.0,
        )

        assert np.isclose(
            guidance["distance"],
            3.0,
        )


    # -----------------------------------------------------------------
    # Waypoint guidance - invalid shapes
    # -----------------------------------------------------------------

    def test_waypoint_guidance_rejects_bad_position_shape():

        with pytest.raises(
            ValueError
        ):

            waypoint_guidance(
                position_world=[
                    0.0,
                    0.0,
                ],
                waypoint_world=[
                    1.0,
                    2.0,
                    3.0,
                ],
            )


    def test_waypoint_guidance_rejects_bad_waypoint_shape():

        with pytest.raises(
            ValueError
        ):

            waypoint_guidance(
                position_world=[
                    0.0,
                    0.0,
                    -5.0,
                ],
                waypoint_world=[
                    1.0,
                    2.0,
                ],
            )


    # -----------------------------------------------------------------
    # Waypoint guidance - non-finite values
    # -----------------------------------------------------------------

    def test_waypoint_guidance_rejects_nonfinite_position():

        with pytest.raises(
            ValueError
        ):

            waypoint_guidance(
                position_world=[
                    0.0,
                    np.nan,
                    -5.0,
                ],
                waypoint_world=[
                    1.0,
                    2.0,
                    -5.0,
                ],
            )


    def test_waypoint_guidance_rejects_nonfinite_waypoint():

        with pytest.raises(
            ValueError
        ):

            waypoint_guidance(
                position_world=[
                    0.0,
                    0.0,
                    -5.0,
                ],
                waypoint_world=[
                    1.0,
                    np.inf,
                    -5.0,
                ],
            )
    '''
)


# ---------------------------------------------------------------------
# Write / replace only the Notebook 09 dedicated test module
# ---------------------------------------------------------------------

test_control_path.write_text(
    test_control_source,
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Configure isolated pytest environment
#
# Critical fix:
#
# repo_root/src is explicitly prepended to PYTHONPATH.
# ---------------------------------------------------------------------

environment = dict(
    os.environ
)

production_src = str(
    (
        repo_root
        / "src"
    ).resolve()
)

existing_pythonpath = environment.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:

    environment[
        "PYTHONPATH"
    ] = (
        production_src
        + os.pathsep
        + existing_pythonpath
    )

else:

    environment[
        "PYTHONPATH"
    ] = production_src


# ---------------------------------------------------------------------
# Run ONLY the dedicated control tests
# ---------------------------------------------------------------------

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        str(
            test_control_path
        ),
    ],
    cwd=repo_root,
    env=environment,
    capture_output=True,
    text=True,
)


# ---------------------------------------------------------------------
# Verify production files remained byte-for-byte unchanged
# ---------------------------------------------------------------------

production_files_unchanged = True

for (
    path,
    before_bytes,
) in production_snapshots.items():

    after_bytes = (
        path.read_bytes()
    )

    if (
        after_bytes
        != before_bytes
    ):

        production_files_unchanged = False

        raise RuntimeError(
            "Production control source unexpectedly changed "
            f"during testing: {path}"
        )


# ---------------------------------------------------------------------
# Report BEFORE final assertion
#
# If pytest ever fails again, its actual diagnostic remains visible.
# ---------------------------------------------------------------------

print("=" * 80)

print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 13 DEDICATED CONTROL TESTS"
)

print("=" * 80)


print("\nTest module:")

print(
    "  "
    + str(
        test_control_path.relative_to(
            repo_root
        )
    )
)


print("\nPytest environment:")

print(
    f"  working directory: "
    f"{repo_root}"
)

print(
    f"  production src:    "
    f"{production_src}"
)


print("\nDedicated pytest stdout:")

if result.stdout.strip():

    print(
        result.stdout.rstrip()
    )

else:

    print(
        "<EMPTY>"
    )


print("\nDedicated pytest stderr:")

if result.stderr.strip():

    print(
        result.stderr.rstrip()
    )

else:

    print(
        "<EMPTY>"
    )


print(
    f"\nReturn code: "
    f"{result.returncode}"
)

print(
    "Production files unchanged: "
    f"{production_files_unchanged}"
)


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

assert result.returncode == 0, (
    "Dedicated control regression tests failed. "
    "Inspect the pytest stdout/stderr printed above."
)


assert (
    production_files_unchanged
)


print("\n" + "=" * 80)

print(
    "STEP 13 DEDICATED CONTROL "
    "TESTS PASSED"
)

print("=" * 80)

print(
    "The production PID and waypoint-guidance API "
    "is protected by dedicated automated tests."
)

print(
    "Pytest imported HydroLab-3D explicitly from "
    "the repository src/ tree."
)

print(
    "Production control files remained byte-for-byte "
    "unchanged during testing."
)

print(
    "The complete repository test suite has not yet "
    "been run."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 13 DEDICATED CONTROL TESTS

Test module:
  tests/test_control.py

Pytest environment:
  working directory: /home/agniacolyte/HydroLab-3D
  production src:    /home/agniacolyte/HydroLab-3D/src

Dedicated pytest stdout:
.........................                                                [100%]
25 passed in 0.06s

Dedicated pytest stderr:
<EMPTY>

Return code: 0
Production files unchanged: True

STEP 13 DEDICATED CONTROL TESTS PASSED
The production PID and waypoint-guidance API is protected by dedicated automated tests.
Pytest imported HydroLab-3D explicitly from the repository src/ tree.
Production control files remained byte-for-byte unchanged during testing.
The complete repository test suite has not yet been run.


## Step 14 - Full Repository Regression Test

The Notebook 09 control layer has now passed:

- notebook-local primitive validation;
- closed-loop depth regulation;
- wrapped heading regulation;
- waypoint-guidance validation;
- ideal 6-DOF waypoint navigation;
- noisy-sensor waypoint navigation;
- production promotion;
- production parity validation;
- fresh-process validation;
- artifact generation;
- 25 dedicated control regression tests.

The final executable requirement is to run the complete HydroLab-3D test suite.

This verifies that the new control package does not break behavior established by earlier notebooks, including:

```text
kinematics
orientation-related utilities
dynamics
drag
buoyancy and gravity
thrusters
ocean currents
6-DOF UUV dynamics
sensor models
control
```

The test process explicitly exposes:

```text
src/
```

through `PYTHONPATH`, matching the repository's source-layout structure.

### Validation Criteria

This step passes if:

1. pytest discovers the complete repository test suite;
2. every discovered test passes;
3. the subprocess exits with return code `0`;
4. no production source file is modified by the test run.

The actual pytest count will be reported from the test runner rather than predicted beforehand.

In [20]:
import os
import subprocess
import sys


# ---------------------------------------------------------------------
# Snapshot production source before full-suite validation
# ---------------------------------------------------------------------

production_root = (
    repo_root
    / "src"
    / "hydrolab3d"
)

production_files = sorted(
    path
    for path in production_root.rglob(
        "*.py"
    )
    if path.is_file()
)

production_snapshots = {
    path:
        path.read_bytes()
    for path in production_files
}


# ---------------------------------------------------------------------
# Configure repository src/ layout for fresh pytest subprocess
# ---------------------------------------------------------------------

environment = dict(
    os.environ
)

production_src = str(
    (
        repo_root
        / "src"
    ).resolve()
)

existing_pythonpath = environment.get(
    "PYTHONPATH",
    "",
)

if existing_pythonpath:

    environment[
        "PYTHONPATH"
    ] = (
        production_src
        + os.pathsep
        + existing_pythonpath
    )

else:

    environment[
        "PYTHONPATH"
    ] = production_src


# ---------------------------------------------------------------------
# Run the COMPLETE repository test suite
# ---------------------------------------------------------------------

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
    ],
    cwd=repo_root,
    env=environment,
    capture_output=True,
    text=True,
)


# ---------------------------------------------------------------------
# Verify test execution did not mutate production source
# ---------------------------------------------------------------------

changed_production_files = []

for (
    path,
    before_bytes,
) in production_snapshots.items():

    after_bytes = (
        path.read_bytes()
    )

    if (
        after_bytes
        != before_bytes
    ):

        changed_production_files.append(
            path
        )


# ---------------------------------------------------------------------
# Report BEFORE assertions
#
# Any failure remains visible for diagnosis.
# ---------------------------------------------------------------------

print("=" * 80)

print(
    "HYDROLAB-3D NOTEBOOK 09 - "
    "STEP 14 FULL REPOSITORY TEST SUITE"
)

print("=" * 80)


print("\nPytest environment:")

print(
    f"  working directory: "
    f"{repo_root}"
)

print(
    f"  production src:    "
    f"{production_src}"
)

print(
    f"  production Python files snapshotted: "
    f"{len(production_snapshots)}"
)


print("\nFull pytest stdout:")

if result.stdout.strip():

    print(
        result.stdout.rstrip()
    )

else:

    print(
        "<EMPTY>"
    )


print("\nFull pytest stderr:")

if result.stderr.strip():

    print(
        result.stderr.rstrip()
    )

else:

    print(
        "<EMPTY>"
    )


print(
    f"\nReturn code: "
    f"{result.returncode}"
)


print(
    "Production source unchanged: "
    f"{len(changed_production_files) == 0}"
)


if changed_production_files:

    print(
        "\nUnexpectedly changed production files:"
    )

    for path in changed_production_files:

        print(
            "  "
            + str(
                path.relative_to(
                    repo_root
                )
            )
        )


# ---------------------------------------------------------------------
# Final executable validation
# ---------------------------------------------------------------------

assert result.returncode == 0, (
    "Full HydroLab-3D regression suite failed. "
    "Inspect pytest stdout/stderr printed above."
)

assert not changed_production_files, (
    "The full test suite unexpectedly modified "
    "production source files."
)


print("\n" + "=" * 80)

print(
    "STEP 14 FULL REPOSITORY "
    "REGRESSION TEST PASSED"
)

print("=" * 80)

print(
    "The complete HydroLab-3D test suite passed "
    "with Notebook 09 control functionality included."
)

print(
    "Earlier notebook functionality remained "
    "regression-safe."
)

print(
    "Production source remained byte-for-byte "
    "unchanged during the full-suite test run."
)

HYDROLAB-3D NOTEBOOK 09 - STEP 14 FULL REPOSITORY TEST SUITE

Pytest environment:
  working directory: /home/agniacolyte/HydroLab-3D
  production src:    /home/agniacolyte/HydroLab-3D/src
  production Python files snapshotted: 51

Full pytest stdout:
........................................................................ [ 47%]
........................................................................ [ 94%]
........                                                                 [100%]
152 passed in 0.68s

Full pytest stderr:
<EMPTY>

Return code: 0
Production source unchanged: True

STEP 14 FULL REPOSITORY REGRESSION TEST PASSED
The complete HydroLab-3D test suite passed with Notebook 09 control functionality included.
Earlier notebook functionality remained regression-safe.
Production source remained byte-for-byte unchanged during the full-suite test run.


# Conclusion

Notebook 09 establishes the first complete closed-loop autonomy layer in HydroLab-3D.

The simulator now supports the control chain:

```text
reference / waypoint
        ↓
guidance
        ↓
depth + heading controllers
        ↓
generalized actuator wrench
        ↓
6-DOF UUV dynamics
        ↓
sensor feedback
        ↺
```

This notebook intentionally remained lightweight and algorithm-first.

It did not introduce a large autonomy framework, estimator stack, mission planner, or high-fidelity actuator allocation system.

Instead, it established the minimum reusable control primitives required for autonomous UUV behavior while preserving HydroLab-3D's existing physics, coordinate conventions, sensor models, and modular architecture.

---

## 1. Production Control Primitives

Two reusable components were promoted into:

```text
src/hydrolab3d/control/
```

### PIDController

The production scalar PID controller implements:

$$
u(t)
=
K_p e(t)
+
K_i
\int e(t)\,dt
+
K_d
\frac{de(t)}{dt}.
$$

Its production behavior includes:

- proportional control;
- integral accumulation;
- derivative response;
- configurable output saturation;
- configurable integral-state clamping;
- controller reset;
- finite-input validation;
- positive-timestep validation;
- diagnostic outputs for internal PID terms.

The controller itself is intentionally domain-agnostic.

It does not know whether its error represents:

```text
depth
heading
velocity
position
or another scalar quantity
```

This keeps the primitive reusable for later HydroLab-3D autonomy notebooks.

### waypoint_guidance

The production waypoint-guidance function converts:

```text
current world position
+
world-frame waypoint
```

into:

```text
desired heading
desired depth
horizontal distance
3D waypoint distance
world-frame displacement
```

For:

$$
\Delta x=x_g-x
$$

and:

$$
\Delta y=y_g-y,
$$

the desired heading is:

$$
\psi_{ref}
=
\operatorname{atan2}
\left(
\Delta y,
\Delta x
\right).
$$

HydroLab-3D retains the previously established world convention:

$$
+z_{world}
\text{ is upward}
$$

and therefore:

$$
depth=-z.
$$

The waypoint depth reference is consequently:

$$
d_{ref}=-z_g.
$$

The guidance function performs geometry only.

It does not perform localization, control, state estimation, or actuation.

---

## 2. Closed-Loop Depth Regulation

Depth control was validated first as an isolated physical loop.

The validated structure was:

```text
desired depth
      ↓
depth error
      ↓
PID
      ↓
vertical force
      ↓
6-DOF plant
      ↓
depth sensor
      ↺
```

Because depth and world vertical position use opposite signs:

$$
d=-z,
$$

a positive depth error requires motion toward negative world $z$.

The controller therefore applied:

$$
F_z
=
-u_d F_{z,\max}.
$$

The experiment began at:

$$
d_0=2.0\ \text{m}
$$

with a target of:

$$
d_{ref}=8.0\ \text{m}.
$$

The final depth was approximately:

$$
8.0403\ \text{m},
$$

corresponding to a final absolute depth error of approximately:

$$
0.0403\ \text{m}.
$$

The initial error of:

$$
6.0\ \text{m}
$$

was reduced by approximately:

$$
99.33\%.
$$

The trajectory reached a maximum depth of approximately:

$$
8.6234\ \text{m},
$$

showing a modest transient overshoot before settling.

The final vertical body velocity was approximately:

$$
0.0074\ \text{m/s},
$$

indicating that the response was close to settled.

No pose or velocity state was modified directly.

All depth motion occurred through the production 6-DOF actuator-wrench interface.

---

## 3. Closed-Loop Heading Regulation

Heading control required circular rather than linear error arithmetic.

The correct heading error was defined using the validated angle wrapper:

$$
e_\psi
=
\operatorname{wrap}_{(-\pi,\pi]}
\left(
\psi_{ref}-\psi
\right).
$$

This prevents incorrect long-path commands near the $-\pi/\pi$ boundary.

Two cases were validated.

### Ordinary heading command

The first case commanded:

```text
0 deg → +90 deg
```

The final heading was approximately:

```text
92.116 deg
```

with a final absolute error of approximately:

```text
2.116 deg
```

and a final yaw rate close to zero.

### Boundary-crossing command

The second case commanded:

```text
+170 deg → -170 deg
```

Naive subtraction would produce:

```text
-340 deg
```

which is physically the wrong rotation direction.

The wrapped controller correctly produced:

```text
+20 deg
```

as the initial shortest-path heading error.

The final absolute heading error was approximately:

```text
0.473 deg
```

with a final yaw rate close to zero.

This validates circular heading regulation across the angular discontinuity.

---

## 4. Waypoint Guidance

Waypoint geometry was independently validated before being connected to the plant.

Cases included:

- east;
- north;
- southwest;
- deeper target;
- shallower target;
- vertical-only displacement;
- general 3D displacement;
- invalid input rejection.

For the general displacement:

```text
[3, 4, -12] m
```

the guidance layer correctly produced:

```text
horizontal distance = 5 m
3D distance = 13 m
```

while respecting the HydroLab-3D depth convention.

This isolation was important.

It ensured that later navigation behavior could be diagnosed separately as:

```text
guidance
versus
control
versus
vehicle dynamics
```

rather than treating the complete autonomy loop as one opaque block.

---

## 5. Closed-Loop 3D Waypoint Navigation

The validated waypoint loop combined:

```text
waypoint guidance
+
heading PID
+
depth PID
+
forward propulsion
+
production 6-DOF dynamics
```

The vehicle began at:

```text
[0, 0, -2] m
```

and targeted:

```text
[20, 12, -8] m.
```

The initial three-dimensional distance was approximately:

```text
24.083 m.
```

The vehicle entered the:

```text
1.0 m
```

success region after approximately:

```text
18.62 s.
```

Its final position was approximately:

```text
[19.7205, 11.0466, -8.0775] m
```

with a final waypoint distance of approximately:

```text
0.9965 m.
```

The waypoint distance therefore decreased by approximately:

```text
95.86%.
```

Again, no direct pose manipulation was used.

The vehicle reached the waypoint only through:

```text
forward force
vertical force
yaw moment
```

applied through the 6-DOF plant.

---

## 6. Passive Attitude Stability

The first combined waypoint experiment revealed an important physical requirement.

A neutrally buoyant vehicle with:

```text
center of gravity = center of buoyancy
```

has no hydrostatic restoring lever for roll or pitch.

Once surge, heave, and rotational dynamics were simultaneously excited, pitch could grow enough that forward body thrust acquired a large world-frame vertical component.

The corrected vehicle retained neutral net buoyancy while using:

```text
center of gravity:
[0.00, 0.00, 0.00] m

center of buoyancy:
[0.00, 0.00, 0.08] m
```

so that the center of buoyancy lay above the center of gravity.

This supplied passive roll/pitch restoring moments.

The successful navigation run experienced a maximum absolute pitch of approximately:

```text
42.68 deg
```

and a maximum absolute roll of approximately:

```text
19.62 deg.
```

By waypoint arrival, the vehicle had recovered to approximately:

```text
roll  = 5.94 deg
pitch = 3.13 deg.
```

The large transient pitch is intentionally retained as an engineering limitation rather than hidden.

It shows that the minimal decoupled controller is effective for demonstrating closed-loop autonomy but is not an optimized full-state marine controller.

Future work may reasonably investigate:

```text
coupled MIMO control
pitch regulation
velocity control
control allocation
model-based control
LQR
MPC
or nonlinear control
```

without changing the basic architecture established here.

---

## 7. Euler-Angle Validity

The waypoint experiment does not impose an arbitrary small-angle requirement.

The actual mathematical limitation of the current Z-Y-X Euler representation occurs near:

$$
\theta=\pm90^\circ.
$$

The trajectory therefore monitored:

$$
|\cos\theta|.
$$

For the ideal waypoint experiment, the minimum observed value was approximately:

```text
0.735
```

corresponding to roughly:

```text
47.3 deg
```

of angular margin from the nearest pitch singularity.

The trajectory therefore remained well within the valid region of the current Euler-angle kinematic representation.

---

## 8. Noisy Sensor Feedback

The waypoint loop was then repeated using the production Notebook 08 sensor models.

The controller no longer received perfect depth and heading.

The configured depth sensor used:

```text
bias       = 0.03 m
noise std  = 0.04 m
```

and the heading sensor used:

```text
bias       = 0.75 deg
noise std  = 1.00 deg.
```

Independent seeded random-number generators preserved exact reproducibility.

The observed depth measurement error had approximately:

```text
mean = 0.0281 m
std  = 0.0393 m
```

and heading measurement error had approximately:

```text
mean = 0.777 deg
std  = 1.026 deg.
```

These values closely match the configured sensor imperfections.

Despite imperfect feedback, the vehicle still reached the waypoint-success region.

The noisy-feedback run reduced distance from:

```text
24.083 m
```

to approximately:

```text
0.996 m
```

in approximately:

```text
19.34 s.
```

Its final position was approximately:

```text
[20.1462, 11.0223, -7.8792] m.
```

This demonstrates that the control architecture can operate on imperfect sensor observations without overwriting simulator truth.

---

## 9. Localization Boundary

One assumption remains intentionally explicit.

Waypoint guidance used:

```text
pose[:3]
```

as a perfect world-position source.

This represents a temporary perfect-localization baseline.

Notebook 08 provides:

```text
depth
heading
IMU
DVL
```

sensor abstractions, but these do not by themselves constitute perfect global underwater XYZ localization.

Notebook 09 therefore does not pretend that a localization system exists where one has not been implemented.

A future estimator may replace the perfect-position source with:

```text
dead reckoning
DVL + IMU integration
acoustic positioning
SLAM
EKF
factor-graph estimation
or another localization method
```

without requiring the waypoint-guidance or controller architecture to be redesigned.

---

## 10. Production Promotion

After notebook-level validation, only the reusable primitives were promoted.

The resulting control package contains:

```text
src/hydrolab3d/control/
├── __init__.py
├── controller.py
├── pid.py
└── waypoint.py
```

The public production API is:

```python
PIDController
waypoint_guidance
```

`controller.py` remains intentionally empty.

An abstract controller hierarchy was not introduced because Notebook 09 did not demonstrate a requirement for one.

This follows the HydroLab-3D philosophy:

```text
validate first
promote only demonstrated abstractions
avoid framework growth without evidence
```

Notebook-specific plant parameters, controller gains, waypoint coordinates, actuator limits, and validation simulations were not promoted into the reusable source package.

---

## 11. Production Parity and Fresh-Process Validation

The promoted code was re-imported from disk and compared against the validated notebook prototypes.

Production PID behavior reproduced:

- proportional response;
- integral accumulation;
- derivative response;
- saturation;
- integral clamping;
- reset behavior;
- invalid timestep rejection.

Production waypoint guidance matched the notebook-local implementation across multiple geometric cases.

The public API was verified to contain exactly:

```text
PIDController
waypoint_guidance
```

A completely separate Python interpreter was then launched.

That clean process had no access to:

```text
NotebookPID
notebook-local waypoint_guidance
cached notebook definitions
Jupyter execution state
```

and independently imported the production package successfully.

This confirmed that the promoted implementation does not depend on notebook state.

---

## 12. Persistent Artifacts

Notebook 09 generated a compact artifact set under:

```text
results/notebooks/09_closed_loop_control/
```

containing:

```text
depth_control_response.png
heading_control_response.png
waypoint_navigation.png
noisy_feedback_navigation.png
summary_metrics.csv
```

Each artifact uses its own experiment-specific history and timebase.

This avoids accidental coupling through mutable notebook variables reused by later experiments.

The summary CSV records the central quantitative validation results, including:

```text
depth-control error
heading-control error
wrapped heading error
waypoint distance
navigation time
maximum pitch excursion
sensor measurement bias behavior
```

No earlier notebook result directory was overwritten.

---

## 13. Automated Regression Tests

A dedicated production-control regression module was added:

```text
tests/test_control.py
```

It covers:

- public control API;
- proportional PID response;
- integral accumulation;
- derivative response;
- output saturation;
- integral clamping;
- reset;
- invalid timestep rejection;
- invalid error rejection;
- invalid limit rejection;
- waypoint heading geometry;
- waypoint depth convention;
- waypoint distance geometry;
- invalid waypoint shapes;
- non-finite waypoint inputs.

The dedicated control suite completed with:

```text
25 passed
```

and production control files remained byte-for-byte unchanged during the test run.

---

## 14. Full Repository Regression Result

The complete HydroLab-3D repository test suite was then executed.

Final result:

```text
152 passed
```

with:

```text
0 failed
```

and a clean subprocess return code.

The test run also verified that all snapshotted production Python source files remained byte-for-byte unchanged.

This establishes that Notebook 09's new control functionality did not regress the previously validated:

```text
kinematics
dynamics
hydrodynamic drag
buoyancy and gravity
thrusters
ocean currents
6-DOF UUV dynamics
sensor models
```

or other inherited repository behavior.

---

## 15. What Notebook 09 Establishes

Before this notebook, HydroLab-3D could simulate:

```text
vehicle motion
forces
hydrodynamic effects
currents
thruster actuation
6-DOF dynamics
sensor observations
```

but those pieces did not yet form an autonomous feedback loop.

Notebook 09 closes that loop.

HydroLab-3D can now express the essential autonomy structure:

```text
observe
   ↓
compare against objective
   ↓
compute control action
   ↓
act through physical dynamics
   ↓
observe again
```

That transition is significant.

The simulator is no longer only capable of answering:

> What happens to this underwater vehicle when forces are applied?

It can now also answer:

> What control action should be applied next so that the vehicle autonomously moves toward an objective?

---

## 16. Current Limitations

The implementation remains deliberately minimal.

Notebook 09 does not yet provide:

- global underwater localization;
- path planning around obstacles;
- sonar-based perception;
- current-aware guidance optimization;
- velocity-loop control;
- explicit roll control;
- explicit pitch control;
- multi-thruster control allocation;
- actuator dynamics beyond the existing lightweight models;
- anti-windup beyond integral-state clamping;
- derivative filtering;
- gain scheduling;
- adaptive control;
- nonlinear model-predictive control;
- mission sequencing;
- collision avoidance;
- multi-agent coordination.

The waypoint controller also exhibits substantial transient pitch during aggressive coupled motion.

This is not concealed.

It defines one clear boundary between the lightweight baseline implemented here and more advanced marine-control research.

---

## 17. Notebook 09 Result

Notebook 09 successfully establishes:

```text
PID control
+
wrapped heading regulation
+
depth regulation
+
waypoint guidance
+
physical 6-DOF actuation
+
sensor-based feedback
+
noisy-feedback robustness
+
production promotion
+
automated regression coverage
```

while preserving HydroLab-3D's core design principles:

```text
NumPy-first
CPU-friendly
headless-first
modular
explicit
reproducible
selective fidelity
algorithm-oriented
```

The final repository validation result is:

```text
152 passed
```

with the new control layer included.

HydroLab-3D therefore ends Notebook 09 with its first validated closed-loop autonomous UUV behavior.

---

# Handoff to Notebook 10

Notebook 09 established the autonomous control primitives and proved them against the existing simulator components.

The next architectural step is to integrate these validated pieces behind a cleaner simulator-facing interaction loop.

Notebook 10 can therefore focus on the higher-level:

```text
Simulator.step()
```

workflow, where:

```text
action
   ↓
actuation
   ↓
dynamics
   ↓
state update
   ↓
sensor observation
   ↓
returned simulation output
```

can be exposed through a unified simulator API.

The control algorithms created here can then sit above that interface rather than manually orchestrating each production subsystem.

Notebook 09 therefore serves as the bridge between:

```text
validated marine simulation components
```

and:

```text
an integrated autonomous simulation environment.
```

---